In [120]:
import re
import os
import json
import time
import ast
import configparser
from pathlib import Path
from datetime import datetime

import pandas as pd
from openai import OpenAI
from fuzzywuzzy import fuzz
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from tqdm import tqdm

In [121]:
# ----------------------------
# File paths
# ----------------------------
CONFIG_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\config_prestep.ini")
ENV_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\.env")
CRF_JSON_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\KnowledgeBase\CRF_descriptions.json")

# ----------------------------
# Helpers
# ----------------------------
def load_env_file(env_path: Path) -> dict:
    env_vars = {}
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_vars[key.strip()] = value.strip().strip('"').strip("'")
    return env_vars


def build_crf_reference_text(crf_json: dict) -> str:
    """
    Turn CRF_descriptions.json into a compact prompt-ready reference block.
    """
    lines = []
    for crf in crf_json.get("CRFs", []):
        name = crf.get("name", "").strip()
        abbreviations = ", ".join(crf.get("abbreviations", []))
        description = crf.get("description", "").strip()

        lines.append(f"Official CRF Name: {name}")
        if abbreviations:
            lines.append(f"Abbreviations: {abbreviations}")
        if description:
            lines.append(f"Description: {description}")
        lines.append("")

    return "\n".join(lines).strip()

# ----------------------------
# Load config
# ----------------------------
config = configparser.ConfigParser()
config.read(CONFIG_PATH, encoding="utf-8")

input_file = config["Files"]["input_file"]
input_worksheet = config["Files"]["input_worksheet"]
output_file = config["Files"]["output_file"]

crf_column = config["Columns"]["crf_column"]
variable_column = config["Columns"]["variable_column"]
description_column = config["Columns"]["description_column"]

crf_id_prestep_instruction = config["Instructions"]["crf_id_prestep"]
form_harmonizer_instruction = config["Instructions"]["form_harmonizer"]
matching_instruction = config["Instructions"]["matching_instruction"]

# Optional model overrides from config
PRESTEP_MODEL = config.get("Models", "prestep_model", fallback="gpt-4.1-mini")
HARMONIZER_MODEL = config.get("Models", "harmonizer_model", fallback="gpt-4.1-mini")
MATCHING_MODEL = config.get("Models", "matching_model", fallback="gpt-4.1-mini")

# ----------------------------
# Load .env + client
# ----------------------------
env_vars = load_env_file(ENV_PATH)
api_key = env_vars.get("OPENAI_API_KEY") or env_vars.get("api_key")

if not api_key:
    raise ValueError("No OpenAI API key found in .env. Expected OPENAI_API_KEY or api_key.")

client = OpenAI(api_key=api_key)

# ----------------------------
# Load local CRF KB
# ----------------------------
with open(CRF_JSON_PATH, "r", encoding="utf-8") as f:
    crf_kb = json.load(f)

crf_reference_text = build_crf_reference_text(crf_kb)

print("✅ Setup loaded successfully.")
print(f"Config path: {CONFIG_PATH}")
print(f".env path: {ENV_PATH}")
print(f"CRF JSON path: {CRF_JSON_PATH}")
print(f"Input file: {input_file}")
print(f"Input worksheet: {input_worksheet}")
print(f"Output file: {output_file}")
print(f"API key loaded: {'Yes' if api_key else 'No'}")
print(f"OpenAI client created: {'Yes' if client else 'No'}")
print(f"Loaded {len(crf_kb.get('CRFs', []))} CRF definitions.")
print(f"Prestep model: {PRESTEP_MODEL}")
print(f"Harmonizer model: {HARMONIZER_MODEL}")
print(f"Matching model: {MATCHING_MODEL}")

print("\n--- CRF reference preview ---")
print(crf_reference_text[:1200] + ("..." if len(crf_reference_text) > 1200 else ""))

✅ Setup loaded successfully.
Config path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\config_prestep.ini
.env path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\.env
CRF JSON path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\KnowledgeBase\CRF_descriptions.json
Input file: in\Testfile.xlsx
Input worksheet: Sheet1
Output file: out\Testfile_2026-05-05.xlsx
API key loaded: Yes
OpenAI client created: Yes
Loaded 24 CRF definitions.
Prestep model: gpt-4.1-mini
Harmonizer model: gpt-4.1-mini
Matching model: gpt-4.1-mini

--- CRF reference preview ---
Official CRF Name: Brief Pain Inventory (BPI)
Abbreviations: BPI, Brief Pain Inventory, B.P.I.
Description: A self-report questionnaire measuring pain severity and interference with daily activities.

Official CRF Name: BPI Pain Interference
Abbreviations: BPI Interference, Pain Interference, BPI-PI
Description: Assesses how pain impacts various aspects of daily life, including mo

In [122]:
# ----------------------------
# Responses API prestep call functions
# ----------------------------
def build_prestep_input(crf_name, variable_name, description_text, acronym_hint=""):
    acronym_block = ""
    if acronym_hint:
        acronym_block = f"\nDeterministic acronym hint from variable name: {acronym_hint}\n"

    return f"""
Study row context:
- Original CRF/Form Name: {crf_name}
- Variable Name: {variable_name}
- Description / Field Label: {description_text}
{acronym_block}
Official HEAL Core CRF reference:
{crf_reference_text}

Task:
Use the study row context, deterministic acronym hint when present, and the official HEAL Core CRF reference to identify
the most likely HEAL Core CRF or state that no confident HEAL Core CRF match exists.

Return JSON with exactly these keys:
- CRF
- Rationale
""".strip()


def call_prestep_responses(crf_name, variable_name, description_text):
    """
    Single Responses API call for prestep CRF identification.
    Returns plain text output.
    """
    user_input = build_prestep_input(crf_name, variable_name, description_text)

    response = client.responses.create(
        model=PRESTEP_MODEL,
        instructions=crf_id_prestep_instruction,
        input=user_input
    )

    return response.output_text.strip()


def call_prestep_responses_with_retry(crf_name, variable_name, description_text, max_retries=3, base_sleep_seconds=2):
    """
    Safe wrapper around the Responses API prestep call.
    Returns a dictionary so failures do not crash the whole run.
    """
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            output_text = call_prestep_responses(crf_name, variable_name, description_text)

            return {
                "Full Response": output_text,
                "Prestep Run Status": "Reviewed",
                "Prestep Attempts": attempt,
                "Prestep Error": ""
            }

        except Exception as e:
            last_error = str(e)
            print(f"[warning] prestep failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "Full Response": "",
        "Prestep Run Status": "ERROR",
        "Prestep Attempts": max_retries,
        "Prestep Error": last_error
    }


print("✅ Responses API prestep call functions loaded.")

✅ Responses API prestep call functions loaded.


In [123]:
# ----------------------------
# Parse / normalize Full Response
# ----------------------------
def parse_full_response_cell(value):
    """
    Robust parser for Full Response values that may look like:
    1. Proper JSON object string
    2. Quoted JSON string with escaped quotes
    3. Multi-line JSON-like string
    """
    if pd.isna(value):
        return {
            "Refined CRF Name": "",
            "Rationale": "",
            "Parsed Full Response": "",
            "Parse Status": "EMPTY",
            "Parse Error": ""
        }

    raw = str(value).strip()

    if not raw:
        return {
            "Refined CRF Name": "",
            "Rationale": "",
            "Parsed Full Response": "",
            "Parse Status": "EMPTY",
            "Parse Error": ""
        }

    errors = []

    # Attempt 1: direct JSON parse
    try:
        parsed = json.loads(raw)

        if isinstance(parsed, dict):
            return {
                "Refined CRF Name": str(parsed.get("CRF", "")).strip(),
                "Rationale": str(parsed.get("Rationale", "")).strip(),
                "Parsed Full Response": json.dumps(parsed, ensure_ascii=False),
                "Parse Status": "PARSED_JSON",
                "Parse Error": ""
            }

        # If first parse returns a string, try parsing that as JSON again
        if isinstance(parsed, str):
            try:
                parsed2 = json.loads(parsed)
                if isinstance(parsed2, dict):
                    return {
                        "Refined CRF Name": str(parsed2.get("CRF", "")).strip(),
                        "Rationale": str(parsed2.get("Rationale", "")).strip(),
                        "Parsed Full Response": json.dumps(parsed2, ensure_ascii=False),
                        "Parse Status": "PARSED_DOUBLE_JSON",
                        "Parse Error": ""
                    }
            except Exception as e2:
                errors.append(f"double_json: {e2}")

    except Exception as e1:
        errors.append(f"json: {e1}")

    # Attempt 2: Python literal parse
    try:
        literal = ast.literal_eval(raw)

        if isinstance(literal, dict):
            return {
                "Refined CRF Name": str(literal.get("CRF", "")).strip(),
                "Rationale": str(literal.get("Rationale", "")).strip(),
                "Parsed Full Response": json.dumps(literal, ensure_ascii=False),
                "Parse Status": "PARSED_LITERAL_DICT",
                "Parse Error": ""
            }

        if isinstance(literal, str):
            try:
                parsed3 = json.loads(literal)
                if isinstance(parsed3, dict):
                    return {
                        "Refined CRF Name": str(parsed3.get("CRF", "")).strip(),
                        "Rationale": str(parsed3.get("Rationale", "")).strip(),
                        "Parsed Full Response": json.dumps(parsed3, ensure_ascii=False),
                        "Parse Status": "PARSED_LITERAL_TO_JSON",
                        "Parse Error": ""
                    }
            except Exception as e3:
                errors.append(f"literal_to_json: {e3}")

    except Exception as e4:
        errors.append(f"literal: {e4}")

    # Fallback
    return {
        "Refined CRF Name": "",
        "Rationale": "",
        "Parsed Full Response": raw,
        "Parse Status": "PARSE_FAILED",
        "Parse Error": " | ".join(errors)
    }


print("✅ Full Response parser loaded.")

✅ Full Response parser loaded.


In [124]:
# ----------------------------
# Full dataframe prestep runner using Responses API + parsing
# ----------------------------
def run_prestep_responses(df, chunk_size=20, checkpoint_every=20, checkpoint_path=None):
    """
    Run prestep generation across the dataframe using the Responses API.

    Writes:
    - Full Response
    - Prestep Run Status
    - Prestep Attempts
    - Prestep Error
    - Refined CRF Name
    - Rationale
    - Parsed Full Response
    - Parse Status
    - Parse Error
    """
    working_df = df.copy()

    # Initialize output columns if missing
    output_cols = [
        "Full Response",
        "Prestep Run Status",
        "Prestep Attempts",
        "Prestep Error",
        "Refined CRF Name",
        "Rationale",
        "Parsed Full Response",
        "Parse Status",
        "Parse Error"
    ]

    for col in output_cols:
        if col not in working_df.columns:
            working_df[col] = ""

    processed_count = 0

    for start in range(0, len(working_df), chunk_size):
        chunk = working_df.iloc[start:start + chunk_size]

        print(f"\nProcessing rows {start} to {start + len(chunk) - 1}...")

        for idx, row in chunk.iterrows():
            crf_value = str(row[crf_column]) if pd.notna(row[crf_column]) else ""
            var_value = str(row[variable_column]) if pd.notna(row[variable_column]) else ""
            desc_value = str(row[description_column]) if pd.notna(row[description_column]) else ""

            result = call_prestep_responses_with_retry(
                crf_name=crf_value,
                variable_name=var_value,
                description_text=desc_value,
                max_retries=3,
                base_sleep_seconds=2
            )

            # Save raw prestep call outputs
            working_df.at[idx, "Full Response"] = result["Full Response"]
            working_df.at[idx, "Prestep Run Status"] = result["Prestep Run Status"]
            working_df.at[idx, "Prestep Attempts"] = result["Prestep Attempts"]
            working_df.at[idx, "Prestep Error"] = result["Prestep Error"]

            # Parse immediately so downstream steps get clean columns
            parsed = parse_full_response_cell(result["Full Response"])

            working_df.at[idx, "Refined CRF Name"] = parsed["Refined CRF Name"]
            working_df.at[idx, "Rationale"] = parsed["Rationale"]
            working_df.at[idx, "Parsed Full Response"] = parsed["Parsed Full Response"]
            working_df.at[idx, "Parse Status"] = parsed["Parse Status"]
            working_df.at[idx, "Parse Error"] = parsed["Parse Error"]

            processed_count += 1

            if checkpoint_path and processed_count % checkpoint_every == 0:
                working_df.to_excel(checkpoint_path, index=False)
                print(f"Checkpoint saved after {processed_count} rows to: {checkpoint_path}")

    return working_df


print("✅ run_prestep_responses() loaded.")

✅ run_prestep_responses() loaded.


In [125]:
# ----------------------------
# Acronym finder + shared JSON parsing helpers
# ----------------------------
def load_acronym_map_from_config(config, section="Acronyms"):
    """
    Load acronym mappings from config_prestep.ini.

    Example config section:
    [Acronyms]
    gad7 = GAD-7
    bpi = BPI
    promis = PROMIS
    """
    if not config.has_section(section):
        print(f"⚠️ No [{section}] section found in config. Acronym finder will return blanks.")
        return {}

    acronym_map = {
        key.strip().lower(): value.strip()
        for key, value in config.items(section)
        if key.strip() and value.strip()
    }

    print(f"✅ Loaded {len(acronym_map)} acronym mappings from [{section}].")
    return acronym_map


ACRONYM_MAP = load_acronym_map_from_config(config)


def detect_heal_cde_acronym(variable_name, acronym_map=ACRONYM_MAP):
    """
    Detect likely HEAL CDE instrument acronym from a variable name.

    Uses acronym mappings loaded from config_prestep.ini.
    Returns the configured acronym label, such as 'BPI' or 'GAD-2'.
    """
    if pd.isna(variable_name):
        return ""

    if not acronym_map:
        return ""

    v = str(variable_name).strip().lower()

    # Compact version helps catch CDE-style variable names:
    # Example: GAD2FeelNervScale -> gad2feelnervscale
    compact = re.sub(r"[^a-z0-9]", "", v)

    # Sort longest first so gad7/gad2/phq9 match before gad/phq
    for key in sorted(acronym_map.keys(), key=len, reverse=True):
        key_compact = re.sub(r"[^a-z0-9]", "", key.lower())

        if compact.startswith(key_compact) or key_compact in compact:
            return acronym_map[key]

    return ""


print("✅ Config-driven acronym finder loaded.")


def parse_json_object_from_text(text):
    """
    Generic helper for extracting a JSON object from model text output.
    Handles:
    - normal JSON objects
    - quoted JSON strings
    - escaped strings
    """
    if text is None:
        return None

    raw = str(text).strip()
    if not raw:
        return None

    # Attempt 1: direct JSON
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, str):
            parsed2 = json.loads(parsed)
            if isinstance(parsed2, dict):
                return parsed2
    except Exception:
        pass

    # Attempt 2: Python literal -> maybe dict or JSON string
    try:
        literal = ast.literal_eval(raw)
        if isinstance(literal, dict):
            return literal
        if isinstance(literal, str):
            parsed3 = json.loads(literal)
            if isinstance(parsed3, dict):
                return parsed3
    except Exception:
        pass

    return None


print("✅ Acronym finder + shared JSON helpers loaded.")

✅ Loaded 19 acronym mappings from [Acronyms].
✅ Config-driven acronym finder loaded.
✅ Acronym finder + shared JSON helpers loaded.


In [126]:
# ----------------------------
# Responses API form harmonizer
# ----------------------------
def batcher(seq, size=20):
    """Yield successive size-sized chunks from seq."""
    for pos in range(0, len(seq), size):
        yield seq[pos:pos + size]


def call_form_harmonizer_responses(batch):
    """
    Call the Responses API to harmonize a batch of refined CRF names.
    Expects a JSON object with a top-level 'mapping' key.
    """
    user_input = json.dumps(batch, ensure_ascii=False, indent=2)

    response = client.responses.create(
        model=HARMONIZER_MODEL,
        instructions=form_harmonizer_instruction,
        input=user_input
    )

    raw_text = response.output_text.strip()
    parsed = parse_json_object_from_text(raw_text)

    if not parsed:
        raise ValueError(f"Could not parse harmonizer output as JSON.\nRaw output:\n{raw_text}")

    if "mapping" in parsed and isinstance(parsed["mapping"], dict):
        mapping = parsed["mapping"]
    elif isinstance(parsed, dict):
        mapping = parsed
    else:
        raise ValueError(f"Harmonizer output JSON did not contain a usable mapping.\nParsed output:\n{parsed}")

    return mapping, raw_text


def call_form_harmonizer_with_retry(batch, max_retries=3, base_sleep_seconds=2):
    """
    Safe wrapper around the Responses API harmonizer call.
    """
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            mapping, raw_text = call_form_harmonizer_responses(batch)
            return {
                "mapping": mapping,
                "raw_harmonizer_response": raw_text,
                "harmonizer_status": "Reviewed",
                "harmonizer_attempts": attempt,
                "harmonizer_error": ""
            }

        except Exception as e:
            last_error = str(e)
            print(f"[warning] harmonizer failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "mapping": {},
        "raw_harmonizer_response": "",
        "harmonizer_status": "ERROR",
        "harmonizer_attempts": max_retries,
        "harmonizer_error": last_error
    }


def run_form_harmonizer(refined_df, batch_size=20):
    """
    Harmonize refined CRF names into canonical CRF names using batched Responses API calls.
    """
    working_df = refined_df.copy()

    # Build and dedupe the payload
    seen = set()
    unique_entries = []

    for orig, rat in zip(working_df["Refined CRF Name"], working_df["Rationale"]):
        key = (_norm_value(orig), _norm_value(rat))
        if key not in seen:
            seen.add(key)
            unique_entries.append({
                "original": _norm_value(orig),
                "rationale": _norm_value(rat)
            })

    if not unique_entries:
        working_df["Canonical CRF Name"] = working_df["Refined CRF Name"]
        print("[Harmonizer] No entries to harmonize, using identity mapping.")
        return working_df

    combined_mapping = {}

    for batch_num, batch in enumerate(batcher(unique_entries, size=batch_size), start=1):
        print(f"\n[Harmonizer] Sending batch {batch_num} of {len(batch)}:")
        for entry in batch:
            print("   ", entry)

        result = call_form_harmonizer_with_retry(batch, max_retries=3, base_sleep_seconds=2)
        mapping = result["mapping"]

        if not mapping:
            print("[Harmonizer] Empty mapping returned; defaulting this batch to identity mapping.")
            mapping = {entry["original"]: entry["original"] for entry in batch}

        print("\n[Harmonizer] Parsed mapping (original → harmonized):")
        for orig, canon in mapping.items():
            print(f"   '{orig}' -> '{canon}'")

        combined_mapping.update(mapping)

    working_df["Canonical CRF Name"] = working_df["Refined CRF Name"].map(
        lambda x: combined_mapping.get(_norm_value(x), _norm_value(x))
    )

    print("\n[Harmonizer] Final Canonical CRF Name results:")
    print(
        working_df[["Refined CRF Name", "Canonical CRF Name"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return working_df


def _norm_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


print("✅ Responses API form harmonizer loaded.")

✅ Responses API form harmonizer loaded.


In [127]:
# ----------------------------
# Responses API HEAL Core CRF matcher
# ----------------------------
def build_heal_match_input(full_prestep_response, acronym_hint=None):
    """
    Build the user input for the HEAL Core CRF matching step.
    """
    acronym_block = ""
    if acronym_hint:
        acronym_block = f"\nAcronym hint detected from variable name: {acronym_hint}\n"

    return (
        f"Prestep output:\n{full_prestep_response}\n"
        f"{acronym_block}\n"
        "Please respond in strict JSON with keys "
        "\"heal_core_crf\", \"confidence\", and \"rationale\". "
        "Do not wrap in markdown or add any extra fields."
    ).strip()


def call_heal_match_responses(full_prestep_response, acronym_hint=None):
    """
    Single Responses API call for HEAL Core CRF matching.
    Returns a parsed dictionary.
    """
    user_input = build_heal_match_input(full_prestep_response, acronym_hint)

    response = client.responses.create(
        model=MATCHING_MODEL,
        instructions=matching_instruction,
        input=user_input
    )

    raw_text = response.output_text.strip()
    parsed = parse_json_object_from_text(raw_text)

    if not parsed:
        raise ValueError(f"Could not parse HEAL match output as JSON.\nRaw output:\n{raw_text}")

    return parsed, raw_text


def call_heal_match_with_retry(full_prestep_response, acronym_hint=None, max_retries=3, base_sleep_seconds=2):
    """
    Safe wrapper around the Responses API HEAL Core CRF match call.
    """
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            parsed, raw_text = call_heal_match_responses(full_prestep_response, acronym_hint)

            match = str(parsed.get("heal_core_crf", "No CRF match")).strip()
            conf = str(parsed.get("confidence", "")).strip()
            rationale = str(parsed.get("rationale", "")).strip()

            return {
                "HEAL Core CRF Match": match,
                "Prestep CRF Confidence": conf,
                "Match Rationale": rationale,
                "Raw HEAL Match Response": raw_text,
                "HEAL Match Status": "Reviewed",
                "HEAL Match Attempts": attempt,
                "HEAL Match Error": ""
            }

        except Exception as e:
            last_error = str(e)
            print(f"[warning] HEAL match failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "HEAL Core CRF Match": "No CRF match",
        "Prestep CRF Confidence": "",
        "Match Rationale": "",
        "Raw HEAL Match Response": "",
        "HEAL Match Status": "ERROR",
        "HEAL Match Attempts": max_retries,
        "HEAL Match Error": last_error
    }


def run_heal_match(df, chunk_size=20):
    """
    Run HEAL Core CRF matching over the dataframe.

    Requires:
    - Full Response
    - CDE Acronym Finder

    Optional:
    - Parse Status
      If present, rows with parsing failures will be skipped before calling the LLM.
    """
    working_df = df.copy()

    output_cols = [
        "HEAL Core CRF Match",
        "Prestep CRF Confidence",
        "Match Rationale",
        "Raw HEAL Match Response",
        "HEAL Match Status",
        "HEAL Match Attempts",
        "HEAL Match Error"
    ]

    for col in output_cols:
        if col not in working_df.columns:
            working_df[col] = ""

    for start in range(0, len(working_df), chunk_size):
        chunk = working_df.iloc[start:start + chunk_size]
        print(f"\n[HEAL Match] Processing rows {start} to {start + len(chunk) - 1}...")

        for idx, row in chunk.iterrows():
            full_response = str(row["Full Response"]) if pd.notna(row["Full Response"]) else ""
            acronym_hint = str(row["CDE Acronym Finder"]) if pd.notna(row["CDE Acronym Finder"]) else ""

            # ------------------------------------------------------------
            # Skip logic:
            # Do not call the LLM if the prestep response is empty or failed parsing.
            # This saves API calls and makes the audit trail cleaner.
            # ------------------------------------------------------------
            parse_status = str(row["Parse Status"]).strip() if "Parse Status" in working_df.columns and pd.notna(row["Parse Status"]) else ""

            if not full_response.strip():
                working_df.at[idx, "HEAL Core CRF Match"] = "No CRF match"
                working_df.at[idx, "Prestep CRF Confidence"] = ""
                working_df.at[idx, "Match Rationale"] = "Skipped because Full Response was empty."
                working_df.at[idx, "Raw HEAL Match Response"] = ""
                working_df.at[idx, "HEAL Match Status"] = "SKIPPED_EMPTY_PRESTEP"
                working_df.at[idx, "HEAL Match Attempts"] = 0
                working_df.at[idx, "HEAL Match Error"] = "Full Response was empty."
                continue

            if parse_status.upper() in ["PARSE_FAILED", "FAILED", "ERROR"]:
                working_df.at[idx, "HEAL Core CRF Match"] = "No CRF match"
                working_df.at[idx, "Prestep CRF Confidence"] = ""
                working_df.at[idx, "Match Rationale"] = "Skipped because prestep output could not be parsed."
                working_df.at[idx, "Raw HEAL Match Response"] = ""
                working_df.at[idx, "HEAL Match Status"] = "SKIPPED_PARSE_FAILED"
                working_df.at[idx, "HEAL Match Attempts"] = 0
                working_df.at[idx, "HEAL Match Error"] = f"Parse Status was {parse_status}."
                continue

            # ------------------------------------------------------------
            # Only rows that pass the skip checks are sent to the LLM.
            # ------------------------------------------------------------
            result = call_heal_match_with_retry(
                full_prestep_response=full_response,
                acronym_hint=acronym_hint,
                max_retries=3,
                base_sleep_seconds=2
            )

            working_df.at[idx, "HEAL Core CRF Match"] = result["HEAL Core CRF Match"]
            working_df.at[idx, "Prestep CRF Confidence"] = result["Prestep CRF Confidence"]
            working_df.at[idx, "Match Rationale"] = result["Match Rationale"]
            working_df.at[idx, "Raw HEAL Match Response"] = result["Raw HEAL Match Response"]
            working_df.at[idx, "HEAL Match Status"] = result["HEAL Match Status"]
            working_df.at[idx, "HEAL Match Attempts"] = result["HEAL Match Attempts"]
            working_df.at[idx, "HEAL Match Error"] = result["HEAL Match Error"]

    # Clean final output display:
    # If there is no HEAL Core CRF Match, blank out the prestep confidence field.
    mask_no_crf = working_df["HEAL Core CRF Match"].astype(str).str.strip().eq("No CRF match")
    working_df.loc[mask_no_crf, "Prestep CRF Confidence"] = ""

    return working_df


print("✅ Responses API HEAL Core CRF matcher loaded.")

✅ Responses API HEAL Core CRF matcher loaded.


In [128]:
# ----------------------------
# Prestep orchestrator: acronym finder + prestep + harmonizer + HEAL match
# Produces prestep_df for downstream CDE matching
# ----------------------------
full_input_df = pd.read_excel(input_file, sheet_name=input_worksheet).copy()

# Step 1: keep only the columns needed for the CRF workflow
data_dict_df = full_input_df[[crf_column, variable_column, description_column]].copy()

# Step 2: acronym finder
data_dict_df["CDE Acronym Finder"] = data_dict_df[variable_column].apply(detect_heal_cde_acronym)

print("\n[CDE Acronym Finder] Preview:")
print(data_dict_df[[variable_column, "CDE Acronym Finder"]].head(10).to_string(index=False))

# Step 3: prestep refinement via Responses API
prestep_checkpoint_file = Path(output_file).with_name(
    Path(output_file).stem + "_prestep_checkpoint.xlsx"
)

refined_df = run_prestep_responses(
    data_dict_df,
    chunk_size=20,
    checkpoint_every=20,
    checkpoint_path=prestep_checkpoint_file
)

print("\n[Prestep] Sample after refinement:")
print(
    refined_df[
        [crf_column, variable_column, "Refined CRF Name", "Rationale", "Prestep Run Status", "Parse Status"]
    ].head(10).to_string(index=False)
)

# Step 4: harmonize refined CRF names
harmonized_df = run_form_harmonizer(refined_df, batch_size=20)

print("\n[Harmonizer] Sample after canonical naming:")
print(
    harmonized_df[
        ["Refined CRF Name", "Canonical CRF Name"]
    ].drop_duplicates().head(20).to_string(index=False)
)

# Step 5: merge harmonized fields back into the full input
enhanced_df = full_input_df.join(
    harmonized_df[[
        "CDE Acronym Finder",
        "Full Response",
        "Refined CRF Name",
        "Rationale",
        "Parsed Full Response",
        "Parse Status",
        "Parse Error",
        "Prestep Run Status",
        "Prestep Attempts",
        "Prestep Error",
        "Canonical CRF Name"
    ]],
    how="left"
)

# Step 6: HEAL Core CRF match
final_df = run_heal_match(enhanced_df, chunk_size=20)

print("\n[HEAL Match] Sample after matching:")
print(
    final_df[
        [
            variable_column,
            "Refined CRF Name",
            "Canonical CRF Name",
            "HEAL Core CRF Match",
            "Prestep CRF Confidence",
            "Match Rationale",
            "HEAL Match Status"
        ]
    ].head(10).to_string(index=False)
)

# Step 7: reusable dataframe for downstream CDE matching
prestep_df = final_df.copy()

print("\n✅ Prestep workflow complete.")
print(f"Rows prepared for CDE matching: {len(prestep_df)}")
print("Created dataframe: prestep_df")
print(f"Checkpoint file: {prestep_checkpoint_file}")


[CDE Acronym Finder] Preview:
                   name CDE Acronym Finder
           imh_birthhcr                   
            imh_birthlt                   
             imhbirthwt                   
      BPIAvgPainRtngScl                BPI
  BPICurrentPainRtngScl                BPI
BPILeastPnLst24HRtngScl                BPI
BPIWrstPnLast24HRtngScl                BPI
        GAD2FeelNervScl              GAD-2
      GAD2NotStopWryScl              GAD-2
     GAD7EasyAnnoyedScl              GAD-7

Processing rows 0 to 13...

[Prestep] Sample after refinement:
                          section                    name                        Refined CRF Name                                                                                                                                                                                                  Rationale Prestep Run Status Parse Status
a_infant_medical_history_01_month            imh_birthhcr                  Infant Medical History  

In [129]:
# ----------------------------
# Bridge cell: use prestep_df as input for v4-style CDE matching
# ----------------------------

# Confirm prestep_df exists
if "prestep_df" not in globals():
    raise NameError(
        "prestep_df does not exist yet. Run the prestep orchestrator cell before this bridge cell."
    )

# Study dataframe for downstream CDE matching
# This replaces reading the old prestep output Excel file / EnhancedDD sheet.
study_df = prestep_df.copy()

print("✅ Using prestep_df as the study dataframe for CDE matching.")
print(f"Rows in study_df: {len(study_df)}")
print(f"Columns in study_df: {len(study_df.columns)}")


# ----------------------------
# Helper functions for config values
# ----------------------------

def get_config_value(section, option, fallback=None):
    """
    Safely get a string value from config.
    """
    if config.has_section(section) and config.has_option(section, option):
        return config.get(section, option)
    return fallback


def get_config_bool(section, option, fallback=False):
    """
    Safely get a boolean value from config.
    Accepts values like true/false, yes/no, 1/0.
    """
    if config.has_section(section) and config.has_option(section, option):
        return config.getboolean(section, option)
    return fallback


def get_config_int(section, option, fallback=0):
    """
    Safely get an integer value from config.
    """
    if config.has_section(section) and config.has_option(section, option):
        return config.getint(section, option)
    return fallback


# ----------------------------
# Study variable columns
# These should match the original input DD columns now preserved in prestep_df.
# ----------------------------

FORM_NAME_COLUMN = get_config_value("Columns", "crf_column", fallback="Form Name")
VARIABLE_NAME_COLUMN = get_config_value("Columns", "variable_column", fallback="Variable / Field Name")
FIELD_LABEL_COLUMN = get_config_value("Columns", "description_column", fallback="Field Label")
ENCODING_COLUMN = get_config_value("Columns", "encoding_column", fallback="Choices, Calculations, OR Slider Labels")

# CRF signal created during prestep
EXPECTED_CRF_COL = get_config_value("Columns", "expected_crf_column", fallback="HEAL Core CRF Match")


# ----------------------------
# HEAL CDE knowledge base
# ----------------------------

CDE_FILE = get_config_value("Files", "heal_cde_knowledge_base_file", fallback=None)

if not CDE_FILE:
    raise ValueError(
        "Missing HEAL CDE knowledge base path. Add heal_cde_knowledge_base_file under [Files] in config_prestep.ini."
    )

CDE_FILE = Path(CDE_FILE)

if not CDE_FILE.exists():
    raise FileNotFoundError(f"HEAL CDE knowledge base file not found: {CDE_FILE}")

print(f"✅ HEAL CDE knowledge base file found: {CDE_FILE}")


# ----------------------------
# Output directory
# ----------------------------

output_path = Path(output_file)
OUTPUT_DIR = output_path.parent

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("📁 Output directory:", OUTPUT_DIR.resolve())


# ----------------------------
# CRF-aware concept retrieval settings
# ----------------------------

PREFER_EXPECTED_CRF_FOR_CONCEPT = get_config_bool(
    "ConceptRetrieval",
    "prefer_expected_crf_for_concept",
    fallback=True
)

CRF_MAP_THRESHOLD = get_config_int(
    "ConceptRetrieval",
    "crf_map_threshold",
    fallback=85
)

CRF_CONCEPT_BONUS = get_config_int(
    "ConceptRetrieval",
    "crf_concept_bonus",
    fallback=8
)

TOP_N_CONCEPT_CANDIDATES = get_config_int(
    "ConceptRetrieval",
    "top_n_concept_candidates",
    fallback=3
)


# ----------------------------
# Stage 1: concept retrieval thresholds
# ----------------------------

CONCEPT_THRESHOLDS = {
    "high": get_config_int("ConceptThresholds", "high", fallback=75),
    "medium": get_config_int("ConceptThresholds", "medium", fallback=50),
    "minimum_score": get_config_int("ConceptThresholds", "minimum_score", fallback=25)
}


# ----------------------------
# Stage 2: implementation fidelity thresholds
# ----------------------------

FIDELITY_THRESHOLDS = {
    "high": get_config_int("FidelityThresholds", "high", fallback=80),
    "medium": get_config_int("FidelityThresholds", "medium", fallback=50)
}


# ----------------------------
# Sanity checks
# ----------------------------

required_study_cols = [
    FORM_NAME_COLUMN,
    VARIABLE_NAME_COLUMN,
    FIELD_LABEL_COLUMN,
    ENCODING_COLUMN,
    EXPECTED_CRF_COL
]

missing_study_cols = [col for col in required_study_cols if col not in study_df.columns]

if missing_study_cols:
    print("⚠️ Missing expected columns in study_df:")
    for col in missing_study_cols:
        print(f"   - {col}")
else:
    print("✅ All expected study columns are present in study_df.")


print("\n✅ Bridge setup complete.")
print("Ready for v4-style CDE matching cells.")
print("\nKey objects created:")
print("   study_df")
print("   CDE_FILE")
print("   OUTPUT_DIR")
print("   FORM_NAME_COLUMN")
print("   VARIABLE_NAME_COLUMN")
print("   FIELD_LABEL_COLUMN")
print("   ENCODING_COLUMN")
print("   EXPECTED_CRF_COL")
print("   CONCEPT_THRESHOLDS")
print("   FIDELITY_THRESHOLDS")
print("   TOP_N_CONCEPT_CANDIDATES")

✅ Using prestep_df as the study dataframe for CDE matching.
Rows in study_df: 14
Columns in study_df: 48
✅ HEAL CDE knowledge base file found: KnowledgeBase\Compiled_CORE_CDEs list_English_one sheet_as of 2025-01-28.xlsx
📁 Output directory: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out
✅ All expected study columns are present in study_df.

✅ Bridge setup complete.
Ready for v4-style CDE matching cells.

Key objects created:
   study_df
   CDE_FILE
   OUTPUT_DIR
   FORM_NAME_COLUMN
   VARIABLE_NAME_COLUMN
   FIELD_LABEL_COLUMN
   ENCODING_COLUMN
   EXPECTED_CRF_COL
   CONCEPT_THRESHOLDS
   FIDELITY_THRESHOLDS
   TOP_N_CONCEPT_CANDIDATES


In [130]:
# ============================================================
# STAGE 1 + STAGE 2 SCORING HELPERS
# Uses config-driven concept weights and thresholds.
#
# Requires bridge cell variables:
# - config
# - CRF_MAP_THRESHOLD
# - CRF_CONCEPT_BONUS
# - PREFER_EXPECTED_CRF_FOR_CONCEPT
# - CONCEPT_THRESHOLDS
# - FIDELITY_THRESHOLDS
# ============================================================

# ----------------------------
# Helper: safely read float values from config
# ----------------------------

def get_config_float(section, option, fallback=0.0):
    """
    Safely get a float value from config.
    """
    if config.has_section(section) and config.has_option(section, option):
        return config.getfloat(section, option)
    return fallback


# ----------------------------
# Concept scoring weights
# ----------------------------
# These control how much each row feature contributes to Stage 1 concept retrieval.

VAR_NAME_WEIGHT = get_config_float(
    "ConceptWeights",
    "var_name_weight",
    fallback=0.25
)

FIELD_LABEL_WEIGHT = get_config_float(
    "ConceptWeights",
    "field_label_weight",
    fallback=0.50
)

FORM_CONTEXT_WEIGHT = get_config_float(
    "ConceptWeights",
    "form_context_weight",
    fallback=0.15
)

CRF_CONTEXT_WEIGHT = get_config_float(
    "ConceptWeights",
    "crf_context_weight",
    fallback=0.10
)

print("✅ Concept weights loaded:")
print(f"   VAR_NAME_WEIGHT: {VAR_NAME_WEIGHT}")
print(f"   FIELD_LABEL_WEIGHT: {FIELD_LABEL_WEIGHT}")
print(f"   FORM_CONTEXT_WEIGHT: {FORM_CONTEXT_WEIGHT}")
print(f"   CRF_CONTEXT_WEIGHT: {CRF_CONTEXT_WEIGHT}")


def normalize_string(s):
    """
    Normalize strings for fuzzy concept/fidelity comparison.
    """
    if pd.isna(s) or s is None:
        return ""

    s = str(s).lower().strip()
    s = re.sub(r"[^a-zA-Z0-9\s=]", " ", s)
    s = re.sub(r"\s+", " ", s)

    return s.strip()


def similarity_score(str1, str2):
    """
    Token-set fuzzy similarity, 0-100.
    """
    str1 = normalize_string(str1)
    str2 = normalize_string(str2)

    if not str1 and not str2:
        return 0

    if not str1 or not str2:
        return 0

    return fuzz.token_set_ratio(str1, str2)


# ----------------------------
# CRF helpers
# ----------------------------

def normalize_crf_label(s):
    """
    Normalize CRF/form labels so slightly different wording can still map together.
    """
    if not isinstance(s, str):
        return ""

    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    junk = {
        "questionnaire",
        "scale",
        "inventory",
        "form",
        "survey",
        "assessment"
    }

    tokens = [t for t in s.split() if t not in junk]

    return " ".join(tokens)


def build_crf_mapper(kb_crf_names):
    """
    Returns a function that maps arbitrary CRF labels to the closest KB CRF Name.
    """
    kb = [
        str(x).strip()
        for x in kb_crf_names
        if pd.notna(x) and str(x).strip()
    ]

    kb_norm = {
        name: normalize_crf_label(name)
        for name in kb
    }

    def map_crf(label):
        if not isinstance(label, str) or not label.strip():
            return None

        norm = normalize_crf_label(label)

        # Exact normalized hit
        for kb_name, kb_norm_name in kb_norm.items():
            if norm == kb_norm_name:
                return kb_name

        # Fuzzy fallback
        best_name = None
        best_score = -1

        for kb_name, kb_norm_name in kb_norm.items():
            score = fuzz.token_set_ratio(norm, kb_norm_name)

            if score > best_score:
                best_name = kb_name
                best_score = score

        return best_name if best_score >= CRF_MAP_THRESHOLD else None

    return map_crf


# ----------------------------
# Stage 1: concept retrieval
# ----------------------------

def concept_similarity_score(
    study_var_name,
    study_field_label,
    study_form_name,
    expected_crf,
    cde_var_name,
    cde_question_text,
    cde_crf_name
):
    """
    Score conceptual closeness between a study row and a HEAL CDE candidate.

    This is the primary selector for the closest HEAL CDE concept.
    Encoding is intentionally excluded here.
    """
    var_score = similarity_score(study_var_name, cde_var_name)
    label_score = similarity_score(study_field_label, cde_question_text)
    form_score = similarity_score(study_form_name, cde_crf_name)
    crf_score = similarity_score(expected_crf, cde_crf_name)

    score = (
        var_score * VAR_NAME_WEIGHT +
        label_score * FIELD_LABEL_WEIGHT +
        form_score * FORM_CONTEXT_WEIGHT +
        crf_score * CRF_CONTEXT_WEIGHT
    )

    # Small bonus if candidate lives inside the expected CRF context
    if (
        PREFER_EXPECTED_CRF_FOR_CONCEPT
        and expected_crf
        and cde_crf_name
        and normalize_crf_label(expected_crf) == normalize_crf_label(cde_crf_name)
    ):
        score += CRF_CONCEPT_BONUS

    return min(score, 100)


# ----------------------------
# Stage 2: implementation fidelity
# ----------------------------

def encoding_fidelity_score(study_encoding, cde_encoding):
    """
    Score how similarly the concept appears to be implemented.

    This does not decide the concept winner.
    It describes implementation closeness after concept retrieval.
    """
    study_encoding = normalize_string(study_encoding)
    cde_encoding = normalize_string(cde_encoding)

    if not study_encoding and not cde_encoding:
        return 0

    if not study_encoding or not cde_encoding:
        return 0

    return similarity_score(study_encoding, cde_encoding)


def classify_concept_score(score):
    """
    Convert a concept similarity score into a review-friendly label.
    """
    if pd.isna(score):
        return "No score"

    if score >= CONCEPT_THRESHOLDS["high"]:
        return "High concept match"

    elif score >= CONCEPT_THRESHOLDS["medium"]:
        return "Possible concept match"

    elif score >= CONCEPT_THRESHOLDS["minimum_score"]:
        return "Weak concept match"

    return "No confident concept match"


def classify_fidelity_score(score):
    """
    Convert an encoding fidelity score into a review-friendly label.
    """
    if pd.isna(score):
        return "No score"

    if score >= FIDELITY_THRESHOLDS["high"]:
        return "Closely implemented"

    elif score >= FIDELITY_THRESHOLDS["medium"]:
        return "Concept captured, encoding differs"

    return "Low implementation fidelity"


print("\n✅ Stage 1/Stage 2 scoring helpers loaded.")
print("   - Concept similarity now drives candidate selection")
print("   - Encoding fidelity is now a separate descriptive score")
print("   - Concept weights are loaded from config")

✅ Concept weights loaded:
   VAR_NAME_WEIGHT: 0.25
   FIELD_LABEL_WEIGHT: 0.5
   FORM_CONTEXT_WEIGHT: 0.15
   CRF_CONTEXT_WEIGHT: 0.1

✅ Stage 1/Stage 2 scoring helpers loaded.
   - Concept similarity now drives candidate selection
   - Encoding fidelity is now a separate descriptive score
   - Concept weights are loaded from config


In [131]:
# ============================================================
# SELECTIVE STAGE 1 CONCEPT-FAMILY VIEW
# Creates a concept-retrieval view of the HEAL CDE knowledge base.
#
# Requires:
# - cde_df later when build_stage1_concept_view(cde_df) is called
# - config
# ============================================================

def load_concept_family_whitelist_from_config(config, section="ConceptFamilies"):
    """
    Load concept families to collapse during Stage 1 concept retrieval.

    Expected config format:
    [ConceptFamilies]
    family_1 = Demographics | Race

    Returns:
    set of tuples, e.g. {("Demographics", "Race")}
    """
    default_whitelist = {
        ("Demographics", "Race"),
    }

    if not config.has_section(section):
        print(f"ℹ️ No [{section}] section found in config. Using default concept family whitelist.")
        return default_whitelist

    whitelist = set()

    for _, value in config.items(section):
        if not value or "|" not in value:
            continue

        parts = [p.strip() for p in value.split("|")]

        if len(parts) == 2 and all(parts):
            whitelist.add((parts[0], parts[1]))

    if not whitelist:
        print(f"ℹ️ [{section}] section found, but no valid families listed. Using default whitelist.")
        return default_whitelist

    print(f"✅ Loaded {len(whitelist)} concept family whitelist item(s) from [{section}].")
    return whitelist


# Only collapse known parent-concept families here.
# Start tiny and safe.
CONCEPT_FAMILY_WHITELIST = load_concept_family_whitelist_from_config(config)


def safe_unique_join(series, max_items=None, separator=" ; "):
    """
    Clean, deduplicate, optionally limit, and join values from a pandas Series.
    """
    values = sorted({
        str(v).strip()
        for v in series.dropna().tolist()
        if str(v).strip()
    })

    if max_items is not None:
        values = values[:max_items]

    return separator.join(values)


def build_stage1_concept_view(cde_df):
    """
    Build a Stage 1 concept-retrieval table.

    Behavior:
    - For whitelisted parent concepts, such as Demographics | Race,
      collapse child rows into one synthetic concept row.
    - For everything else, keep the original KB rows as-is.

    Returns a dataframe that still has the same key columns used downstream:
    - CRF Name
    - CDE Name
    - Variable Name
    - Definition
    - Short Description
    - Additional Notes (Question Text)
    - PV Description
    """
    required_cols = [
        "CRF Name",
        "CDE Name",
        "Variable Name",
        "Definition",
        "Short Description",
        "Additional Notes (Question Text)",
        "PV Description"
    ]

    missing_cols = [col for col in required_cols if col not in cde_df.columns]

    if missing_cols:
        raise ValueError(
            "The HEAL CDE knowledge base is missing required columns: "
            + ", ".join(missing_cols)
        )

    rows = []

    # Group by CRF + CDE Name so we can selectively collapse families.
    grouped = cde_df.groupby(["CRF Name", "CDE Name"], dropna=False)

    for (crf_name, cde_name), grp in grouped:
        key = (str(crf_name).strip(), str(cde_name).strip())

        # ------------------------------------------------------------
        # FAMILY MODE:
        # Collapse whitelisted child rows into one synthetic parent concept.
        # ------------------------------------------------------------
        if key in CONCEPT_FAMILY_WHITELIST:
            member_vars = sorted({
                str(v).strip()
                for v in grp["Variable Name"].dropna().tolist()
                if str(v).strip()
            })

            definitions_full = safe_unique_join(grp["Definition"])
            short_descs_full = safe_unique_join(grp["Short Description"])

            definitions_preview = safe_unique_join(grp["Definition"], max_items=2)
            question_text_preview = safe_unique_join(
                grp["Additional Notes (Question Text)"],
                max_items=2
            )

            family_question_parts = [
                f"CDE Name: {cde_name}",
                f"Member variables: {', '.join(member_vars)}" if member_vars else "",
                f"Definitions: {definitions_preview}" if definitions_preview else "",
                f"Question text: {question_text_preview}" if question_text_preview else ""
            ]

            family_question_text = " | ".join([
                part for part in family_question_parts
                if part
            ])

            rows.append({
                "CRF Name": crf_name,
                "CDE Name": cde_name,
                "Variable Name": cde_name,
                "Definition": definitions_full,
                "Short Description": short_descs_full,
                "Additional Notes (Question Text)": family_question_text,
                "PV Description": "",
                "Stage1 Concept Mode": "family",
                "Stage1 Member Variables": ", ".join(member_vars)
            })

        # ------------------------------------------------------------
        # ROW MODE:
        # Keep original KB rows.
        # ------------------------------------------------------------
        else:
            for _, r in grp.iterrows():
                rows.append({
                    "CRF Name": r.get("CRF Name", ""),
                    "CDE Name": r.get("CDE Name", ""),
                    "Variable Name": r.get("Variable Name", ""),
                    "Definition": r.get("Definition", ""),
                    "Short Description": r.get("Short Description", ""),
                    "Additional Notes (Question Text)": r.get("Additional Notes (Question Text)", ""),
                    "PV Description": r.get("PV Description", ""),
                    "Stage1 Concept Mode": "row",
                    "Stage1 Member Variables": ""
                })

    concept_df = pd.DataFrame(rows)

    print("✅ Stage 1 concept view created.")
    print(f"   Original KB rows: {len(cde_df)}")
    print(f"   Stage 1 concept rows: {len(concept_df)}")

    family_rows = concept_df[concept_df["Stage1 Concept Mode"] == "family"]

    if not family_rows.empty:
        print("\n🧩 Family-mode concepts currently enabled:")
        display(family_rows[["CRF Name", "CDE Name", "Variable Name", "Stage1 Member Variables"]])
    else:
        print("\n🧩 No family-mode concepts enabled yet.")

    return concept_df


print("✅ Stage 1 concept-family helper loaded.")

✅ Loaded 4 concept family whitelist item(s) from [ConceptFamilies].
✅ Stage 1 concept-family helper loaded.


In [132]:
def main(dry_run=False, save_output=False):
    """
    Main function to run the v5 VLMD CDE matching process.

    v5 behavior:
    - Uses study_df created from prestep_df in the bridge cell.
    - Uses config-loaded column names and thresholds.
    - Passes study_df into compare_encodings().
    - If save_output=False, creates DDtoCRFtoVLMDCDE_df but does not save Excel yet.
    """

    print("🚀 Starting HEAL CDE Variable Level Metadata Matching")
    print("   Input source: study_df created from prestep_df")

    # ----------------------------
    # Threshold summary
    # ----------------------------
    print(
        f"\n📋 Concept retrieval thresholds: "
        f"High ≥{CONCEPT_THRESHOLDS['high']}%, "
        f"Possible ≥{CONCEPT_THRESHOLDS['medium']}%, "
        f"Minimum ≥{CONCEPT_THRESHOLDS['minimum_score']}%"
    )

    print(
        f"📋 Encoding fidelity thresholds: "
        f"High ≥{FIDELITY_THRESHOLDS['high']}%, "
        f"Medium ≥{FIDELITY_THRESHOLDS['medium']}%"
    )

    # ----------------------------
    # CRF-aware retrieval summary
    # ----------------------------
    print("\n🧭 Concept retrieval settings:")
    print(f"   Expected CRF column        : {EXPECTED_CRF_COL}")
    print(f"   Prefer expected CRF        : {PREFER_EXPECTED_CRF_FOR_CONCEPT}")
    print(f"   CRF mapping threshold      : {CRF_MAP_THRESHOLD}")
    print(f"   CRF concept bonus          : {CRF_CONCEPT_BONUS}")
    print(f"   Top N concept candidates   : {TOP_N_CONCEPT_CANDIDATES}")
    print("   Potential Match 2/3 are alternate concept candidates.")

    # ----------------------------
    # Preflight checks
    # ----------------------------
    if "study_df" not in globals():
        raise NameError(
            "study_df does not exist. Run the bridge cell after creating prestep_df."
        )

    if study_df.empty:
        raise ValueError("study_df is empty. Check the prestep and bridge cells.")

    required_cols = [
        VARIABLE_NAME_COLUMN,
        FIELD_LABEL_COLUMN,
        ENCODING_COLUMN,
        FORM_NAME_COLUMN
    ]

    missing_cols = [col for col in required_cols if col not in study_df.columns]

    if missing_cols:
        raise ValueError(
            "study_df is missing required study columns: "
            + ", ".join(missing_cols)
        )

    if EXPECTED_CRF_COL not in study_df.columns:
        print(
            f"\n⚠️ Warning: Expected CRF column '{EXPECTED_CRF_COL}' not found in study_df.\n"
            "   Concept retrieval will continue without CRF preference.\n"
        )
    else:
        nonblank_crf_count = (
            study_df[EXPECTED_CRF_COL]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        print(
            f"\n✅ Expected CRF column found: {EXPECTED_CRF_COL}"
            f"\n   Nonblank expected CRF values: {nonblank_crf_count} / {len(study_df)}"
        )

    print("\n[study_df preview]")

    display_cols = [
        FORM_NAME_COLUMN,
        VARIABLE_NAME_COLUMN,
        FIELD_LABEL_COLUMN,
        ENCODING_COLUMN,
        EXPECTED_CRF_COL
    ]

    display_cols = [col for col in display_cols if col in study_df.columns]

    display(study_df[display_cols].head(5))

    # ----------------------------
    # Run CDE matching
    # ----------------------------
    output_file_created = compare_encodings(
        study_df=study_df,
        encoding_column=ENCODING_COLUMN,
        field_label_column=FIELD_LABEL_COLUMN,
        cde_file=CDE_FILE,
        variable_name_column=VARIABLE_NAME_COLUMN,
        form_name_column=FORM_NAME_COLUMN,
        expected_crf_column=EXPECTED_CRF_COL,
        dry_run=dry_run,
        save_output=save_output
    )

    if dry_run:
        print("\n🧪 Dry run complete. No output file written.")
        return

    if not output_file_created:
        print("\n✅ No Excel output file was created because save_output=False.")
        print("   Continue using DDtoCRFtoVLMDCDE_df for the next step.")
        return

    print("\n✅ Output file created:")
    print("   📄", os.path.abspath(output_file_created))

    # ----------------------------
    # Apply final Excel formatting
    # ----------------------------
    try:
        apply_color_coding(output_file_created)
    except Exception as e:
        print(f"\n⚠️ Color coding failed. File is still valid: {e}")

    print("\n🎉 VLMD CDE matching complete!")
    print(f"📊 Results saved to: {output_file_created}")
    print("📋 Review concept matches, encoding fidelity, and the Low_Confidence_Analysis sheet.")

In [133]:
# ============================================================
# STAGE A DISPLAY POLICY HELPERS
# ============================================================

def has_usable_crf_anchor(raw_crf_value):
    """
    True if the row has a real HEAL Core CRF Match we can use as an anchor.

    Examples treated as unusable:
    - blank
    - NaN
    - "No CRF match"
    """
    if pd.isna(raw_crf_value):
        return False

    value = str(raw_crf_value).strip()

    if value == "" or value.lower() == "no crf match":
        return False

    return True


def classify_stage_a_display_status(
    concept_score,
    has_anchor,
    expected_kb=None,
    best_crf_name=None,
    retrieval_mode=None
):
    """
    Final display status policy for Stage A.

    Rules:
    - Anchored rows are only allowed to be High if:
        1) score is high
        2) winner stayed inside the expected CRF lane
        3) retrieval came from anchored_crf_first
    - Anchored rows otherwise fall back to Possible / Weak
    - Unanchored rows are capped at Possible
    """
    if pd.isna(concept_score):
        return "Weak concept match"

    score = float(concept_score)

    if has_anchor:
        stayed_in_anchor_lane = (
            expected_kb is not None
            and best_crf_name is not None
            and str(best_crf_name).strip() == str(expected_kb).strip()
            and str(retrieval_mode).strip() == "anchored_crf_first"
        )

        if score >= CONCEPT_THRESHOLDS["high"] and stayed_in_anchor_lane:
            return "High concept match"

        if score >= CONCEPT_THRESHOLDS["medium"]:
            return "Possible concept match"

        return "Weak concept match"

    # Unanchored rows can never be High
    if score >= CONCEPT_THRESHOLDS["medium"]:
        return "Possible concept match"

    return "Weak concept match"


def should_blank_main_concept(display_status):
    """
    If True, later blank out the main concept columns in VLMD_Results.
    """
    return str(display_status).strip() == "Weak concept match"


print("✅ Stage A display policy helpers loaded.")
print("   - usable CRF anchor detector")
print("   - anchored rows eligible for High")
print("   - unanchored rows capped at Possible")
print("   - weak rows marked for blanking later")

✅ Stage A display policy helpers loaded.
   - usable CRF anchor detector
   - anchored rows eligible for High
   - unanchored rows capped at Possible
   - weak rows marked for blanking later


In [134]:
# ============================================================
# COPYRIGHT-SENSITIVE FAMILY RULES + VLMD SKIP RULES
# Loads protected primary family rules and skip-VLMD rules
# from config_prestep.ini.
#
# Expected config format:
#
# [ProtectedPrimaryFamilies]
# brief pain inventory (bpi) = brief pain inventory (bpi) | bpi | bpi pain interference
# bpi = brief pain inventory (bpi) | bpi | bpi pain interference
# pcs6 = pcs6 | pcs-6
# pcs-6 = pcs6 | pcs-6
#
# [SkipVLMDMatchingCRFs]
# bpi = true
# brief pain inventory (bpi) = true
# bpi pain interference = true
#
# Important:
# - ProtectedPrimaryFamilies controls primary Best Match blocking.
# - SkipVLMDMatchingCRFs controls whether VLMD matching is skipped entirely.
# ============================================================

def normalize_protected_crf_label(value):
    """
    Normalize CRF labels for protected-family and skip-VLMD comparisons.

    Kept separate from normalize_crf_label() so we do not override
    the broader CRF normalization helper used elsewhere.
    """
    if pd.isna(value) or value is None:
        return ""

    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9\s()\-]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def load_protected_primary_family_rules_from_config(
    config,
    section="ProtectedPrimaryFamilies",
    delimiter="|"
):
    """
    Load protected primary family rules from config.

    Returns:
    {
        "normalized expected crf": {
            "normalized allowed crf 1",
            "normalized allowed crf 2"
        }
    }
    """
    if not config.has_section(section):
        print(f"⚠️ No [{section}] section found in config. No protected primary family rules loaded.")
        return {}

    protected_rules = {}

    for expected_label, allowed_labels_raw in config.items(section):
        expected_norm = normalize_protected_crf_label(expected_label)

        allowed_norms = {
            normalize_protected_crf_label(label)
            for label in str(allowed_labels_raw).split(delimiter)
            if normalize_protected_crf_label(label)
        }

        # Make sure the expected label itself is allowed by default.
        # This prevents accidental self-blocking if the config value is incomplete.
        if expected_norm:
            allowed_norms.add(expected_norm)

        if expected_norm and allowed_norms:
            protected_rules[expected_norm] = allowed_norms

    print(f"✅ Loaded {len(protected_rules)} protected primary family rule(s) from [{section}].")

    if protected_rules:
        print("Protected CRF families:")
        for expected_norm, allowed_norms in protected_rules.items():
            print(f"   - {expected_norm}: {', '.join(sorted(allowed_norms))}")

    return protected_rules


def load_skip_vlmd_matching_crfs_from_config(
    config,
    section="SkipVLMDMatchingCRFs"
):
    """
    Load CRFs where VLMD matching should be skipped entirely.

    Expected config format:
    [SkipVLMDMatchingCRFs]
    bpi = true
    brief pain inventory (bpi) = true
    bpi pain interference = true

    Returns:
    set of normalized CRF labels
    """
    if not config.has_section(section):
        print(f"ℹ️ No [{section}] section found in config. No VLMD skip rules loaded.")
        return set()

    skip_crfs = set()

    for crf_label, should_skip_raw in config.items(section):
        should_skip = str(should_skip_raw).strip().lower() in {
            "true",
            "yes",
            "1",
            "y"
        }

        if should_skip:
            crf_norm = normalize_protected_crf_label(crf_label)

            if crf_norm:
                skip_crfs.add(crf_norm)

    print(f"✅ Loaded {len(skip_crfs)} VLMD skip rule(s) from [{section}].")

    if skip_crfs:
        print("CRFs that will skip VLMD matching:")
        for crf_norm in sorted(skip_crfs):
            print(f"   - {crf_norm}")

    return skip_crfs


# ------------------------------------------------------------
# Load config-driven rule sets
# ------------------------------------------------------------

# Protected primary family rules:
# If the expected CRF is in one of these protected groups,
# only allow primary best-match assignment from the same allowed family.
# Cross-family "proxy" candidates can still be shown, but NOT accepted as Best Match.
PROTECTED_PRIMARY_FAMILY_RULES = load_protected_primary_family_rules_from_config(config)

# VLMD skip rules:
# If the expected CRF is listed here, skip variable-level matching entirely.
# CRF-level identification is retained from prestep_df, but VLMD CDE columns remain blank.
SKIP_VLMD_MATCHING_CRFS = load_skip_vlmd_matching_crfs_from_config(config)


def is_copyright_sensitive_expected_crf(expected_kb):
    """
    True if the expected HEAL Core CRF belongs to a protected primary family.
    """
    expected_norm = normalize_protected_crf_label(expected_kb)
    return expected_norm in PROTECTED_PRIMARY_FAMILY_RULES


def should_skip_vlmd_matching(expected_kb):
    """
    True if VLMD matching should be skipped entirely for this expected CRF.

    Used for copyrighted or unavailable VLMD cases where CRF-level matching
    should be retained, but variable-level CDE matching should not be assigned.
    """
    expected_norm = normalize_protected_crf_label(expected_kb)
    return expected_norm in SKIP_VLMD_MATCHING_CRFS


def is_blocked_primary_candidate(expected_kb, candidate_crf):
    """
    For protected families only:
    - block cross-family candidates from becoming the primary best match
    - still allow them to exist as reviewable proxy suggestions
    """
    expected_norm = normalize_protected_crf_label(expected_kb)
    candidate_norm = normalize_protected_crf_label(candidate_crf)

    if expected_norm not in PROTECTED_PRIMARY_FAMILY_RULES:
        return False

    allowed_family = PROTECTED_PRIMARY_FAMILY_RULES[expected_norm]

    return candidate_norm not in allowed_family


print("\n✅ Copyright-sensitive family helpers loaded.")
print("   - protected families loaded from config")
print("   - VLMD skip rules loaded from config")
print("   - cross-family proxy candidates can be shown but blocked as primary Best Match")
print("   - selected CRFs can skip VLMD matching entirely")

✅ Loaded 6 protected primary family rule(s) from [ProtectedPrimaryFamilies].
Protected CRF families:
   - brief pain inventory (bpi): bpi, brief pain inventory, brief pain inventory (bpi)
   - bpi pain interference: bpi, bpi interference, bpi pain interference, brief pain inventory (bpi), pain interference
   - bpi pain severity: bpi, bpi pain severity, bpi severity, brief pain inventory (bpi), pain severity
   - pcs-6: pain catastrophizing scale 6, pain catastrophizing scale-6, pcs 6, pcs-6, pcs6
   - pcs-13: pain catastrophizing scale 13, pain catastrophizing scale-13, pcs 13, pcs-13, pcs13
   - pedsql inventory: pediatric quality of life inventory, pedsql, pedsql inventory, pedsql pediatric quality of life inventory
✅ Loaded 10 VLMD skip rule(s) from [SkipVLMDMatchingCRFs].
CRFs that will skip VLMD matching:
   - bpi
   - bpi pain interference
   - brief pain inventory (bpi)
   - pcs-13
   - pcs-6
   - pcs13
   - pcs6
   - pedsql
   - pedsql (pediatric quality of life inventory)
   

In [135]:
CONFIDENCE_THRESHOLDS = CONCEPT_THRESHOLDS
RESTRICT_TO_EXPECTED_CRF_FIRST = PREFER_EXPECTED_CRF_FOR_CONCEPT
ALLOW_GLOBAL_FALLBACK_FOR_BEST_MATCH = True


def print_matching_summary(final_df):
    """Updated summary using the new clearer final-match columns."""
    processed_df = final_df[final_df['Final HEAL CDE Concept Match'].notna()].copy()
    total_processed = len(processed_df)

    if total_processed == 0:
        print("📊 No concept matches found.")
        return

    high_concept = len(processed_df[processed_df['Final Concept Match Score'] >= CONCEPT_THRESHOLDS['high']])
    medium_concept = len(processed_df[
        (processed_df['Final Concept Match Score'] >= CONCEPT_THRESHOLDS['medium']) &
        (processed_df['Final Concept Match Score'] < CONCEPT_THRESHOLDS['high'])
    ])
    low_concept = len(processed_df[processed_df['Final Concept Match Score'] < CONCEPT_THRESHOLDS['medium']])

    family_mode_count = len(processed_df[processed_df['Final Concept Match Mode'] == 'family'])

    fidelity_df = processed_df[processed_df['Final Encoding Fidelity Score'].notna()].copy()
    total_fidelity = len(fidelity_df)

    print(f"\n📊 Matching Summary:")
    print(f"   Total variables with a final concept match: {total_processed}")
    print(f"   🧠 High concept matches (≥{CONCEPT_THRESHOLDS['high']}): {high_concept} ({high_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Possible concept matches ({CONCEPT_THRESHOLDS['medium']}-{CONCEPT_THRESHOLDS['high']-1}): {medium_concept} ({medium_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Weak concept matches (<{CONCEPT_THRESHOLDS['medium']}): {low_concept} ({low_concept/total_processed*100:.1f}%)")
    print(f"   🧩 Family-level concept matches: {family_mode_count}")

    if total_fidelity > 0:
        high_fidelity = len(fidelity_df[fidelity_df['Final Encoding Fidelity Score'] >= FIDELITY_THRESHOLDS['high']])
        medium_fidelity = len(fidelity_df[
            (fidelity_df['Final Encoding Fidelity Score'] >= FIDELITY_THRESHOLDS['medium']) &
            (fidelity_df['Final Encoding Fidelity Score'] < FIDELITY_THRESHOLDS['high'])
        ])
        low_fidelity = len(fidelity_df[fidelity_df['Final Encoding Fidelity Score'] < FIDELITY_THRESHOLDS['medium']])

        print(f"\n   🛠️ High implementation fidelity (≥{FIDELITY_THRESHOLDS['high']}): {high_fidelity} ({high_fidelity/total_fidelity*100:.1f}%)")
        print(f"   🛠️ Medium implementation fidelity ({FIDELITY_THRESHOLDS['medium']}-{FIDELITY_THRESHOLDS['high']-1}): {medium_fidelity} ({medium_fidelity/total_fidelity*100:.1f}%)")
        print(f"   🛠️ Low implementation fidelity (<{FIDELITY_THRESHOLDS['medium']}): {low_fidelity} ({low_fidelity/total_fidelity*100:.1f}%)")
    else:
        print("\n   🛠️ No row-level implementation fidelity scores were available.")

In [136]:
def compare_encodings(
    study_df,
    encoding_column,
    field_label_column,
    cde_file,
    variable_name_column,
    form_name_column,
    expected_crf_column,
    dry_run=False,
    save_output=False
):
    """
    v5 behavior:
    - Uses study_df already created from prestep_df.
    - Does NOT read the study data dictionary from an intermediate Excel file.
    - Loads the HEAL CDE knowledge base from cde_file.
    - Stage 1 selects candidates by concept similarity.
    - Stage 1 uses the selective concept-family view of the KB.
    - Stage 2 reports encoding fidelity separately.
    - Protected-family rules can block cross-family proxy candidates from becoming the primary Best Match.
    - Writes final output workbook with VLMD_Results and optional Low_Confidence_Analysis sheets.
    """

    # ----------------------------
    # Setup / preflight
    # ----------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print("📁 Output directory (absolute):", os.path.abspath(OUTPUT_DIR))

    if study_df is None or study_df.empty:
        raise ValueError("study_df is empty or missing. Run the prestep + bridge cells first.")

    study_df = study_df.copy()

    print("📂 Using in-memory study_df from prestep_df.")
    print(f"✅ Processing {len(study_df)} study variable row(s).")

    required_cols = [
        variable_name_column,
        field_label_column,
        encoding_column,
        form_name_column
    ]

    missing_required = [col for col in required_cols if col not in study_df.columns]

    if missing_required:
        raise ValueError(
            "study_df is missing required columns: "
            + ", ".join(missing_required)
        )

    if expected_crf_column in study_df.columns:
        no_crf_count = (
            study_df[expected_crf_column]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("no crf match")
            .sum()
        )

        nonblank_crf_count = (
            study_df[expected_crf_column]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        print(
            f"🧭 Expected CRF column found: {expected_crf_column}\n"
            f"   Nonblank expected CRF values: {nonblank_crf_count} / {len(study_df)}\n"
            f"   Rows with 'No CRF match': {no_crf_count}"
        )
    else:
        print(
            f"⚠️ Expected CRF column '{expected_crf_column}' not found in study_df. "
            "Concept retrieval will proceed without CRF preference."
        )
        expected_crf_column = None

    if dry_run:
        print(f"🔍 DRY RUN: Would process {len(study_df)} variables against HEAL CDEs.")
        return None

    # ----------------------------
    # Local column aliases
    # ----------------------------
    var_col = variable_name_column
    form_col = form_name_column

    print("\n📋 Column mapping for CDE matching:")
    print(f"   Variable name column : {var_col}")
    print(f"   Field label column   : {field_label_column}")
    print(f"   Encoding column      : {encoding_column}")
    print(f"   Form name column     : {form_col}")
    print(f"   Expected CRF column  : {expected_crf_column}")

    # ----------------------------
    # Initialize output columns
    # ----------------------------
    new_cols = [
        # Stage A / concept retrieval columns
        "Closest HEAL CDE Concept",
        "Concept Match Score",
        "Concept Match CRF",
        "Concept Match Status",
        "Concept Match Mode",
        "Stage A Retrieval Mode",

        # Stage B / implementation fidelity columns
        "Encoding Fidelity Score",
        "Encoding Fidelity Status",

        # Final display columns
        "Final HEAL CDE Concept Match",
        "Final Concept Match Score",
        "Final Concept Match CRF",
        "Final Concept Match Status",
        "Final Concept Match Mode",
        "Final Encoding Fidelity Score",
        "Final Encoding Fidelity Status",

        # Legacy compatibility columns
        "Best Match CDE Name",
        "Best Match Score",
        "Best Match CRF Name",
        "Best Match Source",

        # Top concept alternates
        "Potential Match 2 - CDE Name",
        "Potential Match 2 - Score",
        "Potential Match 2 - CRF Name",
        "Potential Match 3 - CDE Name",
        "Potential Match 3 - Score",
        "Potential Match 3 - CRF Name",

        # Copyright-sensitive / proxy-block columns
        "Protected Family Rule Applied",
        "Protected Family Expected CRF",
        "Blocked Primary Candidate",
        "Blocked Primary Candidate Score",
        "Blocked Primary Candidate CRF",
        "Blocked Primary Candidate Fidelity",
        "Protected Family Note"
    ]

    for col in new_cols:
        study_df[col] = None

    # ----------------------------
    # Normalize study fields
    # ----------------------------
    print("\n🔧 Normalizing study variables...")

    study_df["Normalized Text"] = study_df[field_label_column].apply(normalize_string)
    study_df["Normalized Encoding"] = study_df[encoding_column].apply(normalize_string)
    study_df["Normalized Variable Name"] = study_df[var_col].apply(normalize_string)
    study_df["Normalized Form Name"] = study_df[form_col].apply(normalize_string)

    study_df["Normalized Combined"] = study_df.apply(
        lambda row: normalize_string(
            f"{row.get(var_col, '')} | "
            f"{row.get(field_label_column, '')} | "
            f"{row.get(encoding_column, '')}"
        ),
        axis=1
    )

    # ----------------------------
    # Load KB + build Stage 1 concept view
    # ----------------------------
    print("\n📚 Loading HEAL CDE database...")

    cde_file = Path(cde_file)

    if not cde_file.exists():
        raise FileNotFoundError(f"HEAL CDE knowledge base file not found: {cde_file}")

    cde_df = pd.read_excel(cde_file, sheet_name="ALL")

    required_kb_cols = [
        "CRF Name",
        "CDE Name",
        "Variable Name",
        "Definition",
        "Short Description",
        "Additional Notes (Question Text)",
        "PV Description"
    ]

    missing_kb_cols = [col for col in required_kb_cols if col not in cde_df.columns]

    if missing_kb_cols:
        raise ValueError(
            "HEAL CDE knowledge base is missing required columns: "
            + ", ".join(missing_kb_cols)
        )

    cde_df = cde_df.dropna(
        subset=["CDE Name", "Variable Name", "Additional Notes (Question Text)"],
        how="all"
    ).copy()

    cde_df["CRF Name"] = cde_df["CRF Name"].astype(str).str.strip()
    cde_df["CDE Name"] = cde_df["CDE Name"].astype(str).str.strip()

    kb_crf_names = sorted(cde_df["CRF Name"].dropna().unique())
    map_crf = build_crf_mapper(kb_crf_names)

    cde_df["Normalized CDE Variable Name"] = cde_df["Variable Name"].apply(normalize_string)
    cde_df["Normalized Text"] = cde_df["Additional Notes (Question Text)"].apply(normalize_string)
    cde_df["Normalized Encoding"] = cde_df["PV Description"].apply(normalize_string)

    stage1_cde_df = build_stage1_concept_view(cde_df).copy()

    stage1_cde_df["Normalized CDE Variable Name"] = stage1_cde_df["Variable Name"].apply(normalize_string)
    stage1_cde_df["Normalized Text"] = stage1_cde_df["Additional Notes (Question Text)"].apply(normalize_string)
    stage1_cde_df["Normalized Encoding"] = stage1_cde_df["PV Description"].apply(normalize_string)

    print(f"✅ Loaded {len(cde_df)} raw KB rows for implementation detail.")
    print(f"✅ Built {len(stage1_cde_df)} Stage 1 concept rows for concept retrieval.")

    # ----------------------------
    # Track lower-confidence concept matches
    # ----------------------------
    low_confidence_matches = []

    # ----------------------------
    # Matching
    # ----------------------------
    print("\n🔍 Running concept retrieval first, then encoding fidelity scoring...")

    for idx, row in tqdm(
        study_df.iterrows(),
        total=len(study_df),
        desc="Matching variables",
        unit="vars"
    ):

        if not row.get("Normalized Variable Name", "") and not row.get("Normalized Text", ""):
            continue

        expected_raw = row.get(expected_crf_column, None) if expected_crf_column else None
        has_anchor = has_usable_crf_anchor(expected_raw)

        expected_kb = map_crf(expected_raw) if has_anchor else None

        study_var_name = row.get(var_col, "")
        study_field_label = row.get(field_label_column, "")
        study_form_name = row.get(form_col, "")
        study_encoding = row.get(encoding_column, "")

        candidates = []

        # ------------------------------------------------------------
        # Stage A: concept retrieval uses stage1_cde_df
        # ------------------------------------------------------------
        for _, cde_row in stage1_cde_df.iterrows():
            cde_var = cde_row.get("Variable Name", "")
            cde_text = cde_row.get("Additional Notes (Question Text)", "")
            cde_crf = cde_row.get("CRF Name", "")
            concept_mode = cde_row.get("Stage1 Concept Mode", "row")
            member_vars = cde_row.get("Stage1 Member Variables", "")

            concept_score = concept_similarity_score(
                study_var_name=study_var_name,
                study_field_label=study_field_label,
                study_form_name=study_form_name,
                expected_crf=expected_kb,
                cde_var_name=cde_var,
                cde_question_text=cde_text,
                cde_crf_name=cde_crf
            )

            if concept_score < CONCEPT_THRESHOLDS["minimum_score"]:
                continue

            # Fidelity is descriptive. For family-mode synthetic rows, PV may be blank.
            # We still compute it, but the concept winner is selected by concept score.
            cde_encoding = cde_row.get("PV Description", "")
            fidelity_score = encoding_fidelity_score(study_encoding, cde_encoding)

            blocked_for_primary = is_blocked_primary_candidate(expected_kb, cde_crf)

            retrieval_mode = "anchored_crf_first" if has_anchor and expected_kb else "global"

            candidates.append({
                "cde_name": cde_var,
                "concept_score": round(concept_score, 1),
                "crf_name": cde_crf,
                "fidelity_score": round(fidelity_score, 1),
                "preferred_crf_hit": (
                    expected_kb is not None
                    and normalize_crf_label(cde_crf) == normalize_crf_label(expected_kb)
                ),
                "blocked_for_primary": blocked_for_primary,
                "concept_mode": concept_mode,
                "member_vars": member_vars,
                "retrieval_mode": retrieval_mode
            })

        if not candidates:
            continue

        # Sort by concept score first, then prefer expected CRF, then fidelity
        candidates = sorted(
            candidates,
            key=lambda x: (
                x["concept_score"],
                1 if x["preferred_crf_hit"] else 0,
                x["fidelity_score"]
            ),
            reverse=True
        )

        # De-dupe by concept name + CRF to avoid duplicate candidate display clutter
        unique_candidates = []
        seen = set()

        for cand in candidates:
            dedupe_key = (
                str(cand["cde_name"]).strip().lower(),
                str(cand["crf_name"]).strip().lower()
            )

            if dedupe_key in seen:
                continue

            unique_candidates.append(cand)
            seen.add(dedupe_key)

        protected_mode = is_copyright_sensitive_expected_crf(expected_kb)

        if protected_mode:
            study_df.at[idx, "Protected Family Rule Applied"] = "Yes"
            study_df.at[idx, "Protected Family Expected CRF"] = expected_kb
        else:
            study_df.at[idx, "Protected Family Rule Applied"] = ""
            study_df.at[idx, "Protected Family Expected CRF"] = ""

        allowed_primary_candidates = [
            cand for cand in unique_candidates
            if not cand["blocked_for_primary"]
        ]

        blocked_primary_candidates = [
            cand for cand in unique_candidates
            if cand["blocked_for_primary"]
        ]

        best = allowed_primary_candidates[0] if allowed_primary_candidates else None

        # ============================================================
        # CASE 1: normal / allowed primary winner exists
        # ============================================================
        if best is not None:
            top_candidates = allowed_primary_candidates[:TOP_N_CONCEPT_CANDIDATES]

            stage_a_status = classify_stage_a_display_status(
                concept_score=best["concept_score"],
                has_anchor=has_anchor,
                expected_kb=expected_kb,
                best_crf_name=best["crf_name"],
                retrieval_mode=best["retrieval_mode"]
            )

            fidelity_status = classify_fidelity_score(best["fidelity_score"])

            # Stage A / concept retrieval columns
            study_df.at[idx, "Closest HEAL CDE Concept"] = best["cde_name"]
            study_df.at[idx, "Concept Match Score"] = best["concept_score"]
            study_df.at[idx, "Concept Match CRF"] = best["crf_name"]
            study_df.at[idx, "Concept Match Status"] = stage_a_status
            study_df.at[idx, "Concept Match Mode"] = best["concept_mode"]
            study_df.at[idx, "Stage A Retrieval Mode"] = best["retrieval_mode"]

            # Stage B / fidelity columns
            study_df.at[idx, "Encoding Fidelity Score"] = best["fidelity_score"]
            study_df.at[idx, "Encoding Fidelity Status"] = fidelity_status

            # Final display columns
            if should_blank_main_concept(stage_a_status):
                study_df.at[idx, "Final HEAL CDE Concept Match"] = None
                study_df.at[idx, "Final Concept Match Score"] = None
                study_df.at[idx, "Final Concept Match CRF"] = None
                study_df.at[idx, "Final Concept Match Status"] = stage_a_status
                study_df.at[idx, "Final Concept Match Mode"] = best["concept_mode"]
                study_df.at[idx, "Final Encoding Fidelity Score"] = None
                study_df.at[idx, "Final Encoding Fidelity Status"] = None
            else:
                study_df.at[idx, "Final HEAL CDE Concept Match"] = best["cde_name"]
                study_df.at[idx, "Final Concept Match Score"] = best["concept_score"]
                study_df.at[idx, "Final Concept Match CRF"] = best["crf_name"]
                study_df.at[idx, "Final Concept Match Status"] = stage_a_status
                study_df.at[idx, "Final Concept Match Mode"] = best["concept_mode"]
                study_df.at[idx, "Final Encoding Fidelity Score"] = best["fidelity_score"]
                study_df.at[idx, "Final Encoding Fidelity Status"] = fidelity_status

            # Legacy compatibility columns
            study_df.at[idx, "Best Match CDE Name"] = study_df.at[idx, "Final HEAL CDE Concept Match"]
            study_df.at[idx, "Best Match Score"] = study_df.at[idx, "Final Concept Match Score"]
            study_df.at[idx, "Best Match CRF Name"] = study_df.at[idx, "Final Concept Match CRF"]
            study_df.at[idx, "Best Match Source"] = "Concept retrieval"

            # Reset blocked-proxy columns for clean rows
            study_df.at[idx, "Blocked Primary Candidate"] = None
            study_df.at[idx, "Blocked Primary Candidate Score"] = None
            study_df.at[idx, "Blocked Primary Candidate CRF"] = None
            study_df.at[idx, "Blocked Primary Candidate Fidelity"] = None
            study_df.at[idx, "Protected Family Note"] = None

            # Top alternates from allowed candidates only
            if len(top_candidates) > 1:
                study_df.at[idx, "Potential Match 2 - CDE Name"] = top_candidates[1]["cde_name"]
                study_df.at[idx, "Potential Match 2 - Score"] = top_candidates[1]["concept_score"]
                study_df.at[idx, "Potential Match 2 - CRF Name"] = top_candidates[1]["crf_name"]

            if len(top_candidates) > 2:
                study_df.at[idx, "Potential Match 3 - CDE Name"] = top_candidates[2]["cde_name"]
                study_df.at[idx, "Potential Match 3 - Score"] = top_candidates[2]["concept_score"]
                study_df.at[idx, "Potential Match 3 - CRF Name"] = top_candidates[2]["crf_name"]

            # Review sheet for weaker / non-high matches
            if stage_a_status != "High concept match":
                study_text_raw = f"{study_var_name} | {study_field_label} | {study_encoding}".strip()

                if len(study_text_raw) > 140:
                    study_text_raw = study_text_raw[:137] + "..."

                low_confidence_matches.append({
                    "Row": idx + 2,
                    "Study_Variable": study_var_name if study_var_name else "Unknown",
                    "Study_Form": study_form_name,
                    "Study_Text": study_text_raw,
                    "Closest_HEAL_CDE_Concept": study_df.at[idx, "Final HEAL CDE Concept Match"],
                    "Concept_Score": study_df.at[idx, "Final Concept Match Score"],
                    "Concept_CRF": study_df.at[idx, "Final Concept Match CRF"],
                    "Concept_Status": stage_a_status,
                    "Encoding_Fidelity_Score": study_df.at[idx, "Final Encoding Fidelity Score"],
                    "Encoding_Fidelity_Status": study_df.at[idx, "Final Encoding Fidelity Status"],
                    "Expected_CRF": expected_raw,
                    "Expected_CRF_Mapped_To_KB": expected_kb,
                    "Retrieval_Mode": best["retrieval_mode"],
                    "Concept_Mode": best["concept_mode"],
                    "Protected_Family_Note": ""
                })

        # ============================================================
        # CASE 2: protected family row has only blocked cross-family proxies
        #         -> do NOT write a primary best match
        # ============================================================
        elif protected_mode and blocked_primary_candidates:
            blocked = blocked_primary_candidates[0]

            # Leave primary/best-match/final columns blank to avoid false exact assignment
            study_df.at[idx, "Closest HEAL CDE Concept"] = None
            study_df.at[idx, "Concept Match Score"] = None
            study_df.at[idx, "Concept Match CRF"] = None
            study_df.at[idx, "Concept Match Status"] = "Protected-family review needed"
            study_df.at[idx, "Concept Match Mode"] = blocked["concept_mode"]
            study_df.at[idx, "Stage A Retrieval Mode"] = blocked["retrieval_mode"]

            study_df.at[idx, "Encoding Fidelity Score"] = None
            study_df.at[idx, "Encoding Fidelity Status"] = None

            study_df.at[idx, "Final HEAL CDE Concept Match"] = None
            study_df.at[idx, "Final Concept Match Score"] = None
            study_df.at[idx, "Final Concept Match CRF"] = None
            study_df.at[idx, "Final Concept Match Status"] = "Protected-family review needed"
            study_df.at[idx, "Final Concept Match Mode"] = blocked["concept_mode"]
            study_df.at[idx, "Final Encoding Fidelity Score"] = None
            study_df.at[idx, "Final Encoding Fidelity Status"] = None

            study_df.at[idx, "Best Match CDE Name"] = None
            study_df.at[idx, "Best Match Score"] = None
            study_df.at[idx, "Best Match CRF Name"] = None
            study_df.at[idx, "Best Match Source"] = "Protected-family proxy blocked"

            study_df.at[idx, "Potential Match 2 - CDE Name"] = None
            study_df.at[idx, "Potential Match 2 - Score"] = None
            study_df.at[idx, "Potential Match 2 - CRF Name"] = None
            study_df.at[idx, "Potential Match 3 - CDE Name"] = None
            study_df.at[idx, "Potential Match 3 - Score"] = None
            study_df.at[idx, "Potential Match 3 - CRF Name"] = None

            # Store blocked proxy for human review
            study_df.at[idx, "Blocked Primary Candidate"] = blocked["cde_name"]
            study_df.at[idx, "Blocked Primary Candidate Score"] = blocked["concept_score"]
            study_df.at[idx, "Blocked Primary Candidate CRF"] = blocked["crf_name"]
            study_df.at[idx, "Blocked Primary Candidate Fidelity"] = blocked["fidelity_score"]
            study_df.at[idx, "Protected Family Note"] = (
                f"Expected CRF '{expected_kb}' is protected. "
                f"Cross-family proxy '{blocked['cde_name']}' from '{blocked['crf_name']}' "
                "was blocked from becoming the primary best match."
            )

            study_text_raw = f"{study_var_name} | {study_field_label} | {study_encoding}".strip()

            if len(study_text_raw) > 140:
                study_text_raw = study_text_raw[:137] + "..."

            low_confidence_matches.append({
                "Row": idx + 2,
                "Study_Variable": study_var_name if study_var_name else "Unknown",
                "Study_Form": study_form_name,
                "Study_Text": study_text_raw,
                "Closest_HEAL_CDE_Concept": "",
                "Concept_Score": "",
                "Concept_CRF": "",
                "Concept_Status": "Protected-family review needed",
                "Encoding_Fidelity_Score": "",
                "Encoding_Fidelity_Status": "",
                "Expected_CRF": expected_raw,
                "Expected_CRF_Mapped_To_KB": expected_kb,
                "Retrieval_Mode": blocked["retrieval_mode"],
                "Concept_Mode": blocked["concept_mode"],
                "Protected_Family_Note": (
                    f"Blocked proxy candidate: {blocked['cde_name']} ({blocked['crf_name']}) | "
                    f"Concept={blocked['concept_score']} | Fidelity={blocked['fidelity_score']}"
                )
            })

        else:
            continue

    # ----------------------------
    # Final output + duplicate audit
    # ----------------------------
    final_df = study_df.copy()

    # Add duplicate-audit review columns if helper exists
    if "add_duplicate_audit_columns" in globals():
        final_df = add_duplicate_audit_columns(final_df, study_var_col=var_col)
    else:
        print("ℹ️ add_duplicate_audit_columns() not found. Skipping duplicate audit columns.")

    # Save the final matched dataframe in memory for inspection/reuse.
    # This gives us a notebook-level dataframe that includes:
    # original DD columns + prestep CRF fields + VLMD CDE matching outputs.
    global DDtoCRFtoVLMDCDE_df
    DDtoCRFtoVLMDCDE_df = final_df.copy()

    print(f"✅ Created dataframe: DDtoCRFtoVLMDCDE_df with {len(DDtoCRFtoVLMDCDE_df)} rows and {len(DDtoCRFtoVLMDCDE_df.columns)} columns.")

    low_conf_df = (
        pd.DataFrame(low_confidence_matches)
        if low_confidence_matches
        else pd.DataFrame()
    )

    # Respect config output path as the base, while preserving old v4 timestamp behavior.
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    config_output_path = Path(output_file) if "output_file" in globals() else Path(OUTPUT_DIR) / "CDE_ID_v5_output.xlsx"

    final_output_path = config_output_path.with_name(
        f"{config_output_path.stem}_vlmd_conceptsplit_{timestamp}{config_output_path.suffix}"
    )

    print("\n💾 Saving results...")

    with pd.ExcelWriter(final_output_path, engine="openpyxl") as writer:
        final_df.to_excel(writer, sheet_name="VLMD_Results", index=False)

        if not low_conf_df.empty:
            low_conf_df.to_excel(writer, sheet_name="Low_Confidence_Analysis", index=False)
            print(f"📝 {len(low_conf_df)} lower-confidence concept matches saved for analysis.")
        else:
            print("📝 No Low_Confidence_Analysis sheet created.")

    print(f"💾 CDE matching complete. Results saved to {final_output_path}")

    print_matching_summary(final_df)

    return str(final_output_path)


print("✅ v5 compare_encodings() loaded.")
print("   - Uses study_df from prestep_df instead of reading an intermediate Excel file.")
print("   - Stage 1 concept retrieval uses the concept-family view.")
print("   - Encoding is reported separately as implementation fidelity.")
print("   - Protected primary family rules are applied before assigning Best Match.")

✅ v5 compare_encodings() loaded.
   - Uses study_df from prestep_df instead of reading an intermediate Excel file.
   - Stage 1 concept retrieval uses the concept-family view.
   - Encoding is reported separately as implementation fidelity.
   - Protected primary family rules are applied before assigning Best Match.


In [137]:
def print_matching_summary(final_df):
    """
    Summary for v5 matching output.

    Prefer final display columns when available, because weak/internal matches
    may be blanked from the final accepted match fields.
    """

    concept_col = (
        "Final HEAL CDE Concept Match"
        if "Final HEAL CDE Concept Match" in final_df.columns
        else "Closest HEAL CDE Concept"
    )

    score_col = (
        "Final Concept Match Score"
        if "Final Concept Match Score" in final_df.columns
        else "Concept Match Score"
    )

    status_col = (
        "Final Concept Match Status"
        if "Final Concept Match Status" in final_df.columns
        else "Concept Match Status"
    )

    fidelity_score_col = (
        "Final Encoding Fidelity Score"
        if "Final Encoding Fidelity Score" in final_df.columns
        else "Encoding Fidelity Score"
    )

    fidelity_status_col = (
        "Final Encoding Fidelity Status"
        if "Final Encoding Fidelity Status" in final_df.columns
        else "Encoding Fidelity Status"
    )

    processed_df = final_df[
        final_df[concept_col].notna()
        & final_df[concept_col].astype(str).str.strip().ne("")
    ].copy()

    total_processed = len(processed_df)

    if total_processed == 0:
        print("📊 No final concept matches found.")
        return

    high_concept = len(processed_df[processed_df[status_col] == "High concept match"])
    possible_concept = len(processed_df[processed_df[status_col] == "Possible concept match"])
    weak_concept = len(processed_df[processed_df[status_col] == "Weak concept match"])
    protected_review = len(processed_df[processed_df[status_col] == "Protected-family review needed"])

    family_mode_count = 0
    if "Final Concept Match Mode" in processed_df.columns:
        family_mode_count = len(processed_df[processed_df["Final Concept Match Mode"] == "family"])

    fidelity_df = processed_df[
        processed_df[fidelity_score_col].notna()
        & processed_df[fidelity_score_col].astype(str).str.strip().ne("")
    ].copy()

    total_fidelity = len(fidelity_df)

    print("\n📊 Matching Summary:")
    print(f"   Total variables with a final concept match: {total_processed}")
    print(f"   🧠 High concept matches: {high_concept} ({high_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Possible concept matches: {possible_concept} ({possible_concept/total_processed*100:.1f}%)")
    print(f"   🧠 Weak concept matches: {weak_concept} ({weak_concept/total_processed*100:.1f}%)")

    if protected_review:
        print(f"   🛡️ Protected-family review needed: {protected_review} ({protected_review/total_processed*100:.1f}%)")

    print(f"   🧩 Family-level concept matches: {family_mode_count}")

    if total_fidelity > 0:
        high_fidelity = len(fidelity_df[fidelity_df[fidelity_status_col] == "Closely implemented"])
        medium_fidelity = len(fidelity_df[fidelity_df[fidelity_status_col] == "Concept captured, encoding differs"])
        low_fidelity = len(fidelity_df[fidelity_df[fidelity_status_col] == "Low implementation fidelity"])

        print(f"\n   🛠️ Rows with implementation fidelity scores: {total_fidelity}")
        print(f"   🛠️ High implementation fidelity: {high_fidelity} ({high_fidelity/total_fidelity*100:.1f}%)")
        print(f"   🛠️ Medium implementation fidelity: {medium_fidelity} ({medium_fidelity/total_fidelity*100:.1f}%)")
        print(f"   🛠️ Low implementation fidelity: {low_fidelity} ({low_fidelity/total_fidelity*100:.1f}%)")
    else:
        print("\n   🛠️ No row-level implementation fidelity scores were available.")

In [138]:
main(dry_run=False)

🚀 Starting HEAL CDE Variable Level Metadata Matching
   Input source: study_df created from prestep_df

📋 Concept retrieval thresholds: High ≥75%, Possible ≥50%, Minimum ≥25%
📋 Encoding fidelity thresholds: High ≥80%, Medium ≥50%

🧭 Concept retrieval settings:
   Expected CRF column        : HEAL Core CRF Match
   Prefer expected CRF        : True
   CRF mapping threshold      : 85
   CRF concept bonus          : 8
   Top N concept candidates   : 3
   Potential Match 2/3 are alternate concept candidates.

✅ Expected CRF column found: HEAL Core CRF Match
   Nonblank expected CRF values: 14 / 14

[study_df preview]


,section,name,description,enumLabels,HEAL Core CRF Match
0,a_infant_medical_history_01_month,imh_birthhcr,5. Head circumference at birth,NaN,No CRF match
1,a_infant_medical_history_01_month,imh_birthlt,6. Length at birth,NaN,No CRF match
2,a_infant_medical_history_01_month,imhbirthwt,4. Weight at birth,NaN,Demographics
3,bpisev,BPIAvgPainRtngScl,Average Pain,0=No pain (0)|1=1|2=2|3=3|4=4|5=5|6=6|7=7|8=8|...,BPI Pain Severity
4,bpisev,BPICurrentPainRtngScl,Current Pain,0=No pain (0)|1=1|2=2|3=3|4=4|5=5|6=6|7=7|8=8|...,BPI Pain Severity


📁 Output directory (absolute): c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out
📂 Using in-memory study_df from prestep_df.
✅ Processing 14 study variable row(s).
🧭 Expected CRF column found: HEAL Core CRF Match
   Nonblank expected CRF values: 14 / 14
   Rows with 'No CRF match': 3

📋 Column mapping for CDE matching:
   Variable name column : name
   Field label column   : description
   Encoding column      : enumLabels
   Form name column     : section
   Expected CRF column  : HEAL Core CRF Match

🔧 Normalizing study variables...

📚 Loading HEAL CDE database...
✅ Stage 1 concept view created.
   Original KB rows: 425
   Stage 1 concept rows: 407

🧩 Family-mode concepts currently enabled:


,CRF Name,CDE Name,Variable Name,Stage1 Member Variables
40,Demographics,Parent Race,Parent Race,"AI_AN_p, Asian_p, Bl_AA_p, NH_PI_p, Not_Rep_p,..."
41,Demographics,Race,Race,"AI_AN, Asian, Bl_AA, NH_PI, Not_Rep, Unkn, White"
42,Demographics,Race child,Race child,"AI_AN, Asian, Bl_AA, NH_PI, Not_Rep, Unkn, White"


✅ Loaded 425 raw KB rows for implementation detail.
✅ Built 407 Stage 1 concept rows for concept retrieval.

🔍 Running concept retrieval first, then encoding fidelity scoring...


Matching variables: 100%|██████████| 14/14 [00:03<00:00,  4.22vars/s]


ℹ️ add_duplicate_audit_columns() not found. Skipping duplicate audit columns.
✅ Created dataframe: DDtoCRFtoVLMDCDE_df with 14 rows and 85 columns.

💾 Saving results...
📝 5 lower-confidence concept matches saved for analysis.
💾 CDE matching complete. Results saved to out\Testfile_2026-05-05_vlmd_conceptsplit_20260505_130027.xlsx

📊 Matching Summary:
   Total variables with a final concept match: 12
   🧠 High concept matches: 9 (75.0%)
   🧠 Possible concept matches: 3 (25.0%)
   🧠 Weak concept matches: 0 (0.0%)
   🧩 Family-level concept matches: 0

   🛠️ Rows with implementation fidelity scores: 12
   🛠️ High implementation fidelity: 11 (91.7%)
   🛠️ Medium implementation fidelity: 0 (0.0%)
   🛠️ Low implementation fidelity: 1 (8.3%)

✅ Output file created:
   📄 c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\Testfile_2026-05-05_vlmd_conceptsplit_20260505_130027.xlsx

⚠️ Color coding failed. File is still valid: name 'apply_color_coding' is not defined

🎉 VLMD CDE

In [139]:
# ----------------------------
# Step 1: Confirm VLMD handoff dataframe exists
# ----------------------------

if "DDtoCRFtoVLMDCDE_df" not in globals():
    raise NameError(
        "DDtoCRFtoVLMDCDE_df does not exist yet. "
        "Run main(dry_run=False) before starting the recon step."
    )

print("✅ Handoff dataframe found!")
print(f"Rows: {len(DDtoCRFtoVLMDCDE_df)}")
print(f"Columns: {len(DDtoCRFtoVLMDCDE_df.columns)}")

display(DDtoCRFtoVLMDCDE_df.head())

✅ Handoff dataframe found!
Rows: 14
Columns: 85


,schemaVersion,section,name,title,description,type,format,constraints.required,constraints.maxLength,constraints.enum,...,Blocked Primary Candidate,Blocked Primary Candidate Score,Blocked Primary Candidate CRF,Blocked Primary Candidate Fidelity,Protected Family Note,Normalized Text,Normalized Encoding,Normalized Variable Name,Normalized Form Name,Normalized Combined
0,0.3.2,a_infant_medical_history_01_month,imh_birthhcr,5. Head circumference at birth,5. Head circumference at birth,number,NaN,NaN,NaN,NaN,...,None,None,None,None,None,5 head circumference at birth,,imh birthhcr,a infant medical history 01 month,imh birthhcr 5 head circumference at birth nan
1,0.3.2,a_infant_medical_history_01_month,imh_birthlt,6. Length at birth,6. Length at birth,number,NaN,NaN,NaN,NaN,...,None,None,None,None,None,6 length at birth,,imh birthlt,a infant medical history 01 month,imh birthlt 6 length at birth nan
2,0.3.2,a_infant_medical_history_01_month,imhbirthwt,4. Weight at birth,4. Weight at birth,number,NaN,NaN,NaN,NaN,...,None,None,None,None,None,4 weight at birth,,imhbirthwt,a infant medical history 01 month,imhbirthwt 4 weight at birth nan
3,0.3.2,bpisev,BPIAvgPainRtngScl,Average Pain,Average Pain,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,None,None,None,None,None,average pain,0=no pain 0 1=1 2=2 3=3 4=4 5=5 6=6 7=7 8=8 9=...,bpiavgpainrtngscl,bpisev,bpiavgpainrtngscl average pain 0=no pain 0 1=1...
4,0.3.2,bpisev,BPICurrentPainRtngScl,Current Pain,Current Pain,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,None,None,None,None,None,current pain,0=no pain 0 1=1 2=2 3=3 4=4 5=5 6=6 7=7 8=8 9=...,bpicurrentpainrtngscl,bpisev,bpicurrentpainrtngscl current pain 0=no pain 0...


In [140]:
# ============================================================
# STEP 2: Create recon input dataframe from VLMD handoff dataframe
# ============================================================
# Goal:
# - Use the in-memory VLMD output from revamp_v5
# - Do NOT read an intermediate Excel file
# - Prepare a clean dataframe for the reconciliation/adjudication step

import pandas as pd
import json

# ----------------------------
# Start from the confirmed handoff dataframe
# ----------------------------
recon_input_df = DDtoCRFtoVLMDCDE_df.copy()

print("✅ Created recon_input_df from DDtoCRFtoVLMDCDE_df")
print(f"Rows: {len(recon_input_df)}")
print(f"Columns: {len(recon_input_df.columns)}")


# ----------------------------
# Small text helper
# ----------------------------
def norm_text(value):
    """
    Normalize values for safe checks.
    Treat NaN/None as blank strings.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


# ----------------------------
# Column resolution helper
# ----------------------------
def pick_first_existing(columns, candidates, required=False, label=None):
    """
    Pick the first matching column name from a list of possible names.
    This lets recon work even if column names evolved between notebook versions.
    """
    for candidate in candidates:
        if candidate in columns:
            return candidate

    if required:
        raise KeyError(
            f"Could not find required column for {label or candidates}. "
            f"Tried: {candidates}"
        )

    return None


# ----------------------------
# Resolve the columns recon will need
# ----------------------------
resolved_cols = {
    "variable_name": pick_first_existing(
        recon_input_df.columns,
        ["Variable / Field Name", "Variable Name", "name", "var"],
        required=True,
        label="variable_name"
    ),

    "form_name": pick_first_existing(
        recon_input_df.columns,
        ["Form Name", "form_name", "form"],
        required=False,
        label="form_name"
    ),

    "question_text": pick_first_existing(
        recon_input_df.columns,
        ["Field Label", "Question Text", "Description", "Variable Label"],
        required=False,
        label="question_text"
    ),

    "prestep_crf_match": pick_first_existing(
        recon_input_df.columns,
        ["HEAL Core CRF Match"],
        required=False,
        label="prestep_crf_match"
    ),

    "prestep_confidence": pick_first_existing(
        recon_input_df.columns,
        ["Prestep CRF Confidence", "Confidence Level"],
        required=False,
        label="prestep_confidence"
    ),

    "match_rationale": pick_first_existing(
        recon_input_df.columns,
        ["Match Rationale", "Rationale"],
        required=False,
        label="match_rationale"
    ),

    "final_concept_match": pick_first_existing(
        recon_input_df.columns,
        ["Final HEAL CDE Concept Match", "Best Match CDE Name", "Closest HEAL CDE Concept"],
        required=False,
        label="final_concept_match"
    ),

    "best_match_score": pick_first_existing(
        recon_input_df.columns,
        ["Final Concept Match Score", "Best Match Score", "Concept Match Score"],
        required=False,
        label="best_match_score"
    ),

    "best_match_crf": pick_first_existing(
        recon_input_df.columns,
        ["Final Concept Match CRF", "Best Match CRF Name", "Concept Match CRF"],
        required=False,
        label="best_match_crf"
    ),

    "final_concept_status": pick_first_existing(
        recon_input_df.columns,
        ["Final Concept Match Status", "Concept Match Status", "Final Concept Status"],
        required=False,
        label="final_concept_status"
    ),

    "encoding_fidelity_score": pick_first_existing(
        recon_input_df.columns,
        ["Final Encoding Fidelity Score", "Encoding Fidelity Score"],
        required=False,
        label="encoding_fidelity_score"
    ),

    "potential_match_2": pick_first_existing(
        recon_input_df.columns,
        ["Potential Match 2 - CDE Name", "Potential Match 2", "Potential Match 2 CDE Name"],
        required=False,
        label="potential_match_2"
    ),

    "potential_match_2_crf": pick_first_existing(
        recon_input_df.columns,
        ["Potential Match 2 - CRF Name", "Potential Match 2 CRF Name"],
        required=False,
        label="potential_match_2_crf"
    ),

    "potential_match_3": pick_first_existing(
        recon_input_df.columns,
        ["Potential Match 3 - CDE Name", "Potential Match 3", "Potential Match 3 CDE Name"],
        required=False,
        label="potential_match_3"
    ),

    "potential_match_3_crf": pick_first_existing(
        recon_input_df.columns,
        ["Potential Match 3 - CRF Name", "Potential Match 3 CRF Name"],
        required=False,
        label="potential_match_3_crf"
    ),

    "best_match_source": pick_first_existing(
        recon_input_df.columns,
        ["Best Match Source"],
        required=False,
        label="best_match_source"
    ),

    "protected_family_rule_applied": pick_first_existing(
        recon_input_df.columns,
        ["Protected Family Rule Applied"],
        required=False,
        label="protected_family_rule_applied"
    ),

    "protected_family_expected_crf": pick_first_existing(
        recon_input_df.columns,
        ["Protected Family Expected CRF"],
        required=False,
        label="protected_family_expected_crf"
    ),

    "blocked_primary_candidate": pick_first_existing(
        recon_input_df.columns,
        ["Blocked Primary Candidate"],
        required=False,
        label="blocked_primary_candidate"
    ),

    "blocked_primary_candidate_score": pick_first_existing(
        recon_input_df.columns,
        ["Blocked Primary Candidate Score"],
        required=False,
        label="blocked_primary_candidate_score"
    ),

    "blocked_primary_candidate_crf": pick_first_existing(
        recon_input_df.columns,
        ["Blocked Primary Candidate CRF"],
        required=False,
        label="blocked_primary_candidate_crf"
    ),

    "blocked_primary_candidate_fidelity": pick_first_existing(
        recon_input_df.columns,
        ["Blocked Primary Candidate Fidelity"],
        required=False,
        label="blocked_primary_candidate_fidelity"
    ),

    "protected_family_note": pick_first_existing(
        recon_input_df.columns,
        ["Protected Family Note"],
        required=False,
        label="protected_family_note"
    ),
}


# ----------------------------
# Print resolved columns
# ----------------------------
print("\n🧭 Resolved recon columns:")
for key, value in resolved_cols.items():
    print(f"  {key}: {value}")


# ----------------------------
# Quick eligibility preview
# This does NOT run the LLM.
# It just tells us how many rows look recon-ready.
# ----------------------------
status_col = resolved_cols["final_concept_status"]

candidate_cols = [
    resolved_cols["final_concept_match"],
    resolved_cols["potential_match_2"],
    resolved_cols["potential_match_3"],
]
candidate_cols = [c for c in candidate_cols if c is not None]

has_candidate_mask = recon_input_df[candidate_cols].apply(
    lambda row: any(norm_text(value) for value in row),
    axis=1
) if candidate_cols else pd.Series(False, index=recon_input_df.index)

if status_col:
    eligible_status_mask = recon_input_df[status_col].isin(
        {"High concept match", "Possible concept match"}
    )
else:
    eligible_status_mask = pd.Series(True, index=recon_input_df.index)

eligible_recon_preview_mask = eligible_status_mask & has_candidate_mask

print("\n📊 Recon preview counts:")
print(f"Rows with at least one candidate: {has_candidate_mask.sum()}")
print(f"Rows with eligible concept status: {eligible_status_mask.sum()}")
print(f"Rows that look eligible for recon: {eligible_recon_preview_mask.sum()}")


# ----------------------------
# Preview likely recon rows
# ----------------------------
preview_cols = [
    resolved_cols["variable_name"],
    resolved_cols["form_name"],
    resolved_cols["prestep_crf_match"],
    resolved_cols["final_concept_status"],
    resolved_cols["final_concept_match"],
    resolved_cols["best_match_score"],
    resolved_cols["encoding_fidelity_score"],
    resolved_cols["potential_match_2"],
    resolved_cols["potential_match_3"],
]

preview_cols = [c for c in preview_cols if c is not None]

display(recon_input_df.loc[eligible_recon_preview_mask, preview_cols].head(20))

✅ Created recon_input_df from DDtoCRFtoVLMDCDE_df
Rows: 14
Columns: 85

🧭 Resolved recon columns:
  variable_name: name
  form_name: None
  question_text: None
  prestep_crf_match: HEAL Core CRF Match
  prestep_confidence: Prestep CRF Confidence
  match_rationale: Match Rationale
  final_concept_match: Final HEAL CDE Concept Match
  best_match_score: Final Concept Match Score
  best_match_crf: Final Concept Match CRF
  final_concept_status: Final Concept Match Status
  encoding_fidelity_score: Final Encoding Fidelity Score
  potential_match_2: Potential Match 2 - CDE Name
  potential_match_2_crf: Potential Match 2 - CRF Name
  potential_match_3: Potential Match 3 - CDE Name
  potential_match_3_crf: Potential Match 3 - CRF Name
  best_match_source: Best Match Source
  protected_family_rule_applied: Protected Family Rule Applied
  protected_family_expected_crf: Protected Family Expected CRF
  blocked_primary_candidate: Blocked Primary Candidate
  blocked_primary_candidate_score: Blocked 

,name,HEAL Core CRF Match,Final Concept Match Status,Final HEAL CDE Concept Match,Final Concept Match Score,Final Encoding Fidelity Score,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name
2,imhbirthwt,Demographics,Possible concept match,BRTHDTC,66.0,0,Sex,MARISTAT
3,BPIAvgPainRtngScl,BPI Pain Severity,High concept match,BPIAvgPainRatingScl,99.3,100,BPIAvgPain7dRtngScale,BPILeastPainRatingScl
4,BPICurrentPainRtngScl,BPI Pain Severity,Possible concept match,BPICurrentPainRatingScl,74.5,100,BPICurntPainRtngScale,BPILeastPainRatingScl
5,BPILeastPnLst24HRtngScl,BPI Pain Severity,High concept match,BPILeastPainRatingScl,95.0,100,BPIWorstPainRatingScl,BPIAvgPainRatingScl
6,BPIWrstPnLast24HRtngScl,BPI Pain Severity,High concept match,BPIWorstPainRatingScl,94.0,100,BPILeastPainRatingScl,BPIWrstPain7dRtngScale
7,GAD2FeelNervScl,GAD2 Pain (Generalized Anxiety Disorder),High concept match,GAD2FeelNervScale,98.4,100,GAD2FeelNervScl,GAD7FeelAfrdScl
8,GAD2NotStopWryScl,No CRF match,Possible concept match,GAD2NotStopWryScl,81.0,100,GAD2NotStopWryScale,GAD7WryTooMchScl
9,GAD7EasyAnnoyedScl,GAD7,High concept match,GAD7EasyAnnoyedScl,100,100,GAD7TroubRelxScl,GAD7TotScore
10,GAD7FeelAfrdScl,GAD7,High concept match,GAD7FeelAfrdScl,100,100,GAD2FeelNervScl,GAD7TotScore
11,GAD7RstlessScl,GAD7,High concept match,GAD7RstlessScl,100,100,GAD2NotStopWryScl,GAD7TotScore


In [141]:
# ============================================================
# STEP 3: Load recon settings and prompt from config
# ============================================================

import configparser

# ----------------------------
# Make sure config is loaded
# ----------------------------
config = configparser.ConfigParser()
config.optionxform = str  # preserve key casing where helpful
config.read(CONFIG_PATH, encoding="utf-8")


# ----------------------------
# Load recon prompt
# ----------------------------
RECON_ADJUDICATION_INSTRUCTION = config.get(
    "Instructions",
    "recon_adjudication_instruction",
    fallback=""
).strip()

if not RECON_ADJUDICATION_INSTRUCTION:
    raise ValueError(
        "Missing recon_adjudication_instruction in [Instructions] section of config."
    )

print("✅ Loaded recon_adjudication_instruction from config.")


# ----------------------------
# Load recon settings
# ----------------------------
RECON_ENABLED = config.getboolean(
    "ReconSettings",
    "enabled",
    fallback=True
)

RECON_DRY_RUN = config.getboolean(
    "ReconSettings",
    "dry_run",
    fallback=False
)

eligible_statuses_raw = config.get(
    "ReconSettings",
    "eligible_concept_statuses",
    fallback="High concept match | Possible concept match"
)

RECON_ELIGIBLE_CONCEPT_STATUSES = [
    status.strip()
    for status in eligible_statuses_raw.split("|")
    if status.strip()
]

max_rows_raw = config.get(
    "ReconSettings",
    "max_rows",
    fallback=""
).strip()

RECON_MAX_ROWS = int(max_rows_raw) if max_rows_raw else None


print("\n🧭 Recon settings:")
print(f"  Enabled: {RECON_ENABLED}")
print(f"  Dry run: {RECON_DRY_RUN}")
print(f"  Eligible statuses: {RECON_ELIGIBLE_CONCEPT_STATUSES}")
print(f"  Max rows: {RECON_MAX_ROWS}")

✅ Loaded recon_adjudication_instruction from config.

🧭 Recon settings:
  Enabled: True
  Dry run: False
  Eligible statuses: ['High concept match', 'Possible concept match']
  Max rows: None


In [142]:
# Preview first part of recon prompt
print(RECON_ADJUDICATION_INSTRUCTION[:800])

You are an expert metadata steward reviewing candidate HEAL Common Data Element matches.
Your task is to choose the best official HEAL CDE match for one study data dictionary row, using only the provided candidate matches and row context.
You must not invent new CDE names, CRF names, or variable names.
Choose from the provided candidate list only.
If none of the candidates are appropriate, return No Match.
Do not accept a candidate based only on similar permissible values or numeric scale ranges.
The candidate must also align with the study row concept, CRF/form context, and question/definition meaning.
Prefer candidates supported by multiple signals, including:
- the study variable name
- the study form name
- the field label or question text
- permissible values or encoding labels
- the 


In [143]:
# ============================================================
# STEP 4: Create recon candidate dataframe
# ============================================================
# Goal:
# - Filter recon_input_df to rows that are eligible for reconciliation
# - Respect config settings from [ReconSettings]
# - Apply max_rows for safe testing
# - Do NOT call the LLM yet

# ----------------------------
# Safety check
# ----------------------------
if "recon_input_df" not in globals():
    raise NameError("recon_input_df does not exist. Run Step 2 first.")

if "resolved_cols" not in globals():
    raise NameError("resolved_cols does not exist. Run Step 2 first.")

if "RECON_ENABLED" not in globals():
    raise NameError("Recon settings are not loaded. Run Step 3 first.")


# ----------------------------
# Start with full recon input
# ----------------------------
recon_working_df = recon_input_df.copy()

# Preserve original row index so we can merge results back later
recon_working_df["recon_source_index"] = recon_working_df.index


# ----------------------------
# Identify candidate columns
# ----------------------------
candidate_cols = [
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
]

candidate_cols = [col for col in candidate_cols if col is not None]


if not candidate_cols:
    raise ValueError(
        "No candidate CDE columns were resolved. "
        "Check resolved_cols for final_concept_match, potential_match_2, and potential_match_3."
    )


# ----------------------------
# Row has at least one candidate
# ----------------------------
has_candidate_mask = recon_working_df[candidate_cols].apply(
    lambda row: any(norm_text(value) for value in row),
    axis=1
)


# ----------------------------
# Row has eligible concept status
# ----------------------------
status_col = resolved_cols.get("final_concept_status")

if status_col:
    eligible_status_mask = recon_working_df[status_col].isin(RECON_ELIGIBLE_CONCEPT_STATUSES)
else:
    # If there is no status column, use candidate presence only.
    # This keeps the step flexible, but the status column is preferred.
    eligible_status_mask = pd.Series(True, index=recon_working_df.index)


# ----------------------------
# Combine eligibility rules
# ----------------------------
eligible_recon_mask = has_candidate_mask & eligible_status_mask


# ----------------------------
# Apply enabled / max_rows settings
# ----------------------------
if RECON_ENABLED:
    recon_candidate_df = recon_working_df.loc[eligible_recon_mask].copy()
else:
    recon_candidate_df = recon_working_df.iloc[0:0].copy()

rows_eligible_before_max = len(recon_candidate_df)

if RECON_MAX_ROWS is not None:
    recon_candidate_df = recon_candidate_df.head(RECON_MAX_ROWS).copy()


# ----------------------------
# Print summary
# ----------------------------
print("✅ Created recon_candidate_df")
print(f"Recon enabled: {RECON_ENABLED}")
print(f"Eligible concept statuses: {RECON_ELIGIBLE_CONCEPT_STATUSES}")
print(f"Rows in recon_input_df: {len(recon_input_df)}")
print(f"Rows with at least one candidate: {has_candidate_mask.sum()}")
print(f"Rows with eligible concept status: {eligible_status_mask.sum()}")
print(f"Rows eligible before max_rows: {rows_eligible_before_max}")
print(f"Rows selected for recon: {len(recon_candidate_df)}")


# ----------------------------
# Preview selected rows
# ----------------------------
preview_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("form_name"),
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_status"),
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("best_match_score"),
    resolved_cols.get("encoding_fidelity_score"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_2_crf"),
    resolved_cols.get("potential_match_3"),
    resolved_cols.get("potential_match_3_crf"),
    resolved_cols.get("protected_family_rule_applied"),
    resolved_cols.get("blocked_primary_candidate"),
]

preview_cols = [col for col in preview_cols if col is not None and col in recon_candidate_df.columns]

display(recon_candidate_df[preview_cols].head(20))

✅ Created recon_candidate_df
Recon enabled: True
Eligible concept statuses: ['High concept match', 'Possible concept match']
Rows in recon_input_df: 14
Rows with at least one candidate: 14
Rows with eligible concept status: 12
Rows eligible before max_rows: 12
Rows selected for recon: 12


,recon_source_index,name,HEAL Core CRF Match,Final Concept Match Status,Final HEAL CDE Concept Match,Final Concept Match Score,Final Encoding Fidelity Score,Potential Match 2 - CDE Name,Potential Match 2 - CRF Name,Potential Match 3 - CDE Name,Potential Match 3 - CRF Name,Protected Family Rule Applied,Blocked Primary Candidate
2,2,imhbirthwt,Demographics,Possible concept match,BRTHDTC,66.0,0,Sex,Demographics,MARISTAT,Demographics,,None
3,3,BPIAvgPainRtngScl,BPI Pain Severity,High concept match,BPIAvgPainRatingScl,99.3,100,BPIAvgPain7dRtngScale,BPI Pain Severity,BPILeastPainRatingScl,BPI Pain Severity,Yes,None
4,4,BPICurrentPainRtngScl,BPI Pain Severity,Possible concept match,BPICurrentPainRatingScl,74.5,100,BPICurntPainRtngScale,BPI Pain Severity,BPILeastPainRatingScl,BPI Pain Severity,Yes,None
5,5,BPILeastPnLst24HRtngScl,BPI Pain Severity,High concept match,BPILeastPainRatingScl,95.0,100,BPIWorstPainRatingScl,BPI Pain Severity,BPIAvgPainRatingScl,BPI Pain Severity,Yes,None
6,6,BPIWrstPnLast24HRtngScl,BPI Pain Severity,High concept match,BPIWorstPainRatingScl,94.0,100,BPILeastPainRatingScl,BPI Pain Severity,BPIWrstPain7dRtngScale,BPI Pain Severity,Yes,None
7,7,GAD2FeelNervScl,GAD2 Pain (Generalized Anxiety Disorder),High concept match,GAD2FeelNervScale,98.4,100,GAD2FeelNervScl,GAD7,GAD7FeelAfrdScl,GAD7,,None
8,8,GAD2NotStopWryScl,No CRF match,Possible concept match,GAD2NotStopWryScl,81.0,100,GAD2NotStopWryScale,GAD2 Pain,GAD7WryTooMchScl,GAD7,,None
9,9,GAD7EasyAnnoyedScl,GAD7,High concept match,GAD7EasyAnnoyedScl,100,100,GAD7TroubRelxScl,GAD7,GAD7TotScore,GAD7,,None
10,10,GAD7FeelAfrdScl,GAD7,High concept match,GAD7FeelAfrdScl,100,100,GAD2FeelNervScl,GAD7,GAD7TotScore,GAD7,,None
11,11,GAD7RstlessScl,GAD7,High concept match,GAD7RstlessScl,100,100,GAD2NotStopWryScl,GAD7,GAD7TotScore,GAD7,,None


In [144]:
# ============================================================
# STEP 5: Build official candidate packets from HEAL CDE JSON
# ============================================================
# Goal:
# - Load the official HEAL CDE JSON knowledge base
# - Look up candidate matches from recon_candidate_df
# - Build structured candidate packets for recon/adjudication
# - Do NOT call the LLM yet

import json
from pathlib import Path
import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "recon_candidate_df" not in globals():
    raise NameError("recon_candidate_df does not exist. Run Step 4 first.")

if "resolved_cols" not in globals():
    raise NameError("resolved_cols does not exist. Run Step 2 first.")

if "config" not in globals():
    raise NameError("config does not exist. Run Step 3 first.")


# ----------------------------
# Helper: safe text normalization
# ----------------------------
def norm_text_safe(value):
    """
    Normalize values for safe checks.
    Treat NaN/None as blank strings.
    Safely handles lists, tuples, sets, and dicts from flattened JSON fields.
    """
    if value is None:
        return ""

    # Handle list-like values before pd.isna, because pd.isna(list) returns an array
    if isinstance(value, (list, tuple, set)):
        cleaned_items = []

        for item in value:
            cleaned_item = norm_text_safe(item)
            if cleaned_item:
                cleaned_items.append(cleaned_item)

        return " | ".join(cleaned_items)

    # Handle dict values safely
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


def normalize_lookup_key(value):
    """
    Normalize text for matching candidate values to official KB entries.
    Keeps matching flexible across spacing/case differences.
    """
    return norm_text_safe(value).lower()


# ----------------------------
# Helper: get config value using several possible keys
# ----------------------------
def get_first_config_value(config_obj, section, possible_keys, fallback=None):
    """
    Try several possible config keys and return the first one found.
    This makes the notebook tolerant of slightly different config naming.
    """
    if not config_obj.has_section(section):
        return fallback

    for key in possible_keys:
        if config_obj.has_option(section, key):
            value = config_obj.get(section, key, fallback="").strip()
            if value:
                return value

    return fallback


# ----------------------------
# Find HEAL CDE JSON path from config
# ----------------------------
heal_cde_json_path_raw = get_first_config_value(
    config,
    section="Files",
    possible_keys=[
        "KB_JSON_PATH",
        "kb_json_path",
        "heal_cde_json_path",
        "HEAL_CDE_JSON_PATH",
        "cde_json_path",
        "CDE_JSON_PATH",
        "knowledge_base_json_path",
        "json_path",
    ],
    fallback="KnowledgeBase/All_HEALPAINCDEsDD_row_level_flattened.json"
)

# Normalize Windows-style backslashes so pathlib can handle the path cleanly
heal_cde_json_path_clean = heal_cde_json_path_raw.replace("\\", "/")


candidate_json_paths = []

# Raw path from config
candidate_json_paths.append(Path(heal_cde_json_path_clean))

# Relative to config file location
if "CONFIG_PATH" in globals():
    candidate_json_paths.append(Path(CONFIG_PATH).parent / heal_cde_json_path_clean)

# Relative to current working directory
candidate_json_paths.append(Path.cwd() / heal_cde_json_path_clean)

# Common fallback location inside project
candidate_json_paths.append(Path.cwd() / "KnowledgeBase" / "All_HEALPAINCDEsDD_row_level_flattened.json")


HEAL_CDE_JSON_PATH = None

for possible_path in candidate_json_paths:
    if possible_path.exists():
        HEAL_CDE_JSON_PATH = possible_path
        break


if HEAL_CDE_JSON_PATH is None:
    tried_paths = "\n".join([f"  - {p}" for p in candidate_json_paths])
    raise FileNotFoundError(
        "Could not find the HEAL CDE JSON file.\n"
        "Check the [Files] section in config_prestep.ini and confirm KB_JSON_PATH points to the JSON file.\n\n"
        f"Current KB_JSON_PATH value: {heal_cde_json_path_raw}\n\n"
        f"Tried:\n{tried_paths}"
    )

print(f"✅ Found HEAL CDE JSON: {HEAL_CDE_JSON_PATH}")


# ----------------------------
# Load official HEAL CDE JSON
# ----------------------------
with open(HEAL_CDE_JSON_PATH, "r", encoding="utf-8") as f:
    heal_cde_json = json.load(f)

print(f"✅ Loaded HEAL CDE JSON object type: {type(heal_cde_json).__name__}")
print(f"✅ Raw HEAL CDE JSON length: {len(heal_cde_json)}")


# ----------------------------
# Normalize JSON shape
# ----------------------------
# Supports both:
# 1. Dict-style JSON: {"VariableName": {...}, ...}
# 2. List-style flattened JSON: [{...}, {...}, ...]

if isinstance(heal_cde_json, dict):
    heal_cde_entries = list(heal_cde_json.items())

elif isinstance(heal_cde_json, list):
    heal_cde_entries = []

    for i, entry in enumerate(heal_cde_json):
        if not isinstance(entry, dict):
            continue

        entry_key = (
            norm_text_safe(entry.get("Variable Name"))
            or norm_text_safe(entry.get("variable_name"))
            or norm_text_safe(entry.get("CDE Name"))
            or norm_text_safe(entry.get("cde_name"))
            or f"row_{i}"
        )

        heal_cde_entries.append((entry_key, entry))

else:
    raise TypeError(
        f"Unsupported HEAL CDE JSON structure: {type(heal_cde_json)}. "
        "Expected dict or list."
    )

print(f"✅ Normalized HEAL CDE entries: {len(heal_cde_entries)}")


# ----------------------------
# Build flexible lookup map
# ----------------------------
official_lookup = {}


def add_to_lookup(key, entry_key, entry):
    """
    Add one lookup key to the official lookup dictionary.
    A key may point to multiple official entries, because some CDE Names repeat
    across variables, such as Race or SDOH.
    """
    normalized_key = normalize_lookup_key(key)

    if not normalized_key:
        return

    official_lookup.setdefault(normalized_key, [])

    official_lookup[normalized_key].append({
        "json_key": entry_key,
        "entry": entry
    })


for entry_key, entry in heal_cde_entries:
    # Entry key is often the official variable name
    add_to_lookup(entry_key, entry_key, entry)

    # Also index common official fields
    for field in [
        "Variable Name",
        "variable_name",
        "CDE Name",
        "cde_name",
        "Short Description",
        "short_description",
        "Additional Notes (Question Text)",
        "additional_notes_question_text",
    ]:
        value = entry.get(field)
        add_to_lookup(value, entry_key, entry)


print(f"✅ Built official lookup keys: {len(official_lookup)}")


# ----------------------------
# Helper: get official field from possible names
# ----------------------------
def get_entry_value(entry, possible_fields, fallback=None):
    """
    Pull a value from an official KB entry using several possible field names.
    Helpful because flattened JSON may use slightly different names.
    """
    for field in possible_fields:
        if field not in entry:
            continue

        value = entry.get(field)
        clean_value = norm_text_safe(value)

        if clean_value:
            return value

    return fallback


# ----------------------------
# Helper: summarize official KB entry
# ----------------------------
def summarize_official_entry(entry_key, entry):
    """
    Convert a full official KB entry into a compact packet for LLM adjudication.
    Keep only fields useful for choosing among candidates.
    """
    return {
        "official_json_key": entry_key,
        "official_cde_name": get_entry_value(entry, ["CDE Name", "cde_name"]),
        "official_variable_name": get_entry_value(entry, ["Variable Name", "variable_name"]),
        "official_crf_name": get_entry_value(entry, ["CRF Name", "crf_name", "Form Name", "form_name"]),
        "official_domain": get_entry_value(entry, ["Domain", "domain"]),
        "official_definition": get_entry_value(entry, ["Definition", "definition"]),
        "official_short_description": get_entry_value(entry, ["Short Description", "short_description"]),
        "official_question_text": get_entry_value(
            entry,
            ["Additional Notes (Question Text)", "additional_notes_question_text", "Question Text", "question_text"]
        ),
        "official_permissible_values": get_entry_value(
            entry,
            ["Permissible Values", "permissible_values"]
        ),
        "official_pv_description": get_entry_value(
            entry,
            ["PV Description", "pv_description"]
        ),
        "official_data_type": get_entry_value(entry, ["Data Type", "data_type"]),
        "official_population": get_entry_value(entry, ["Population", "population"]),
        "official_classification": get_entry_value(entry, ["Classification", "classification"]),
    }


# ----------------------------
# Helper: find official entries for a candidate value
# ----------------------------
def find_official_candidates(candidate_value, max_entries_per_candidate=5):
    """
    Find official KB entries for a candidate value.
    Returns a list because a candidate CDE name may map to multiple variables.
    """
    normalized_value = normalize_lookup_key(candidate_value)

    if not normalized_value:
        return []

    matches = official_lookup.get(normalized_value, [])

    # Deduplicate by JSON key
    seen = set()
    deduped_matches = []

    for match in matches:
        entry_key = match["json_key"]
        if entry_key not in seen:
            seen.add(entry_key)
            deduped_matches.append(match)

    return deduped_matches[:max_entries_per_candidate]


# ----------------------------
# Candidate source columns from Step 4
# ----------------------------
candidate_source_specs = [
    {
        "rank": 1,
        "candidate_col": resolved_cols.get("final_concept_match"),
        "candidate_crf_col": resolved_cols.get("best_match_crf"),
        "candidate_score_col": resolved_cols.get("best_match_score"),
        "candidate_label": "Final HEAL CDE Concept Match",
    },
    {
        "rank": 2,
        "candidate_col": resolved_cols.get("potential_match_2"),
        "candidate_crf_col": resolved_cols.get("potential_match_2_crf"),
        "candidate_score_col": None,
        "candidate_label": "Potential Match 2",
    },
    {
        "rank": 3,
        "candidate_col": resolved_cols.get("potential_match_3"),
        "candidate_crf_col": resolved_cols.get("potential_match_3_crf"),
        "candidate_score_col": None,
        "candidate_label": "Potential Match 3",
    },
]

candidate_source_specs = [
    spec for spec in candidate_source_specs
    if spec["candidate_col"] is not None and spec["candidate_col"] in recon_candidate_df.columns
]

if not candidate_source_specs:
    raise ValueError(
        "No candidate source columns were found in recon_candidate_df. "
        "Check resolved_cols for final_concept_match, potential_match_2, and potential_match_3."
    )

print("\n🧭 Candidate source columns:")
for spec in candidate_source_specs:
    print(f"  Rank {spec['rank']}: {spec['candidate_col']}")


# ----------------------------
# Build official candidate packets for one row
# ----------------------------
def build_official_candidate_packets(row):
    """
    Build official candidate packets for a single recon row.

    Each packet combines:
    - the candidate value surfaced by fuzzy/VLMD matching
    - the source column/rank
    - any candidate CRF/score from the matching step
    - the official KB entry fields
    """
    packets = []
    lookup_notes = []

    for spec in candidate_source_specs:
        candidate_value = norm_text_safe(row.get(spec["candidate_col"]))

        if not candidate_value:
            continue

        candidate_crf_value = ""

        if spec.get("candidate_crf_col") is not None and spec.get("candidate_crf_col") in row.index:
            candidate_crf_value = norm_text_safe(row.get(spec["candidate_crf_col"]))

        candidate_score_value = None

        if spec.get("candidate_score_col") is not None and spec.get("candidate_score_col") in row.index:
            candidate_score_value = row.get(spec["candidate_score_col"])

        official_matches = find_official_candidates(candidate_value)

        if not official_matches:
            lookup_notes.append(
                f"No official KB entry found for {spec['candidate_label']}: {candidate_value}"
            )

            packets.append({
                "candidate_rank": spec["rank"],
                "candidate_source_label": spec["candidate_label"],
                "candidate_source_column": spec["candidate_col"],
                "candidate_input_value": candidate_value,
                "candidate_crf_from_matching": candidate_crf_value,
                "candidate_score_from_matching": candidate_score_value,
                "lookup_status": "No official KB entry found",
                "official_json_key": None,
                "official_cde_name": None,
                "official_variable_name": None,
                "official_crf_name": None,
                "official_domain": None,
                "official_definition": None,
                "official_short_description": None,
                "official_question_text": None,
                "official_permissible_values": None,
                "official_pv_description": None,
                "official_data_type": None,
                "official_population": None,
                "official_classification": None,
            })

            continue

        for official_match in official_matches:
            official_entry_key = official_match["json_key"]
            official_entry = official_match["entry"]

            official_packet = summarize_official_entry(
                official_entry_key,
                official_entry
            )

            packet = {
                "candidate_rank": spec["rank"],
                "candidate_source_label": spec["candidate_label"],
                "candidate_source_column": spec["candidate_col"],
                "candidate_input_value": candidate_value,
                "candidate_crf_from_matching": candidate_crf_value,
                "candidate_score_from_matching": candidate_score_value,
                "lookup_status": "Matched official KB entry",
            }

            packet.update(official_packet)
            packets.append(packet)

    return packets, lookup_notes


# ----------------------------
# Apply packet builder to recon_candidate_df
# ----------------------------
official_candidate_results = recon_candidate_df.apply(
    build_official_candidate_packets,
    axis=1,
    result_type="expand"
)

recon_candidate_df["recon_official_candidates"] = official_candidate_results[0]
recon_candidate_df["recon_candidate_lookup_notes"] = official_candidate_results[1]
recon_candidate_df["recon_official_candidate_count"] = recon_candidate_df[
    "recon_official_candidates"
].apply(
    lambda candidates: sum(
        1 for candidate in candidates
        if isinstance(candidate, dict)
        and candidate.get("lookup_status") == "Matched official KB entry"
    )
)


# ----------------------------
# Print summary
# ----------------------------
rows_with_official_candidates = (
    recon_candidate_df["recon_official_candidate_count"] > 0
).sum()

rows_missing_official_candidates = (
    recon_candidate_df["recon_official_candidate_count"] == 0
).sum()

print("\n✅ Built official candidate packets")
print(f"Rows in recon_candidate_df: {len(recon_candidate_df)}")
print(f"Rows with official candidates: {rows_with_official_candidates}")
print(f"Rows with missing official candidates: {rows_missing_official_candidates}")


# ----------------------------
# Preview candidate packet counts
# ----------------------------
preview_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
    "recon_official_candidate_count",
    "recon_candidate_lookup_notes",
]

preview_cols = [
    col for col in preview_cols
    if col is not None and col in recon_candidate_df.columns
]

display(recon_candidate_df[preview_cols].head(20))


# ----------------------------
# Preview one official candidate packet
# ----------------------------
if len(recon_candidate_df) > 0:
    first_candidates = recon_candidate_df.iloc[0]["recon_official_candidates"]

    print("\n🔎 First row official candidate packet preview:")

    if first_candidates:
        print(json.dumps(first_candidates[0], indent=2, ensure_ascii=False))
    else:
        print("No official candidates found for the first selected recon row.")

✅ Found HEAL CDE JSON: KnowledgeBase\All_HEALPAINCDEsDD_row_level_flattened.json
✅ Loaded HEAL CDE JSON object type: dict
✅ Raw HEAL CDE JSON length: 434
✅ Normalized HEAL CDE entries: 434
✅ Built official lookup keys: 1140

🧭 Candidate source columns:
  Rank 1: Final HEAL CDE Concept Match
  Rank 2: Potential Match 2 - CDE Name
  Rank 3: Potential Match 3 - CDE Name

✅ Built official candidate packets
Rows in recon_candidate_df: 12
Rows with official candidates: 12
Rows with missing official candidates: 0


,recon_source_index,name,Final HEAL CDE Concept Match,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name,recon_official_candidate_count,recon_candidate_lookup_notes
2,2,imhbirthwt,BRTHDTC,Sex,MARISTAT,6,[]
3,3,BPIAvgPainRtngScl,BPIAvgPainRatingScl,BPIAvgPain7dRtngScale,BPILeastPainRatingScl,5,[]
4,4,BPICurrentPainRtngScl,BPICurrentPainRatingScl,BPICurntPainRtngScale,BPILeastPainRatingScl,5,[]
5,5,BPILeastPnLst24HRtngScl,BPILeastPainRatingScl,BPIWorstPainRatingScl,BPIAvgPainRatingScl,6,[]
6,6,BPIWrstPnLast24HRtngScl,BPIWorstPainRatingScl,BPILeastPainRatingScl,BPIWrstPain7dRtngScale,5,[]
7,7,GAD2FeelNervScl,GAD2FeelNervScale,GAD2FeelNervScl,GAD7FeelAfrdScl,12,[]
8,8,GAD2NotStopWryScl,GAD2NotStopWryScl,GAD2NotStopWryScale,GAD7WryTooMchScl,12,[]
9,9,GAD7EasyAnnoyedScl,GAD7EasyAnnoyedScl,GAD7TroubRelxScl,GAD7TotScore,12,[]
10,10,GAD7FeelAfrdScl,GAD7FeelAfrdScl,GAD2FeelNervScl,GAD7TotScore,12,[]
11,11,GAD7RstlessScl,GAD7RstlessScl,GAD2NotStopWryScl,GAD7TotScore,12,[]



🔎 First row official candidate packet preview:
{
  "candidate_rank": 1,
  "candidate_source_label": "Final HEAL CDE Concept Match",
  "candidate_source_column": "Final HEAL CDE Concept Match",
  "candidate_input_value": "BRTHDTC",
  "candidate_crf_from_matching": "Demographics",
  "candidate_score_from_matching": 66.0,
  "lookup_status": "Matched official KB entry",
  "official_json_key": "Demographics__BRTHDTC__1",
  "official_cde_name": "Birth date",
  "official_variable_name": "BRTHDTC",
  "official_crf_name": "Demographics",
  "official_domain": null,
  "official_definition": "BRTHDTC",
  "official_short_description": "Date (and time, if applicable and known) the participant/subject was born",
  "official_question_text": "Date of birth",
  "official_permissible_values": null,
  "official_pv_description": [
    "YYYY-MON-DD"
  ],
  "official_data_type": "Date or Date & Time",
  "official_population": "Adult;Pediatric",
  "official_classification": "Core"
}


In [145]:
# ============================================================
# STEP 5b: Clean official candidate packets into plain strings
# ============================================================
# Goal:
# - Convert list-like official KB values into cleaner plain strings
# - Keep recon payloads readable for the LLM and human review
# - Do NOT call the LLM yet

import json
import pandas as pd


# ----------------------------
# Safety check
# ----------------------------
if "recon_candidate_df" not in globals():
    raise NameError("recon_candidate_df does not exist. Run Step 4 first.")

if "recon_official_candidates" not in recon_candidate_df.columns:
    raise NameError("recon_official_candidates does not exist. Run Step 5 first.")


# ----------------------------
# Helper: convert messy JSON-ish values to readable strings
# ----------------------------
def clean_packet_value(value):
    """
    Convert official KB values into prompt-friendly plain text.

    Behavior:
    - None/NaN -> ""
    - one-item list -> the item as text
    - multi-item list -> joined with " | "
    - dict -> JSON string
    - everything else -> stripped string
    """
    if value is None:
        return ""

    # Handle list-like values before pd.isna
    if isinstance(value, (list, tuple, set)):
        cleaned_items = []

        for item in value:
            cleaned_item = clean_packet_value(item)
            if cleaned_item:
                cleaned_items.append(cleaned_item)

        return " | ".join(cleaned_items)

    # Handle dict values
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


# ----------------------------
# Fields we want to clean inside each candidate packet
# ----------------------------
official_fields_to_clean = [
    "official_cde_name",
    "official_variable_name",
    "official_crf_name",
    "official_domain",
    "official_definition",
    "official_short_description",
    "official_question_text",
    "official_permissible_values",
    "official_pv_description",
    "official_data_type",
    "official_population",
    "official_classification",
    "candidate_input_value",
    "candidate_crf_from_matching",
    "candidate_score_from_matching",
]


# ----------------------------
# Clean one candidate packet
# ----------------------------
def clean_candidate_packet(packet):
    """
    Clean selected values inside a candidate packet while preserving all keys.
    """
    cleaned_packet = packet.copy()

    for field in official_fields_to_clean:
        if field in cleaned_packet:
            cleaned_packet[field] = clean_packet_value(cleaned_packet[field])

    return cleaned_packet


# ----------------------------
# Clean all candidate packets for one row
# ----------------------------
def clean_candidate_packet_list(candidate_packets):
    """
    Clean every packet in a row's candidate list.
    """
    if not isinstance(candidate_packets, list):
        return []

    return [
        clean_candidate_packet(packet)
        for packet in candidate_packets
        if isinstance(packet, dict)
    ]


# ----------------------------
# Apply cleanup
# ----------------------------
recon_candidate_df["recon_official_candidates_clean"] = recon_candidate_df[
    "recon_official_candidates"
].apply(clean_candidate_packet_list)

recon_candidate_df["recon_official_candidate_count_clean"] = recon_candidate_df[
    "recon_official_candidates_clean"
].apply(
    lambda candidates: sum(
        1 for candidate in candidates
        if candidate.get("lookup_status") == "Matched official KB entry"
    )
)


# ----------------------------
# Print summary
# ----------------------------
print("✅ Cleaned official candidate packets")
print(f"Rows in recon_candidate_df: {len(recon_candidate_df)}")
print(
    "Rows with clean official candidates:",
    (recon_candidate_df["recon_official_candidate_count_clean"] > 0).sum()
)
print(
    "Rows missing clean official candidates:",
    (recon_candidate_df["recon_official_candidate_count_clean"] == 0).sum()
)


# ----------------------------
# Preview one cleaned official candidate packet
# ----------------------------
if len(recon_candidate_df) > 0:
    first_clean_candidates = recon_candidate_df.iloc[0]["recon_official_candidates_clean"]

    print("\n🔎 First row CLEAN official candidate packet preview:")

    if first_clean_candidates:
        print(json.dumps(first_clean_candidates[0], indent=2, ensure_ascii=False))
    else:
        print("No clean official candidates found for the first selected recon row.")

✅ Cleaned official candidate packets
Rows in recon_candidate_df: 12
Rows with clean official candidates: 12
Rows missing clean official candidates: 0

🔎 First row CLEAN official candidate packet preview:
{
  "candidate_rank": 1,
  "candidate_source_label": "Final HEAL CDE Concept Match",
  "candidate_source_column": "Final HEAL CDE Concept Match",
  "candidate_input_value": "BRTHDTC",
  "candidate_crf_from_matching": "Demographics",
  "candidate_score_from_matching": "66.0",
  "lookup_status": "Matched official KB entry",
  "official_json_key": "Demographics__BRTHDTC__1",
  "official_cde_name": "Birth date",
  "official_variable_name": "BRTHDTC",
  "official_crf_name": "Demographics",
  "official_domain": "",
  "official_definition": "BRTHDTC",
  "official_short_description": "Date (and time, if applicable and known) the participant/subject was born",
  "official_question_text": "Date of birth",
  "official_permissible_values": "",
  "official_pv_description": "YYYY-MON-DD",
  "officia

In [146]:
# ============================================================
# STEP 5c: Inspect rows missing official candidate packets
# ============================================================
# Goal:
# - Identify which recon rows did not find any official KB candidate
# - Review candidate names and lookup notes before building payloads
# - Do NOT call the LLM yet

missing_official_df = recon_candidate_df[
    recon_candidate_df["recon_official_candidate_count_clean"] == 0
].copy()

print("Rows missing clean official candidates:", len(missing_official_df))

missing_preview_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("form_name"),
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_status"),
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
    "recon_candidate_lookup_notes",
]

missing_preview_cols = [
    col for col in missing_preview_cols
    if col is not None and col in missing_official_df.columns
]

display(missing_official_df[missing_preview_cols])

Rows missing clean official candidates: 0


,recon_source_index,name,HEAL Core CRF Match,Final Concept Match Status,Final HEAL CDE Concept Match,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name,recon_candidate_lookup_notes


In [147]:
# ============================================================
# STEP 5d: Apply protected-family recon guardrails
# ============================================================
# Goal:
# - Detect rows where the pre-step CRF match is copyright/protected-family sensitive
# - Prevent off-family candidates from being sent to LLM adjudication
# - Preserve form-level evidence without forcing a variable-level CDE match
# - Do NOT call the LLM yet

import re
import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "recon_candidate_df" not in globals():
    raise NameError("recon_candidate_df does not exist. Run Step 4 first.")

if "resolved_cols" not in globals():
    raise NameError("resolved_cols does not exist. Run Step 2 first.")

if "config" not in globals():
    raise NameError("config does not exist. Run Step 3 first.")

if "recon_official_candidates_clean" not in recon_candidate_df.columns:
    raise NameError("recon_official_candidates_clean does not exist. Run Step 5b first.")


# ----------------------------
# Helper: normalize comparison text
# ----------------------------
def normalize_family_text(value):
    """
    Normalize family/instrument text for protected-family comparisons.
    Examples:
    - "PCS-6" -> "pcs 6"
    - "Brief Pain Inventory (BPI)" -> "brief pain inventory bpi"
    """
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ----------------------------
# Helper: split config pipe values
# ----------------------------
def split_pipe_terms(value):
    """
    Split config values like:
    bpi pain interference = brief pain inventory (bpi) | bpi pain interference
    """
    if value is None:
        return []

    return [
        normalize_family_text(term)
        for term in str(value).split("|")
        if normalize_family_text(term)
    ]


# ----------------------------
# Load protected-family aliases from config
# ----------------------------
protected_family_aliases = {}

# These are possible sections where protected/copyright-sensitive rules may live.
# The cell will use whichever ones exist in your config.
candidate_protected_sections = [
    "ProtectedPrimaryFamilies",
    "ProtectedFamilies",
    "CopyrightSensitiveFamilies",
    "SkipVLMDMatching",
    "skipVLMDMatching",
]

for section in candidate_protected_sections:
    if not config.has_section(section):
        continue

    for key, value in config.items(section):
        canonical_key = normalize_family_text(key)
        alias_terms = split_pipe_terms(value)

        # Include the key itself as an alias too
        aliases = sorted(set([canonical_key] + alias_terms))

        if canonical_key:
            protected_family_aliases[canonical_key] = aliases


# ----------------------------
# Add conservative fallbacks
# ----------------------------
# These guardrails cover known copyright/protected families.
# They will not override config; they only help if the section is missing/incomplete.
fallback_protected_families = {
    "brief pain inventory bpi": [
        "bpi",
        "brief pain inventory",
        "brief pain inventory bpi",
    ],
    "bpi pain interference": [
        "bpi",
        "brief pain inventory",
        "brief pain inventory bpi",
        "bpi pain interference",
        "bpi interference",
        "pain interference",
    ],
    "bpi pain severity": [
        "bpi",
        "brief pain inventory",
        "brief pain inventory bpi",
        "bpi pain severity",
        "bpi severity",
        "pain severity",
    ],
    "pcs 6": [
        "pcs 6",
        "pcs6",
        "pain catastrophizing scale 6",
    ],
    "pcs 13": [
        "pcs 13",
        "pcs13",
        "pain catastrophizing scale 13",
    ],
    "pedsql inventory": [
        "pedsql",
        "pediatric quality of life inventory",
        "pedsql pediatric quality of life inventory",
        "pedsql inventory",
    ],
}

for canonical_key, aliases in fallback_protected_families.items():
    canonical_key_norm = normalize_family_text(canonical_key)
    alias_terms_norm = [
        normalize_family_text(alias)
        for alias in aliases
        if normalize_family_text(alias)
    ]

    if canonical_key_norm not in protected_family_aliases:
        protected_family_aliases[canonical_key_norm] = alias_terms_norm
    else:
        protected_family_aliases[canonical_key_norm] = sorted(
            set(protected_family_aliases[canonical_key_norm] + alias_terms_norm)
        )


print("✅ Loaded protected-family aliases:")
for family, aliases in protected_family_aliases.items():
    print(f"  {family}: {aliases}")


# ----------------------------
# Resolve useful columns
# ----------------------------
prestep_crf_col = resolved_cols.get("prestep_crf_match")
variable_col = resolved_cols.get("variable_name")
form_col = resolved_cols.get("form_name")

if prestep_crf_col is None:
    raise KeyError(
        "Could not resolve prestep_crf_match column. "
        "Check resolved_cols['prestep_crf_match']."
    )


# ----------------------------
# Helper: find protected family for a row
# ----------------------------
def find_protected_family_for_row(row):
    """
    Check whether the row's HEAL Core CRF Match belongs to a protected family.

    Returns:
    - protected_family_hit: bool
    - protected_family_anchor: matched protected family key
    - protected_family_aliases_for_anchor: aliases for that family
    """
    crf_text = normalize_family_text(row.get(prestep_crf_col))

    if not crf_text:
        return False, "", []

    # Prefer more specific family keys first.
    # Example: bpi pain interference should match before broad bpi.
    sorted_families = sorted(
        protected_family_aliases.items(),
        key=lambda item: len(item[0]),
        reverse=True
    )

    for family_key, aliases in sorted_families:
        family_key_norm = normalize_family_text(family_key)

        # Match either the family key or any aliases against the pre-step CRF text
        all_terms = sorted(
            set([family_key_norm] + aliases),
            key=len,
            reverse=True
        )

        for term in all_terms:
            if not term:
                continue

            if term in crf_text or crf_text in term:
                return True, family_key, aliases

    return False, "", []


# ----------------------------
# Helper: classify whether candidate packets are in-family
# ----------------------------
def candidate_packet_is_in_family(packet, protected_aliases):
    """
    Decide whether one official candidate packet appears to stay within the protected family.

    Uses conservative signals:
    - candidate CRF from matching
    - candidate input value
    - official variable name
    - official CDE name
    - official short description
    - official definition
    - official domain

    This is not fuzzy matching. It only checks whether protected-family aliases appear
    in candidate text.
    """
    if not isinstance(packet, dict):
        return False

    text_fields = [
        "candidate_crf_from_matching",
        "candidate_input_value",
        "official_variable_name",
        "official_cde_name",
        "official_short_description",
        "official_definition",
        "official_domain",
    ]

    combined_text = " ".join(
        normalize_family_text(packet.get(field))
        for field in text_fields
        if packet.get(field) is not None
    )

    if not combined_text:
        return False

    aliases = [
        normalize_family_text(alias)
        for alias in protected_aliases
        if normalize_family_text(alias)
    ]

    # Extra instrument-specific hints.
    # These help identify in-family candidates even if CRF name is blank in the flattened JSON.

    # BPI
    if any(alias in ["bpi", "brief pain inventory", "brief pain inventory bpi"] for alias in aliases):
        if "bpi" in combined_text or "brief pain inventory" in combined_text:
            return True

    # PedsQL
    if any(alias in ["pedsql", "pediatric quality of life inventory"] for alias in aliases):
        if "pedsql" in combined_text or "pediatric quality of life inventory" in combined_text:
            return True

    # PCS-6
    if any(alias in ["pcs 6", "pcs6", "pain catastrophizing scale 6"] for alias in aliases):
        if (
            "pcs 6" in combined_text
            or "pcs6" in combined_text
            or "pain catastrophizing scale 6" in combined_text
        ):
            return True

    # PCS-13
    if any(alias in ["pcs 13", "pcs13", "pain catastrophizing scale 13"] for alias in aliases):
        if (
            "pcs 13" in combined_text
            or "pcs13" in combined_text
            or "pain catastrophizing scale 13" in combined_text
        ):
            return True

    # General alias containment check
    for alias in aliases:
        if alias and alias in combined_text:
            return True

    return False


# ----------------------------
# Helper: apply protected-family guardrail to one row
# ----------------------------
def apply_protected_family_guardrail(row):
    """
    Decide whether recon should skip LLM adjudication for protected-family cases.
    """
    protected_hit, protected_anchor, protected_aliases_for_anchor = find_protected_family_for_row(row)

    candidates = row.get("recon_official_candidates_clean", [])
    if not isinstance(candidates, list):
        candidates = []

    official_candidates = [
        packet for packet in candidates
        if isinstance(packet, dict) and packet.get("lookup_status") == "Matched official KB entry"
    ]

    official_candidate_count = len(official_candidates)

    in_family_candidates = [
        packet for packet in official_candidates
        if candidate_packet_is_in_family(packet, protected_aliases_for_anchor)
    ]

    off_family_candidates = [
        packet for packet in official_candidates
        if not candidate_packet_is_in_family(packet, protected_aliases_for_anchor)
    ]

    in_family_count = len(in_family_candidates)
    off_family_count = len(off_family_candidates)

    skip_llm = False
    review_flag = ""
    rationale = ""

    if protected_hit and official_candidate_count == 0:
        skip_llm = True
        review_flag = "Protected Family - Form-Level Match Only"
        rationale = (
            "Row has a protected-family CRF match, but no official in-family candidate "
            "was available in the knowledge base."
        )

    elif protected_hit and official_candidate_count > 0 and in_family_count == 0:
        skip_llm = True
        review_flag = "Protected Family - Off-Family Candidates Blocked"
        rationale = (
            "Row has a protected-family CRF match, but surfaced official candidates appear "
            "to be outside the protected family."
        )

    elif protected_hit and in_family_count > 0:
        # Keep available for payload building, but flagged for protected-family awareness.
        # Later recon prompt should still treat this as restricted/review-sensitive.
        skip_llm = False
        review_flag = "Protected Family - In-Family Candidate Available"
        rationale = (
            "Row has a protected-family CRF match and at least one candidate appears "
            "to remain within the protected family."
        )

    else:
        skip_llm = False
        review_flag = ""
        rationale = ""

    return pd.Series({
        "recon_protected_family_hit": protected_hit,
        "recon_protected_family_anchor": protected_anchor,
        "recon_protected_family_aliases": protected_aliases_for_anchor,
        "recon_official_candidate_count_for_guardrail": official_candidate_count,
        "recon_in_family_candidate_count": in_family_count,
        "recon_off_family_candidate_count": off_family_count,
        "recon_skip_llm": skip_llm,
        "recon_review_flag": review_flag,
        "recon_preassigned_decision": "Needs Human Review" if skip_llm else "",
        "recon_preassigned_confidence": "Low" if skip_llm else "",
        "recon_preassigned_cde": "",
        "recon_preassigned_crf": row.get(prestep_crf_col, ""),
        "recon_preassigned_variable": "",
        "recon_preassigned_rationale": rationale,
    })


# ----------------------------
# Apply guardrails
# ----------------------------
guardrail_results = recon_candidate_df.apply(
    apply_protected_family_guardrail,
    axis=1
)

for col in guardrail_results.columns:
    recon_candidate_df[col] = guardrail_results[col]


# ----------------------------
# Print summary
# ----------------------------
print("\n✅ Applied protected-family recon guardrails")
print(f"Rows in recon_candidate_df: {len(recon_candidate_df)}")
print(f"Protected-family rows detected: {recon_candidate_df['recon_protected_family_hit'].sum()}")
print(f"Rows preassigned to skip LLM: {recon_candidate_df['recon_skip_llm'].sum()}")

print("\n📊 Review flag counts:")
display(
    recon_candidate_df["recon_review_flag"]
    .fillna("")
    .replace("", "(none)")
    .value_counts()
    .rename_axis("recon_review_flag")
    .reset_index(name="row_count")
)


# ----------------------------
# Preview protected-family guardrail decisions
# ----------------------------
preview_cols = [
    "recon_source_index",
    variable_col,
    form_col,
    prestep_crf_col,
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
    "recon_official_candidate_count_clean",
    "recon_protected_family_hit",
    "recon_protected_family_anchor",
    "recon_in_family_candidate_count",
    "recon_off_family_candidate_count",
    "recon_skip_llm",
    "recon_review_flag",
    "recon_preassigned_decision",
    "recon_preassigned_crf",
    "recon_preassigned_rationale",
]

preview_cols = [
    col for col in preview_cols
    if col is not None and col in recon_candidate_df.columns
]

display(recon_candidate_df[preview_cols])

✅ Loaded protected-family aliases:
  brief pain inventory bpi: ['bpi', 'brief pain inventory', 'brief pain inventory bpi']
  bpi pain interference: ['bpi', 'bpi interference', 'bpi pain interference', 'brief pain inventory', 'brief pain inventory bpi', 'pain interference']
  bpi pain severity: ['bpi', 'bpi pain severity', 'bpi severity', 'brief pain inventory', 'brief pain inventory bpi', 'pain severity']
  pcs 6: ['pain catastrophizing scale 6', 'pcs 6', 'pcs6']
  pcs 13: ['pain catastrophizing scale 13', 'pcs 13', 'pcs13']
  pedsql inventory: ['pediatric quality of life inventory', 'pedsql', 'pedsql inventory', 'pedsql pediatric quality of life inventory']

✅ Applied protected-family recon guardrails
Rows in recon_candidate_df: 12
Protected-family rows detected: 4
Rows preassigned to skip LLM: 0

📊 Review flag counts:


,recon_review_flag,row_count
0,(none),8
1,Protected Family - In-Family Candidate Available,4


,recon_source_index,name,HEAL Core CRF Match,Final HEAL CDE Concept Match,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name,recon_official_candidate_count_clean,recon_protected_family_hit,recon_protected_family_anchor,recon_in_family_candidate_count,recon_off_family_candidate_count,recon_skip_llm,recon_review_flag,recon_preassigned_decision,recon_preassigned_crf,recon_preassigned_rationale
2,2,imhbirthwt,Demographics,BRTHDTC,Sex,MARISTAT,6,False,,0,6,False,,,Demographics,
3,3,BPIAvgPainRtngScl,BPI Pain Severity,BPIAvgPainRatingScl,BPIAvgPain7dRtngScale,BPILeastPainRatingScl,5,True,brief pain inventory bpi,5,0,False,Protected Family - In-Family Candidate Available,,BPI Pain Severity,Row has a protected-family CRF match and at le...
4,4,BPICurrentPainRtngScl,BPI Pain Severity,BPICurrentPainRatingScl,BPICurntPainRtngScale,BPILeastPainRatingScl,5,True,brief pain inventory bpi,5,0,False,Protected Family - In-Family Candidate Available,,BPI Pain Severity,Row has a protected-family CRF match and at le...
5,5,BPILeastPnLst24HRtngScl,BPI Pain Severity,BPILeastPainRatingScl,BPIWorstPainRatingScl,BPIAvgPainRatingScl,6,True,brief pain inventory bpi,6,0,False,Protected Family - In-Family Candidate Available,,BPI Pain Severity,Row has a protected-family CRF match and at le...
6,6,BPIWrstPnLast24HRtngScl,BPI Pain Severity,BPIWorstPainRatingScl,BPILeastPainRatingScl,BPIWrstPain7dRtngScale,5,True,brief pain inventory bpi,5,0,False,Protected Family - In-Family Candidate Available,,BPI Pain Severity,Row has a protected-family CRF match and at le...
7,7,GAD2FeelNervScl,GAD2 Pain (Generalized Anxiety Disorder),GAD2FeelNervScale,GAD2FeelNervScl,GAD7FeelAfrdScl,12,False,,0,12,False,,,GAD2 Pain (Generalized Anxiety Disorder),
8,8,GAD2NotStopWryScl,No CRF match,GAD2NotStopWryScl,GAD2NotStopWryScale,GAD7WryTooMchScl,12,False,,0,12,False,,,No CRF match,
9,9,GAD7EasyAnnoyedScl,GAD7,GAD7EasyAnnoyedScl,GAD7TroubRelxScl,GAD7TotScore,12,False,,0,12,False,,,GAD7,
10,10,GAD7FeelAfrdScl,GAD7,GAD7FeelAfrdScl,GAD2FeelNervScl,GAD7TotScore,12,False,,0,12,False,,,GAD7,
11,11,GAD7RstlessScl,GAD7,GAD7RstlessScl,GAD2NotStopWryScl,GAD7TotScore,12,False,,0,12,False,,,GAD7,


In [148]:
# ============================================================
# STEP 6: Build recon/adjudication payloads
# ============================================================
# Goal:
# - Build structured row-level payloads for LLM reconciliation
# - Exclude rows preassigned to skip LLM by protected-family guardrails
# - Preserve row context, matching context, protected-family context, and official candidates
# - Pull original input columns from config [Columns]
# - Do NOT call the LLM yet

import json
import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "recon_candidate_df" not in globals():
    raise NameError("recon_candidate_df does not exist. Run Step 4 first.")

if "resolved_cols" not in globals():
    raise NameError("resolved_cols does not exist. Run Step 2 first.")

if "config" not in globals():
    raise NameError("config does not exist. Run Step 3 first.")

if "RECON_ADJUDICATION_INSTRUCTION" not in globals():
    raise NameError("RECON_ADJUDICATION_INSTRUCTION does not exist. Run Step 3 first.")

if "recon_official_candidates_clean" not in recon_candidate_df.columns:
    raise NameError("recon_official_candidates_clean does not exist. Run Step 5b first.")

if "recon_skip_llm" not in recon_candidate_df.columns:
    raise NameError("recon_skip_llm does not exist. Run Step 5d first.")


# ============================================================
# Pull original study data dictionary columns from config
# ============================================================

CONFIG_VARIABLE_COL = config.get("Columns", "variable_column", fallback="").strip()
CONFIG_FORM_COL = config.get("Columns", "crf_column", fallback="").strip()
CONFIG_DESCRIPTION_COL = config.get("Columns", "description_column", fallback="").strip()
CONFIG_ENCODING_COL = config.get("Columns", "ENCODING_COLUMN", fallback="").strip()

print("🧭 Step 6 config-driven original columns:")
print(f"  variable_column: {CONFIG_VARIABLE_COL}")
print(f"  crf_column/form column: {CONFIG_FORM_COL}")
print(f"  description_column: {CONFIG_DESCRIPTION_COL}")
print(f"  ENCODING_COLUMN: {CONFIG_ENCODING_COL}")


# ----------------------------
# Helper: safe plain text
# ----------------------------
def payload_text(value):
    """
    Convert values into prompt-safe text.
    """
    if value is None:
        return ""

    if isinstance(value, (list, tuple, set)):
        cleaned_items = [payload_text(item) for item in value if payload_text(item)]
        return " | ".join(cleaned_items)

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


# ----------------------------
# Helper: get row value from direct column name
# ----------------------------
def get_direct_column_value(row, col_name):
    """
    Pull a row value using a direct dataframe column name.
    This is best for original input columns from config.
    """
    if not col_name:
        return ""

    if col_name not in row.index:
        return ""

    return payload_text(row.get(col_name))


# ----------------------------
# Helper: get row value from resolved column
# ----------------------------
def get_resolved_value(row, resolved_key):
    """
    Pull a row value using resolved_cols.
    Returns blank if the column was not resolved or is not present.
    Best for generated pipeline columns.
    """
    col = resolved_cols.get(resolved_key)

    if col is None:
        return ""

    if col not in row.index:
        return ""

    return payload_text(row.get(col))


# ----------------------------
# Helper: reduce candidate packets to fields needed for adjudication
# ----------------------------
def build_payload_candidate_list(candidate_packets):
    """
    Keep candidate packets compact and consistent for the LLM.
    """
    if not isinstance(candidate_packets, list):
        return []

    payload_candidates = []

    for packet in candidate_packets:
        if not isinstance(packet, dict):
            continue

        # Keep only matched official KB entries for adjudication.
        # Non-matched notes are still preserved elsewhere in lookup notes.
        if packet.get("lookup_status") != "Matched official KB entry":
            continue

        payload_candidates.append({
            "candidate_rank": packet.get("candidate_rank"),
            "candidate_source_label": payload_text(packet.get("candidate_source_label")),
            "candidate_input_value": payload_text(packet.get("candidate_input_value")),
            "candidate_crf_from_matching": payload_text(packet.get("candidate_crf_from_matching")),
            "candidate_score_from_matching": payload_text(packet.get("candidate_score_from_matching")),
            "official_json_key": payload_text(packet.get("official_json_key")),
            "official_cde_name": payload_text(packet.get("official_cde_name")),
            "official_variable_name": payload_text(packet.get("official_variable_name")),
            "official_crf_name": payload_text(packet.get("official_crf_name")),
            "official_domain": payload_text(packet.get("official_domain")),
            "official_definition": payload_text(packet.get("official_definition")),
            "official_short_description": payload_text(packet.get("official_short_description")),
            "official_question_text": payload_text(packet.get("official_question_text")),
            "official_permissible_values": payload_text(packet.get("official_permissible_values")),
            "official_pv_description": payload_text(packet.get("official_pv_description")),
            "official_data_type": payload_text(packet.get("official_data_type")),
            "official_population": payload_text(packet.get("official_population")),
            "official_classification": payload_text(packet.get("official_classification")),
        })

    return payload_candidates


# ----------------------------
# Helper: build one row payload
# ----------------------------
def build_recon_payload(row):
    """
    Build the structured case file for one row.
    This is what will be sent to the LLM in the next step.
    """
    official_candidates = build_payload_candidate_list(
        row.get("recon_official_candidates_clean", [])
    )

    payload = {
        "task": "Choose the best official HEAL CDE match from the provided candidates only.",
        "rules": {
            "candidate_constraint": "Choose only from official_candidates. Do not invent CDE names, CRF names, or variable names.",
            "no_candidate_rule": "If no official candidate is appropriate, return No Match or Needs Human Review.",
            "protected_family_rule": "If protected_family_context indicates a protected/copyright-sensitive family, do not choose off-family candidates.",
            "output_format": "Return only valid JSON using the configured response schema."
        },
        "study_row_context": {
            "recon_source_index": payload_text(row.get("recon_source_index")),

            # Config-driven original input fields
            "study_variable_name": get_direct_column_value(row, CONFIG_VARIABLE_COL),
            "study_form_name": get_direct_column_value(row, CONFIG_FORM_COL),
            "study_question_text": get_direct_column_value(row, CONFIG_DESCRIPTION_COL),
            "study_encoding": get_direct_column_value(row, CONFIG_ENCODING_COL),

            # Pipeline-derived fields, if available
            "study_normalized_text": get_resolved_value(row, "normalized_text"),
            "study_normalized_encoding": get_resolved_value(row, "normalized_encoding"),
        },
        "prestep_context": {
            "heal_core_crf_match": get_resolved_value(row, "prestep_crf_match"),
            "prestep_confidence": get_resolved_value(row, "prestep_confidence"),
            "prestep_rationale": get_resolved_value(row, "match_rationale"),
        },
        "matching_context": {
            "final_concept_status": get_resolved_value(row, "final_concept_status"),
            "final_concept_match": get_resolved_value(row, "final_concept_match"),
            "best_match_score": get_resolved_value(row, "best_match_score"),
            "best_match_crf": get_resolved_value(row, "best_match_crf"),
            "encoding_fidelity_score": get_resolved_value(row, "encoding_fidelity_score"),
            "potential_match_2": get_resolved_value(row, "potential_match_2"),
            "potential_match_2_crf": get_resolved_value(row, "potential_match_2_crf"),
            "potential_match_3": get_resolved_value(row, "potential_match_3"),
            "potential_match_3_crf": get_resolved_value(row, "potential_match_3_crf"),
            "best_match_source": get_resolved_value(row, "best_match_source"),
        },
        "protected_family_context": {
            "protected_family_hit": bool(row.get("recon_protected_family_hit", False)),
            "protected_family_anchor": payload_text(row.get("recon_protected_family_anchor")),
            "protected_family_aliases": row.get("recon_protected_family_aliases", []),
            "review_flag": payload_text(row.get("recon_review_flag")),
            "in_family_candidate_count": payload_text(row.get("recon_in_family_candidate_count")),
            "off_family_candidate_count": payload_text(row.get("recon_off_family_candidate_count")),
            "skip_llm": bool(row.get("recon_skip_llm", False)),
            "preassigned_decision": payload_text(row.get("recon_preassigned_decision")),
            "preassigned_rationale": payload_text(row.get("recon_preassigned_rationale")),
        },
        "candidate_lookup_context": {
            "official_candidate_count": payload_text(row.get("recon_official_candidate_count_clean")),
            "lookup_notes": row.get("recon_candidate_lookup_notes", []),
        },
        "official_candidates": official_candidates,
    }

    return payload


# ----------------------------
# Select rows that should go to LLM recon
# ----------------------------
recon_payload_df = recon_candidate_df[
    recon_candidate_df["recon_skip_llm"] == False
].copy()

# Also require at least one clean official candidate.
# Rows with no official candidates should already be guarded, but this keeps the payload step safe.
recon_payload_df = recon_payload_df[
    recon_payload_df["recon_official_candidate_count_clean"] > 0
].copy()


# ----------------------------
# Build payloads
# ----------------------------
recon_payload_df["recon_payload"] = recon_payload_df.apply(
    build_recon_payload,
    axis=1
)

recon_payload_df["recon_payload_candidate_count"] = recon_payload_df[
    "recon_payload"
].apply(
    lambda payload: len(payload.get("official_candidates", []))
)


# ----------------------------
# Print summary
# ----------------------------
print("✅ Built recon payloads")
print(f"Rows in recon_candidate_df: {len(recon_candidate_df)}")
print(f"Rows skipped by protected-family guardrail: {recon_candidate_df['recon_skip_llm'].sum()}")
print(f"Rows available for LLM recon payloads: {len(recon_payload_df)}")
print(
    "Rows with payload candidates:",
    (recon_payload_df["recon_payload_candidate_count"] > 0).sum()
)


# ----------------------------
# Preview payload rows
# ----------------------------
preview_cols = [
    "recon_source_index",
    CONFIG_VARIABLE_COL,
    CONFIG_FORM_COL,
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_status"),
    resolved_cols.get("final_concept_match"),
    "recon_review_flag",
    "recon_payload_candidate_count",
]

preview_cols = [
    col for col in preview_cols
    if col is not None and col in recon_payload_df.columns
]

display(recon_payload_df[preview_cols])


# ----------------------------
# Preview one full payload
# ----------------------------
if len(recon_payload_df) > 0:
    print("\n🔎 First recon payload preview:")
    print(json.dumps(recon_payload_df.iloc[0]["recon_payload"], indent=2, ensure_ascii=False))
else:
    print("\nNo rows available for LLM recon payloads. All selected rows were skipped or lacked official candidates.")

🧭 Step 6 config-driven original columns:
  variable_column: name
  crf_column/form column: section
  description_column: description
  ENCODING_COLUMN: enumLabels
✅ Built recon payloads
Rows in recon_candidate_df: 12
Rows skipped by protected-family guardrail: 0
Rows available for LLM recon payloads: 12
Rows with payload candidates: 12


,recon_source_index,name,section,HEAL Core CRF Match,Final Concept Match Status,Final HEAL CDE Concept Match,recon_review_flag,recon_payload_candidate_count
2,2,imhbirthwt,a_infant_medical_history_01_month,Demographics,Possible concept match,BRTHDTC,,6
3,3,BPIAvgPainRtngScl,bpisev,BPI Pain Severity,High concept match,BPIAvgPainRatingScl,Protected Family - In-Family Candidate Available,5
4,4,BPICurrentPainRtngScl,bpisev,BPI Pain Severity,Possible concept match,BPICurrentPainRatingScl,Protected Family - In-Family Candidate Available,5
5,5,BPILeastPnLst24HRtngScl,bpisev,BPI Pain Severity,High concept match,BPILeastPainRatingScl,Protected Family - In-Family Candidate Available,6
6,6,BPIWrstPnLast24HRtngScl,bpisev,BPI Pain Severity,High concept match,BPIWorstPainRatingScl,Protected Family - In-Family Candidate Available,5
7,7,GAD2FeelNervScl,gad7,GAD2 Pain (Generalized Anxiety Disorder),High concept match,GAD2FeelNervScale,,12
8,8,GAD2NotStopWryScl,gad7,No CRF match,Possible concept match,GAD2NotStopWryScl,,12
9,9,GAD7EasyAnnoyedScl,gad7,GAD7,High concept match,GAD7EasyAnnoyedScl,,12
10,10,GAD7FeelAfrdScl,gad7,GAD7,High concept match,GAD7FeelAfrdScl,,12
11,11,GAD7RstlessScl,gad7,GAD7,High concept match,GAD7RstlessScl,,12



🔎 First recon payload preview:
{
  "task": "Choose the best official HEAL CDE match from the provided candidates only.",
  "rules": {
    "candidate_constraint": "Choose only from official_candidates. Do not invent CDE names, CRF names, or variable names.",
    "no_candidate_rule": "If no official candidate is appropriate, return No Match or Needs Human Review.",
    "protected_family_rule": "If protected_family_context indicates a protected/copyright-sensitive family, do not choose off-family candidates.",
    "output_format": "Return only valid JSON using the configured response schema."
  },
  "study_row_context": {
    "recon_source_index": "2",
    "study_variable_name": "imhbirthwt",
    "study_form_name": "a_infant_medical_history_01_month",
    "study_question_text": "4. Weight at birth",
    "study_encoding": "",
    "study_normalized_text": "",
    "study_normalized_encoding": ""
  },
  "prestep_context": {
    "heal_core_crf_match": "Demographics",
    "prestep_confidence":

In [149]:
# ============================================================
# STEP 6b: Review rows skipped vs rows sent to LLM recon
# ============================================================
# Goal:
# - Confirm protected-family guardrails are behaving as expected
# - Preview which rows will be skipped and which rows will be sent to LLM recon
# - Do NOT call the LLM yet

# ----------------------------
# Rows skipped by guardrail
# ----------------------------
skipped_recon_df = recon_candidate_df[
    recon_candidate_df["recon_skip_llm"] == True
].copy()

print("🛑 Rows skipped by protected-family guardrail:", len(skipped_recon_df))

skip_preview_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("form_name"),
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
    "recon_official_candidate_count_clean",
    "recon_protected_family_anchor",
    "recon_review_flag",
    "recon_preassigned_decision",
    "recon_preassigned_crf",
    "recon_preassigned_rationale",
]

skip_preview_cols = [
    col for col in skip_preview_cols
    if col is not None and col in skipped_recon_df.columns
]

display(skipped_recon_df[skip_preview_cols])


# ----------------------------
# Rows going to LLM recon
# ----------------------------
print("\n✅ Rows prepared for LLM recon:", len(recon_payload_df))

send_preview_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("form_name"),
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
    "recon_official_candidate_count_clean",
    "recon_review_flag",
    "recon_payload_candidate_count",
]

send_preview_cols = [
    col for col in send_preview_cols
    if col is not None and col in recon_payload_df.columns
]

display(recon_payload_df[send_preview_cols])

🛑 Rows skipped by protected-family guardrail: 0


,recon_source_index,name,HEAL Core CRF Match,Final HEAL CDE Concept Match,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name,recon_official_candidate_count_clean,recon_protected_family_anchor,recon_review_flag,recon_preassigned_decision,recon_preassigned_crf,recon_preassigned_rationale



✅ Rows prepared for LLM recon: 12


,recon_source_index,name,HEAL Core CRF Match,Final HEAL CDE Concept Match,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name,recon_official_candidate_count_clean,recon_review_flag,recon_payload_candidate_count
2,2,imhbirthwt,Demographics,BRTHDTC,Sex,MARISTAT,6,,6
3,3,BPIAvgPainRtngScl,BPI Pain Severity,BPIAvgPainRatingScl,BPIAvgPain7dRtngScale,BPILeastPainRatingScl,5,Protected Family - In-Family Candidate Available,5
4,4,BPICurrentPainRtngScl,BPI Pain Severity,BPICurrentPainRatingScl,BPICurntPainRtngScale,BPILeastPainRatingScl,5,Protected Family - In-Family Candidate Available,5
5,5,BPILeastPnLst24HRtngScl,BPI Pain Severity,BPILeastPainRatingScl,BPIWorstPainRatingScl,BPIAvgPainRatingScl,6,Protected Family - In-Family Candidate Available,6
6,6,BPIWrstPnLast24HRtngScl,BPI Pain Severity,BPIWorstPainRatingScl,BPILeastPainRatingScl,BPIWrstPain7dRtngScale,5,Protected Family - In-Family Candidate Available,5
7,7,GAD2FeelNervScl,GAD2 Pain (Generalized Anxiety Disorder),GAD2FeelNervScale,GAD2FeelNervScl,GAD7FeelAfrdScl,12,,12
8,8,GAD2NotStopWryScl,No CRF match,GAD2NotStopWryScl,GAD2NotStopWryScale,GAD7WryTooMchScl,12,,12
9,9,GAD7EasyAnnoyedScl,GAD7,GAD7EasyAnnoyedScl,GAD7TroubRelxScl,GAD7TotScore,12,,12
10,10,GAD7FeelAfrdScl,GAD7,GAD7FeelAfrdScl,GAD2FeelNervScl,GAD7TotScore,12,,12
11,11,GAD7RstlessScl,GAD7,GAD7RstlessScl,GAD2NotStopWryScl,GAD7TotScore,12,,12


In [150]:
# ============================================================
# STEP 7: Run LLM recon/adjudication on prepared payloads
# ============================================================
# Goal:
# - Send each recon payload to the LLM
# - Require structured JSON output
# - Parse and store recon/adjudication results
# - Do NOT export yet

import os
import json
import time
import pandas as pd
from openai import OpenAI


# ----------------------------
# Safety checks
# ----------------------------
if "recon_payload_df" not in globals():
    raise NameError("recon_payload_df does not exist. Run Step 6 first.")

if "RECON_ADJUDICATION_INSTRUCTION" not in globals():
    raise NameError("RECON_ADJUDICATION_INSTRUCTION does not exist. Run Step 3 first.")

if "RECON_DRY_RUN" not in globals():
    raise NameError("RECON_DRY_RUN does not exist. Run Step 3 first.")

if "config" not in globals():
    raise NameError("config does not exist. Run Step 3 first.")

if "recon_payload" not in recon_payload_df.columns:
    raise NameError("recon_payload column does not exist. Run Step 6 first.")


# ----------------------------
# Helper: get config value using several possible keys
# ----------------------------
def get_config_value_anywhere(config_obj, section_key_pairs, fallback=None):
    """
    Try multiple section/key pairs and return the first value found.
    """
    for section, key in section_key_pairs:
        if config_obj.has_section(section) and config_obj.has_option(section, key):
            value = config_obj.get(section, key, fallback="").strip()
            if value:
                return value

    return fallback


# ----------------------------
# Resolve recon model from config
# ----------------------------
RECON_MODEL = get_config_value_anywhere(
    config,
    section_key_pairs=[
        ("ReconSettings", "model"),
        ("OpenAI", "model"),
        ("OpenAI", "MODEL"),
        ("Model", "model"),
        ("Models", "recon_model"),
        ("Models", "model"),
    ],
    fallback="gpt-4.1-mini"
)

print(f"✅ Recon model: {RECON_MODEL}")
print(f"✅ Recon dry run: {RECON_DRY_RUN}")
print(f"✅ Rows available for recon API call: {len(recon_payload_df)}")


# ----------------------------
# Initialize OpenAI client
# ----------------------------
# Reuse an existing client if your notebook already created one.
if "openai_client" in globals():
    recon_client = openai_client
elif "client" in globals():
    recon_client = client
else:
    recon_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# ----------------------------
# JSON schema for recon response
# ----------------------------
recon_json_schema = {
    "name": "recon_adjudication_result",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "best_best_match_cde": {
                "type": "string",
                "description": "The selected official HEAL CDE name, or blank if no candidate is accepted."
            },
            "best_best_match_crf": {
                "type": "string",
                "description": "The selected official CRF/form name when available, or the best supported CRF context."
            },
            "best_best_match_variable": {
                "type": "string",
                "description": "The selected official HEAL CDE variable name, or blank if no candidate is accepted."
            },
            "recon_decision": {
                "type": "string",
                "enum": [
                    "Accept Candidate",
                    "No Match",
                    "Needs Human Review"
                ]
            },
            "recon_confidence": {
                "type": "string",
                "enum": [
                    "High",
                    "Medium",
                    "Low"
                ]
            },
            "recon_rationale": {
                "type": "string",
                "description": "One brief sentence explaining the decision."
            }
        },
        "required": [
            "best_best_match_cde",
            "best_best_match_crf",
            "best_best_match_variable",
            "recon_decision",
            "recon_confidence",
            "recon_rationale"
        ]
    }
}


# ----------------------------
# Helper: extract response text
# ----------------------------
def extract_response_text(response):
    """
    Extract text from a Responses API response.
    Handles common SDK response shapes.
    """
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text

    # Fallback for nested output structures
    try:
        chunks = []

        for item in response.output:
            if hasattr(item, "content"):
                for content_item in item.content:
                    if hasattr(content_item, "text"):
                        chunks.append(content_item.text)

        return "\n".join(chunks).strip()

    except Exception:
        return str(response)


# ----------------------------
# Helper: parse recon JSON
# ----------------------------
def parse_recon_json(raw_text):
    """
    Parse model response into a normalized result dictionary.
    """
    try:
        parsed = json.loads(raw_text)

        return {
            "recon_parsed_successfully": True,
            "best_best_match_cde": parsed.get("best_best_match_cde", ""),
            "best_best_match_crf": parsed.get("best_best_match_crf", ""),
            "best_best_match_variable": parsed.get("best_best_match_variable", ""),
            "recon_decision": parsed.get("recon_decision", ""),
            "recon_confidence": parsed.get("recon_confidence", ""),
            "recon_rationale": parsed.get("recon_rationale", ""),
            "recon_raw_response": raw_text,
            "recon_error": ""
        }

    except Exception as e:
        return {
            "recon_parsed_successfully": False,
            "best_best_match_cde": "",
            "best_best_match_crf": "",
            "best_best_match_variable": "",
            "recon_decision": "Needs Human Review",
            "recon_confidence": "Low",
            "recon_rationale": "Model response could not be parsed as valid JSON.",
            "recon_raw_response": raw_text,
            "recon_error": str(e)
        }


# ----------------------------
# Helper: call recon model for one payload
# ----------------------------
def call_recon_model(payload, row_label="", max_retries=2, sleep_seconds=2):
    """
    Send one payload to the model and return parsed recon result.
    """
    user_payload_text = json.dumps(payload, indent=2, ensure_ascii=False)

    # Dry run lets you test wiring without spending API calls.
    if RECON_DRY_RUN:
        return {
            "recon_parsed_successfully": True,
            "best_best_match_cde": "",
            "best_best_match_crf": "",
            "best_best_match_variable": "",
            "recon_decision": "Needs Human Review",
            "recon_confidence": "Low",
            "recon_rationale": "Dry run only; no model call was made.",
            "recon_raw_response": "",
            "recon_error": ""
        }

    last_error = ""

    for attempt in range(1, max_retries + 2):
        try:
            response = recon_client.responses.create(
                model=RECON_MODEL,
                input=[
                    {
                        "role": "system",
                        "content": RECON_ADJUDICATION_INSTRUCTION
                    },
                    {
                        "role": "user",
                        "content": (
                            "Review the following reconciliation payload and return only valid JSON.\n\n"
                            f"{user_payload_text}"
                        )
                    }
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        "name": recon_json_schema["name"],
                        "strict": recon_json_schema["strict"],
                        "schema": recon_json_schema["schema"]
                    }
                }
            )

            raw_text = extract_response_text(response)
            parsed_result = parse_recon_json(raw_text)

            return parsed_result

        except Exception as e:
            last_error = str(e)
            print(f"⚠️ Recon call failed for {row_label} on attempt {attempt}: {last_error}")

            if attempt <= max_retries:
                time.sleep(sleep_seconds)

    return {
        "recon_parsed_successfully": False,
        "best_best_match_cde": "",
        "best_best_match_crf": "",
        "best_best_match_variable": "",
        "recon_decision": "Needs Human Review",
        "recon_confidence": "Low",
        "recon_rationale": "Recon API call failed after retries.",
        "recon_raw_response": "",
        "recon_error": last_error
    }


# ----------------------------
# Run recon calls
# ----------------------------
recon_results = []

total_rows = len(recon_payload_df)

print("\n🚀 Starting recon adjudication calls...")

for counter, (idx, row) in enumerate(recon_payload_df.iterrows(), start=1):
    source_index = row.get("recon_source_index", idx)
    row_label = f"row {counter}/{total_rows}, source_index={source_index}"

    print(f"Processing {row_label}...")

    result = call_recon_model(
        payload=row["recon_payload"],
        row_label=row_label
    )

    result["recon_source_index"] = source_index
    result["recon_payload_df_index"] = idx

    recon_results.append(result)


# ----------------------------
# Convert results to dataframe
# ----------------------------
recon_results_df = pd.DataFrame(recon_results)

print("\n✅ Recon adjudication complete")
print(f"Rows processed: {len(recon_results_df)}")

if len(recon_results_df) > 0:
    print("\n📊 Recon decision counts:")
    display(
        recon_results_df["recon_decision"]
        .fillna("")
        .replace("", "(blank)")
        .value_counts()
        .rename_axis("recon_decision")
        .reset_index(name="row_count")
    )

    print("\n📊 Recon confidence counts:")
    display(
        recon_results_df["recon_confidence"]
        .fillna("")
        .replace("", "(blank)")
        .value_counts()
        .rename_axis("recon_confidence")
        .reset_index(name="row_count")
    )


# ----------------------------
# Preview recon results
# ----------------------------
display_cols = [
    "recon_source_index",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_parsed_successfully",
    "recon_error",
]

display_cols = [
    col for col in display_cols
    if col in recon_results_df.columns
]

display(recon_results_df[display_cols])

✅ Recon model: gpt-4.1-mini
✅ Recon dry run: False
✅ Rows available for recon API call: 12

🚀 Starting recon adjudication calls...
Processing row 1/12, source_index=2...
Processing row 2/12, source_index=3...
Processing row 3/12, source_index=4...
Processing row 4/12, source_index=5...
Processing row 5/12, source_index=6...
Processing row 6/12, source_index=7...
Processing row 7/12, source_index=8...
Processing row 8/12, source_index=9...
Processing row 9/12, source_index=10...
Processing row 10/12, source_index=11...
Processing row 11/12, source_index=12...
Processing row 12/12, source_index=13...

✅ Recon adjudication complete
Rows processed: 12

📊 Recon decision counts:


,recon_decision,row_count
0,Accept Candidate,11
1,No Match,1



📊 Recon confidence counts:


,recon_confidence,row_count
0,High,12


,recon_source_index,best_best_match_cde,best_best_match_crf,best_best_match_variable,recon_decision,recon_confidence,recon_rationale,recon_parsed_successfully,recon_error
0,2,,,,No Match,High,None of the candidates correspond to birth wei...,True,
1,3,Brief Pain Inventory (BPI) - average pain rati...,BPI Pain Severity,BPIAvgPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,True,
2,4,Brief Pain Inventory (BPI) - current pain rati...,BPI Pain Severity,BPICurrentPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,True,
3,5,Brief Pain Inventory (BPI) - least pain rating...,BPI Pain Severity,BPILeastPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,True,
4,6,Brief Pain Inventory (BPI) - worst pain rating...,BPI Pain Severity,BPIWorstPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,True,
5,7,Generalized Anxiety Disorder (GAD-2) - feeling...,GAD2 Pain,GAD2FeelNervScale,Accept Candidate,High,The candidate exactly matches the study variab...,True,
6,8,Generalized Anxiety Disorder (GAD-7) - not sto...,GAD7,GAD2NotStopWryScl,Accept Candidate,High,The top-ranked candidate exactly matches the s...,True,
7,9,Generalized Anxiety Disorder (GAD-7) - easily ...,GAD7,GAD7EasyAnnoyedScl,Accept Candidate,High,The candidate perfectly matches the study vari...,True,
8,10,Generalized Anxiety Disorder (GAD-7) - feeling...,GAD7,GAD7FeelAfrdScl,Accept Candidate,High,The top-ranked candidate perfectly aligns with...,True,
9,11,Generalized Anxiety Disorder (GAD-7) - restles...,GAD7,GAD7RstlessScl,Accept Candidate,High,The best match candidate perfectly aligns with...,True,


In [151]:
# ============================================================
# STEP 8a: Build unified recon results table
# ============================================================
# Goal:
# - Combine LLM adjudication results with protected-family preassigned results
# - Keep one recon result row per recon_source_index
# - Do NOT merge back to the full dataframe yet
# - Do NOT export yet

import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "recon_candidate_df" not in globals():
    raise NameError("recon_candidate_df does not exist. Run Step 4 first.")

if "recon_results_df" not in globals():
    raise NameError("recon_results_df does not exist. Run Step 7 first.")

if "recon_source_index" not in recon_candidate_df.columns:
    raise NameError("recon_source_index does not exist in recon_candidate_df. Run Step 4 first.")


# ----------------------------
# Helper: safe text
# ----------------------------
def merge_text(value):
    """
    Convert None/NaN to blank strings for clean output columns.
    """
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value).strip()


# ----------------------------
# LLM recon results
# ----------------------------
llm_recon_results_df = recon_results_df.copy()

llm_recon_results_df["recon_result_source"] = "LLM Recon"
llm_recon_results_df["recon_was_skipped"] = False

# Make sure expected columns exist
expected_result_cols = [
    "recon_source_index",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_parsed_successfully",
    "recon_raw_response",
    "recon_error",
    "recon_result_source",
    "recon_was_skipped",
]

for col in expected_result_cols:
    if col not in llm_recon_results_df.columns:
        llm_recon_results_df[col] = ""


# ----------------------------
# Protected-family skipped / preassigned results
# ----------------------------
skipped_recon_rows_df = recon_candidate_df[
    recon_candidate_df.get("recon_skip_llm", False) == True
].copy()

preassigned_results = []

for _, row in skipped_recon_rows_df.iterrows():
    preassigned_results.append({
        "recon_source_index": row.get("recon_source_index"),
        "best_best_match_cde": merge_text(row.get("recon_preassigned_cde")),
        "best_best_match_crf": merge_text(row.get("recon_preassigned_crf")),
        "best_best_match_variable": merge_text(row.get("recon_preassigned_variable")),
        "recon_decision": merge_text(row.get("recon_preassigned_decision")) or "Needs Human Review",
        "recon_confidence": merge_text(row.get("recon_preassigned_confidence")) or "Low",
        "recon_rationale": merge_text(row.get("recon_preassigned_rationale")),
        "recon_parsed_successfully": True,
        "recon_raw_response": "",
        "recon_error": "",
        "recon_result_source": "Protected Family Guardrail",
        "recon_was_skipped": True,
        "recon_review_flag": merge_text(row.get("recon_review_flag")),
        "recon_protected_family_anchor": merge_text(row.get("recon_protected_family_anchor")),
    })

preassigned_recon_results_df = pd.DataFrame(preassigned_results)

# Ensure columns exist even if no rows were skipped
for col in expected_result_cols + ["recon_review_flag", "recon_protected_family_anchor"]:
    if col not in preassigned_recon_results_df.columns:
        preassigned_recon_results_df[col] = ""


# ----------------------------
# Combine LLM + preassigned results
# ----------------------------
unified_recon_results_df = pd.concat(
    [
        llm_recon_results_df,
        preassigned_recon_results_df
    ],
    ignore_index=True
)


# ----------------------------
# Clean and deduplicate
# ----------------------------
# recon_source_index should map back to the original row index from recon_input_df.
unified_recon_results_df["recon_source_index"] = unified_recon_results_df[
    "recon_source_index"
].astype(int)

# If a duplicate ever appears, prefer LLM result over guardrail only if both exist.
# Normally this should not happen.
source_priority = {
    "LLM Recon": 1,
    "Protected Family Guardrail": 2,
}

unified_recon_results_df["recon_result_priority"] = unified_recon_results_df[
    "recon_result_source"
].map(source_priority).fillna(99)

unified_recon_results_df = (
    unified_recon_results_df
    .sort_values(["recon_source_index", "recon_result_priority"])
    .drop_duplicates(subset=["recon_source_index"], keep="first")
    .sort_values("recon_source_index")
    .reset_index(drop=True)
)


# ----------------------------
# Print summary
# ----------------------------
print("✅ Built unified_recon_results_df")
print(f"LLM recon rows: {len(llm_recon_results_df)}")
print(f"Protected-family preassigned rows: {len(preassigned_recon_results_df)}")
print(f"Unified recon result rows: {len(unified_recon_results_df)}")

print("\n📊 Recon result source counts:")
display(
    unified_recon_results_df["recon_result_source"]
    .fillna("")
    .replace("", "(blank)")
    .value_counts()
    .rename_axis("recon_result_source")
    .reset_index(name="row_count")
)

print("\n📊 Recon decision counts:")
display(
    unified_recon_results_df["recon_decision"]
    .fillna("")
    .replace("", "(blank)")
    .value_counts()
    .rename_axis("recon_decision")
    .reset_index(name="row_count")
)


# ----------------------------
# Preview unified results
# ----------------------------
preview_cols = [
    "recon_source_index",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_was_skipped",
    "recon_review_flag",
    "recon_protected_family_anchor",
    "recon_parsed_successfully",
    "recon_error",
]

preview_cols = [
    col for col in preview_cols
    if col in unified_recon_results_df.columns
]

display(unified_recon_results_df[preview_cols])

✅ Built unified_recon_results_df
LLM recon rows: 12
Protected-family preassigned rows: 0
Unified recon result rows: 12

📊 Recon result source counts:


,recon_result_source,row_count
0,LLM Recon,12



📊 Recon decision counts:


,recon_decision,row_count
0,Accept Candidate,11
1,No Match,1


,recon_source_index,best_best_match_cde,best_best_match_crf,best_best_match_variable,recon_decision,recon_confidence,recon_rationale,recon_result_source,recon_was_skipped,recon_review_flag,recon_protected_family_anchor,recon_parsed_successfully,recon_error
0,2,,,,No Match,High,None of the candidates correspond to birth wei...,LLM Recon,False,NaN,NaN,True,
1,3,Brief Pain Inventory (BPI) - average pain rati...,BPI Pain Severity,BPIAvgPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,False,NaN,NaN,True,
2,4,Brief Pain Inventory (BPI) - current pain rati...,BPI Pain Severity,BPICurrentPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,False,NaN,NaN,True,
3,5,Brief Pain Inventory (BPI) - least pain rating...,BPI Pain Severity,BPILeastPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,False,NaN,NaN,True,
4,6,Brief Pain Inventory (BPI) - worst pain rating...,BPI Pain Severity,BPIWorstPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,False,NaN,NaN,True,
5,7,Generalized Anxiety Disorder (GAD-2) - feeling...,GAD2 Pain,GAD2FeelNervScale,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,False,NaN,NaN,True,
6,8,Generalized Anxiety Disorder (GAD-7) - not sto...,GAD7,GAD2NotStopWryScl,Accept Candidate,High,The top-ranked candidate exactly matches the s...,LLM Recon,False,NaN,NaN,True,
7,9,Generalized Anxiety Disorder (GAD-7) - easily ...,GAD7,GAD7EasyAnnoyedScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,False,NaN,NaN,True,
8,10,Generalized Anxiety Disorder (GAD-7) - feeling...,GAD7,GAD7FeelAfrdScl,Accept Candidate,High,The top-ranked candidate perfectly aligns with...,LLM Recon,False,NaN,NaN,True,
9,11,Generalized Anxiety Disorder (GAD-7) - restles...,GAD7,GAD7RstlessScl,Accept Candidate,High,The best match candidate perfectly aligns with...,LLM Recon,False,NaN,NaN,True,


In [152]:
# ============================================================
# STEP 8b: Merge unified recon results back onto recon_candidate_df
# ============================================================
# Goal:
# - Attach unified recon outcomes to the 10-row recon test batch
# - Preserve original matching context and protected-family guardrail columns
# - Create recon_candidate_with_results_df for inspection
# - Do NOT merge to full dataframe yet
# - Do NOT export yet

import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "recon_candidate_df" not in globals():
    raise NameError("recon_candidate_df does not exist. Run Step 4 first.")

if "unified_recon_results_df" not in globals():
    raise NameError("unified_recon_results_df does not exist. Run Step 8a first.")

if "recon_source_index" not in recon_candidate_df.columns:
    raise NameError("recon_source_index does not exist in recon_candidate_df. Run Step 4 first.")

if "recon_source_index" not in unified_recon_results_df.columns:
    raise NameError("recon_source_index does not exist in unified_recon_results_df. Run Step 8a first.")


# ----------------------------
# Prep merge keys
# ----------------------------
recon_candidate_merge_df = recon_candidate_df.copy()
recon_results_merge_df = unified_recon_results_df.copy()

recon_candidate_merge_df["recon_source_index"] = recon_candidate_merge_df["recon_source_index"].astype(int)
recon_results_merge_df["recon_source_index"] = recon_results_merge_df["recon_source_index"].astype(int)


# ----------------------------
# Avoid duplicate/redundant columns from result table
# ----------------------------
# Keep the result columns we want to attach.
recon_result_cols_to_merge = [
    "recon_source_index",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_was_skipped",
    "recon_review_flag",
    "recon_protected_family_anchor",
    "recon_parsed_successfully",
    "recon_raw_response",
    "recon_error",
]

recon_result_cols_to_merge = [
    col for col in recon_result_cols_to_merge
    if col in recon_results_merge_df.columns
]

recon_results_merge_df = recon_results_merge_df[recon_result_cols_to_merge].copy()


# ----------------------------
# Merge onto recon candidate batch
# ----------------------------
recon_candidate_with_results_df = recon_candidate_merge_df.merge(
    recon_results_merge_df,
    on="recon_source_index",
    how="left",
    suffixes=("", "_result")
)


# ----------------------------
# Validate merge
# ----------------------------
rows_before = len(recon_candidate_df)
rows_after = len(recon_candidate_with_results_df)

if rows_before != rows_after:
    raise ValueError(
        f"Row count changed after merge. Before: {rows_before}, After: {rows_after}"
    )

missing_result_count = recon_candidate_with_results_df["recon_decision"].isna().sum()

print("✅ Built recon_candidate_with_results_df")
print(f"Rows before merge: {rows_before}")
print(f"Rows after merge: {rows_after}")
print(f"Rows missing recon decision: {missing_result_count}")


# ----------------------------
# Summary counts
# ----------------------------
print("\n📊 Recon result source counts:")
display(
    recon_candidate_with_results_df["recon_result_source"]
    .fillna("")
    .replace("", "(blank)")
    .value_counts()
    .rename_axis("recon_result_source")
    .reset_index(name="row_count")
)

print("\n📊 Recon decision counts:")
display(
    recon_candidate_with_results_df["recon_decision"]
    .fillna("")
    .replace("", "(blank)")
    .value_counts()
    .rename_axis("recon_decision")
    .reset_index(name="row_count")
)


# ----------------------------
# Preview merged review table
# ----------------------------
review_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("form_name"),
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_status"),
    resolved_cols.get("final_concept_match"),
    resolved_cols.get("best_match_score"),
    resolved_cols.get("encoding_fidelity_score"),
    resolved_cols.get("potential_match_2"),
    resolved_cols.get("potential_match_3"),
    "recon_protected_family_hit",
    "recon_skip_llm",
    "recon_review_flag",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_parsed_successfully",
    "recon_error",
]

review_cols = [
    col for col in review_cols
    if col is not None and col in recon_candidate_with_results_df.columns
]

display(recon_candidate_with_results_df[review_cols])

✅ Built recon_candidate_with_results_df
Rows before merge: 12
Rows after merge: 12
Rows missing recon decision: 0

📊 Recon result source counts:


,recon_result_source,row_count
0,LLM Recon,12



📊 Recon decision counts:


,recon_decision,row_count
0,Accept Candidate,11
1,No Match,1


,recon_source_index,name,HEAL Core CRF Match,Final Concept Match Status,Final HEAL CDE Concept Match,Final Concept Match Score,Final Encoding Fidelity Score,Potential Match 2 - CDE Name,Potential Match 3 - CDE Name,recon_protected_family_hit,...,recon_review_flag,best_best_match_cde,best_best_match_crf,best_best_match_variable,recon_decision,recon_confidence,recon_rationale,recon_result_source,recon_parsed_successfully,recon_error
0,2,imhbirthwt,Demographics,Possible concept match,BRTHDTC,66.0,0,Sex,MARISTAT,False,...,,,,,No Match,High,None of the candidates correspond to birth wei...,LLM Recon,True,
1,3,BPIAvgPainRtngScl,BPI Pain Severity,High concept match,BPIAvgPainRatingScl,99.3,100,BPIAvgPain7dRtngScale,BPILeastPainRatingScl,True,...,Protected Family - In-Family Candidate Available,Brief Pain Inventory (BPI) - average pain rati...,BPI Pain Severity,BPIAvgPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,True,
2,4,BPICurrentPainRtngScl,BPI Pain Severity,Possible concept match,BPICurrentPainRatingScl,74.5,100,BPICurntPainRtngScale,BPILeastPainRatingScl,True,...,Protected Family - In-Family Candidate Available,Brief Pain Inventory (BPI) - current pain rati...,BPI Pain Severity,BPICurrentPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,True,
3,5,BPILeastPnLst24HRtngScl,BPI Pain Severity,High concept match,BPILeastPainRatingScl,95.0,100,BPIWorstPainRatingScl,BPIAvgPainRatingScl,True,...,Protected Family - In-Family Candidate Available,Brief Pain Inventory (BPI) - least pain rating...,BPI Pain Severity,BPILeastPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,True,
4,6,BPIWrstPnLast24HRtngScl,BPI Pain Severity,High concept match,BPIWorstPainRatingScl,94.0,100,BPILeastPainRatingScl,BPIWrstPain7dRtngScale,True,...,Protected Family - In-Family Candidate Available,Brief Pain Inventory (BPI) - worst pain rating...,BPI Pain Severity,BPIWorstPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,True,
5,7,GAD2FeelNervScl,GAD2 Pain (Generalized Anxiety Disorder),High concept match,GAD2FeelNervScale,98.4,100,GAD2FeelNervScl,GAD7FeelAfrdScl,False,...,,Generalized Anxiety Disorder (GAD-2) - feeling...,GAD2 Pain,GAD2FeelNervScale,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,True,
6,8,GAD2NotStopWryScl,No CRF match,Possible concept match,GAD2NotStopWryScl,81.0,100,GAD2NotStopWryScale,GAD7WryTooMchScl,False,...,,Generalized Anxiety Disorder (GAD-7) - not sto...,GAD7,GAD2NotStopWryScl,Accept Candidate,High,The top-ranked candidate exactly matches the s...,LLM Recon,True,
7,9,GAD7EasyAnnoyedScl,GAD7,High concept match,GAD7EasyAnnoyedScl,100,100,GAD7TroubRelxScl,GAD7TotScore,False,...,,Generalized Anxiety Disorder (GAD-7) - easily ...,GAD7,GAD7EasyAnnoyedScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,True,
8,10,GAD7FeelAfrdScl,GAD7,High concept match,GAD7FeelAfrdScl,100,100,GAD2FeelNervScl,GAD7TotScore,False,...,,Generalized Anxiety Disorder (GAD-7) - feeling...,GAD7,GAD7FeelAfrdScl,Accept Candidate,High,The top-ranked candidate perfectly aligns with...,LLM Recon,True,
9,11,GAD7RstlessScl,GAD7,High concept match,GAD7RstlessScl,100,100,GAD2NotStopWryScl,GAD7TotScore,False,...,,Generalized Anxiety Disorder (GAD-7) - restles...,GAD7,GAD7RstlessScl,Accept Candidate,High,The best match candidate perfectly aligns with...,LLM Recon,True,


In [153]:
# ============================================================
# STEP 9a: Merge unified recon results back onto the full dataframe
# ============================================================
# Goal:
# - Attach recon results to the full VLMD output dataframe
# - Preserve all rows, not only the 10-row recon test batch
# - Create final_reconciled_df as the full dataframe for Excel export prep
# - Do NOT export yet

import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "recon_input_df" not in globals():
    raise NameError("recon_input_df does not exist. Run Step 2 first.")

if "unified_recon_results_df" not in globals():
    raise NameError("unified_recon_results_df does not exist. Run Step 8a first.")

if "recon_candidate_with_results_df" not in globals():
    raise NameError("recon_candidate_with_results_df does not exist. Run Step 8b first.")


# ----------------------------
# Prepare full dataframe
# ----------------------------
final_reconciled_df = recon_input_df.copy()

# Create the same source index used during recon
final_reconciled_df["recon_source_index"] = final_reconciled_df.index.astype(int)


# ----------------------------
# Prepare recon results for merge
# ----------------------------
recon_results_for_full_merge_df = unified_recon_results_df.copy()
recon_results_for_full_merge_df["recon_source_index"] = recon_results_for_full_merge_df[
    "recon_source_index"
].astype(int)


# ----------------------------
# Choose recon columns to merge onto full dataframe
# ----------------------------
recon_cols_to_merge = [
    "recon_source_index",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_was_skipped",
    "recon_review_flag",
    "recon_protected_family_anchor",
    "recon_parsed_successfully",
    "recon_raw_response",
    "recon_error",
]

recon_cols_to_merge = [
    col for col in recon_cols_to_merge
    if col in recon_results_for_full_merge_df.columns
]

recon_results_for_full_merge_df = recon_results_for_full_merge_df[
    recon_cols_to_merge
].copy()


# ----------------------------
# Merge recon results onto full dataframe
# ----------------------------
rows_before = len(final_reconciled_df)

final_reconciled_df = final_reconciled_df.merge(
    recon_results_for_full_merge_df,
    on="recon_source_index",
    how="left"
)

rows_after = len(final_reconciled_df)

if rows_before != rows_after:
    raise ValueError(
        f"Row count changed after full merge. Before: {rows_before}, After: {rows_after}"
    )


# ----------------------------
# Fill blank recon fields for rows not processed during test batch
# ----------------------------
recon_output_cols = [
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_review_flag",
    "recon_protected_family_anchor",
    "recon_raw_response",
    "recon_error",
]

for col in recon_output_cols:
    if col in final_reconciled_df.columns:
        final_reconciled_df[col] = final_reconciled_df[col].fillna("")

boolean_recon_cols = [
    "recon_was_skipped",
    "recon_parsed_successfully",
]

for col in boolean_recon_cols:
    if col in final_reconciled_df.columns:
        final_reconciled_df[col] = final_reconciled_df[col].fillna(False)


# ----------------------------
# Add helper flag showing whether row went through recon during this run
# ----------------------------
final_reconciled_df["recon_processed_this_run"] = final_reconciled_df[
    "recon_decision"
].apply(
    lambda value: bool(str(value).strip())
)


# ----------------------------
# Print summary
# ----------------------------
print("✅ Built final_reconciled_df")
print(f"Rows in recon_input_df: {len(recon_input_df)}")
print(f"Rows in final_reconciled_df: {len(final_reconciled_df)}")
print(f"Rows with recon results this run: {final_reconciled_df['recon_processed_this_run'].sum()}")

print("\n📊 Recon processed flag counts:")
display(
    final_reconciled_df["recon_processed_this_run"]
    .value_counts()
    .rename_axis("recon_processed_this_run")
    .reset_index(name="row_count")
)

print("\n📊 Recon decision counts:")
display(
    final_reconciled_df["recon_decision"]
    .fillna("")
    .replace("", "(not processed)")
    .value_counts()
    .rename_axis("recon_decision")
    .reset_index(name="row_count")
)


# ----------------------------
# Preview merged full dataframe rows that were processed
# ----------------------------
preview_cols = [
    "recon_source_index",
    resolved_cols.get("variable_name"),
    resolved_cols.get("form_name"),
    resolved_cols.get("prestep_crf_match"),
    resolved_cols.get("final_concept_match"),
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_processed_this_run",
]

preview_cols = [
    col for col in preview_cols
    if col is not None and col in final_reconciled_df.columns
]

display(
    final_reconciled_df.loc[
        final_reconciled_df["recon_processed_this_run"] == True,
        preview_cols
    ].head(20)
)

✅ Built final_reconciled_df
Rows in recon_input_df: 14
Rows in final_reconciled_df: 14
Rows with recon results this run: 12

📊 Recon processed flag counts:


C:\Users\lmaefos\AppData\Local\Temp\ipykernel_2400\2242305950.py:121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_reconciled_df[col] = final_reconciled_df[col].fillna(False)


,recon_processed_this_run,row_count
0,True,12
1,False,2



📊 Recon decision counts:


,recon_decision,row_count
0,Accept Candidate,11
1,(not processed),2
2,No Match,1


,recon_source_index,name,HEAL Core CRF Match,Final HEAL CDE Concept Match,best_best_match_cde,best_best_match_crf,best_best_match_variable,recon_decision,recon_confidence,recon_rationale,recon_result_source,recon_processed_this_run
2,2,imhbirthwt,Demographics,BRTHDTC,,,,No Match,High,None of the candidates correspond to birth wei...,LLM Recon,True
3,3,BPIAvgPainRtngScl,BPI Pain Severity,BPIAvgPainRatingScl,Brief Pain Inventory (BPI) - average pain rati...,BPI Pain Severity,BPIAvgPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,True
4,4,BPICurrentPainRtngScl,BPI Pain Severity,BPICurrentPainRatingScl,Brief Pain Inventory (BPI) - current pain rati...,BPI Pain Severity,BPICurrentPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,True
5,5,BPILeastPnLst24HRtngScl,BPI Pain Severity,BPILeastPainRatingScl,Brief Pain Inventory (BPI) - least pain rating...,BPI Pain Severity,BPILeastPainRatingScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,True
6,6,BPIWrstPnLast24HRtngScl,BPI Pain Severity,BPIWorstPainRatingScl,Brief Pain Inventory (BPI) - worst pain rating...,BPI Pain Severity,BPIWorstPainRatingScl,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,True
7,7,GAD2FeelNervScl,GAD2 Pain (Generalized Anxiety Disorder),GAD2FeelNervScale,Generalized Anxiety Disorder (GAD-2) - feeling...,GAD2 Pain,GAD2FeelNervScale,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon,True
8,8,GAD2NotStopWryScl,No CRF match,GAD2NotStopWryScl,Generalized Anxiety Disorder (GAD-7) - not sto...,GAD7,GAD2NotStopWryScl,Accept Candidate,High,The top-ranked candidate exactly matches the s...,LLM Recon,True
9,9,GAD7EasyAnnoyedScl,GAD7,GAD7EasyAnnoyedScl,Generalized Anxiety Disorder (GAD-7) - easily ...,GAD7,GAD7EasyAnnoyedScl,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon,True
10,10,GAD7FeelAfrdScl,GAD7,GAD7FeelAfrdScl,Generalized Anxiety Disorder (GAD-7) - feeling...,GAD7,GAD7FeelAfrdScl,Accept Candidate,High,The top-ranked candidate perfectly aligns with...,LLM Recon,True
11,11,GAD7RstlessScl,GAD7,GAD7RstlessScl,Generalized Anxiety Disorder (GAD-7) - restles...,GAD7,GAD7RstlessScl,Accept Candidate,High,The best match candidate perfectly aligns with...,LLM Recon,True


In [154]:
# ============================================================
# STEP 9b: Build final Excel sheet dataframes
# ============================================================
# Goal:
# - Create 3 export-ready dataframes:
#   1. metadata_sheet_df
#   2. final_mapping_sheet_df
#   3. full_outputs_sheet_df
# - Use original input column names from config
# - Do NOT export yet

import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
if "final_reconciled_df" not in globals():
    raise NameError("final_reconciled_df does not exist. Run Step 9a first.")

if "config" not in globals():
    raise NameError("config does not exist. Run Step 3 first.")


# ============================================================
# 1. Pull original input column names from config
# ============================================================

variable_col = config.get("Columns", "variable_column", fallback="").strip()
form_col = config.get("Columns", "crf_column", fallback="").strip()
description_col = config.get("Columns", "description_column", fallback="").strip()
encoding_col = config.get("Columns", "ENCODING_COLUMN", fallback="").strip()

print("🧭 Original columns from config:")
print(f"  variable_col: {variable_col}")
print(f"  form_col: {form_col}")
print(f"  description_col: {description_col}")
print(f"  encoding_col: {encoding_col}")


# ----------------------------
# Validate required original columns
# ----------------------------
if not variable_col or variable_col not in final_reconciled_df.columns:
    raise KeyError(
        f"Config variable_column='{variable_col}' was not found in final_reconciled_df."
    )

if not form_col or form_col not in final_reconciled_df.columns:
    raise KeyError(
        f"Config crf_column='{form_col}' was not found in final_reconciled_df."
    )


# ============================================================
# 2. Identify all original input columns
# ============================================================
# Since your config tells us the main input columns, we start there.
# Then we include any columns that existed before pipeline/recon outputs.
# This can be adjusted manually later if needed.

configured_original_cols = [
    form_col,
    variable_col,
    description_col,
    encoding_col,
]

configured_original_cols = [
    col for col in configured_original_cols
    if col and col in final_reconciled_df.columns
]

# Known pipeline/recon columns to exclude from "original input columns"
known_output_prefixes = (
    "recon_",
)

known_output_exact = {
    "HEAL Core CRF Match",
    "Prestep CRF Confidence",
    "Confidence Level",
    "Rationale",
    "Full Response",
    "Refined CRF Name",
    "Canonical CRF Name",
    "Best Match CDE Name",
    "Best Match CRF Name",
    "Best Match Score",
    "Potential Match 2 - CDE Name",
    "Potential Match 2 - CRF Name",
    "Potential Match 3 - CDE Name",
    "Potential Match 3 - CRF Name",
    "Final HEAL CDE Concept Match",
    "Final Concept Match Status",
    "Final Concept Match Score",
    "Final Concept Match CRF",
    "Final Encoding Fidelity Score",
    "Encoding Fidelity Score",
    "Concept Match Score",
    "Concept Match Status",
    "Closest HEAL CDE Concept",
    "best_best_match_cde",
    "best_best_match_crf",
    "best_best_match_variable",
    "recon_decision",
    "recon_confidence",
    "recon_rationale",
    "recon_result_source",
    "recon_was_skipped",
    "recon_processed_this_run",
    "recon_source_index",
    "recon_raw_response",
    "recon_error",
    "recon_parsed_successfully",
}

# Keep all columns that appear to be original input columns.
# In your test file, this should include section/name/description/enumLabels
# plus any other original DD columns if present.
ORIGINAL_INPUT_COLUMNS = []

for col in final_reconciled_df.columns:
    if col.startswith(known_output_prefixes):
        continue

    if col in known_output_exact:
        continue

    # Keep configured columns
    if col in configured_original_cols:
        ORIGINAL_INPUT_COLUMNS.append(col)
        continue

    # Keep columns that appear before major pipeline outputs,
    # unless they are known generated columns.
    # This is intentionally conservative.
    if col not in known_output_exact:
        # Avoid adding many intermediate matching columns here unless needed.
        # If your original input has more columns, they will usually be retained
        # because they are not in known_output_exact.
        if col in final_reconciled_df.columns:
            ORIGINAL_INPUT_COLUMNS.append(col)

# Deduplicate while preserving order
ORIGINAL_INPUT_COLUMNS = list(dict.fromkeys(ORIGINAL_INPUT_COLUMNS))

print("\n🧭 Original input columns selected:")
for col in ORIGINAL_INPUT_COLUMNS:
    print(f"  - {col}")


# ============================================================
# 3. Metadata sheet
# ============================================================
# Requested columns:
# - original variable name from config variable_column
# - original form name from config crf_column
# - corresponding best_best_match_crf
# - corresponding best_best_match_cde
# - corresponding recon_decision

metadata_sheet_df = pd.DataFrame({
    "Original Variable Name": final_reconciled_df[variable_col],
    "Original Form Name": final_reconciled_df[form_col],
    "Best Best Match CRF": final_reconciled_df.get("best_best_match_crf", ""),
    "Best Best Match CDE": final_reconciled_df.get("best_best_match_cde", ""),
    "Recon Decision": final_reconciled_df.get("recon_decision", ""),
})


# ============================================================
# 4. Final mapping sheet
# ============================================================
# Requested columns:
# - all original columns from input
# - best_best_match_cde renamed to "Best Match CDE Name"
# - best_best_match_crf renamed to "Best Match CRF Name"
# - recon_decision renamed to "Final Decision"
# - recon_confidence
# - recon_rationale
# - recon_result_source

final_mapping_sheet_df = final_reconciled_df[ORIGINAL_INPUT_COLUMNS].copy()

final_mapping_sheet_df["Best Match CDE Name"] = final_reconciled_df.get("best_best_match_cde", "")
final_mapping_sheet_df["Best Match CRF Name"] = final_reconciled_df.get("best_best_match_crf", "")
final_mapping_sheet_df["Final Decision"] = final_reconciled_df.get("recon_decision", "")
final_mapping_sheet_df["recon_confidence"] = final_reconciled_df.get("recon_confidence", "")
final_mapping_sheet_df["recon_rationale"] = final_reconciled_df.get("recon_rationale", "")
final_mapping_sheet_df["recon_result_source"] = final_reconciled_df.get("recon_result_source", "")


# ============================================================
# 5. Full outputs sheet
# ============================================================
# Requested:
# - all outputs
# - including original input columns
# - full audit trail

full_outputs_sheet_df = final_reconciled_df.copy()


# ============================================================
# 6. Print summaries and preview
# ============================================================
print("\n✅ Built sheet dataframes")
print(f"metadata_sheet_df shape: {metadata_sheet_df.shape}")
print(f"final_mapping_sheet_df shape: {final_mapping_sheet_df.shape}")
print(f"full_outputs_sheet_df shape: {full_outputs_sheet_df.shape}")

print("\n📄 metadata sheet preview:")
display(metadata_sheet_df.head(10))

print("\n📄 final-mapping sheet preview:")
display(final_mapping_sheet_df.head(10))

print("\n📄 full outputs sheet preview:")
display(full_outputs_sheet_df.head(5))


# ----------------------------
# Column review
# ----------------------------
print("\n📌 metadata_sheet_df columns:")
print(list(metadata_sheet_df.columns))

print("\n📌 final_mapping_sheet_df columns:")
print(list(final_mapping_sheet_df.columns))

print("\n📌 full_outputs_sheet_df column count:")
print(len(full_outputs_sheet_df.columns))

🧭 Original columns from config:
  variable_col: name
  form_col: section
  description_col: description
  encoding_col: enumLabels

🧭 Original input columns selected:
  - schemaVersion
  - section
  - name
  - title
  - description
  - type
  - format
  - constraints.required
  - constraints.maxLength
  - constraints.enum
  - constraints.pattern
  - constraints.maximum
  - constraints.minimum
  - enumLabels
  - enumOrdered
  - missingValues
  - trueValues
  - falseValues
  - custom
  - standardsMappings[0].instrument.url
  - standardsMappings[0].instrument.source
  - standardsMappings[0].instrument.title
  - standardsMappings[0].instrument.id
  - standardsMappings[0].item.url
  - standardsMappings[0].item.source
  - standardsMappings[0].item.id
  - relatedConcepts[0].url
  - relatedConcepts[0].title
  - relatedConcepts[0].source
  - relatedConcepts[0].id
  - CDE Acronym Finder
  - Parsed Full Response
  - Parse Status
  - Parse Error
  - Prestep Run Status
  - Prestep Attempts
  - Pres

,Original Variable Name,Original Form Name,Best Best Match CRF,Best Best Match CDE,Recon Decision
0,imh_birthhcr,a_infant_medical_history_01_month,,,
1,imh_birthlt,a_infant_medical_history_01_month,,,
2,imhbirthwt,a_infant_medical_history_01_month,,,No Match
3,BPIAvgPainRtngScl,bpisev,BPI Pain Severity,Brief Pain Inventory (BPI) - average pain rati...,Accept Candidate
4,BPICurrentPainRtngScl,bpisev,BPI Pain Severity,Brief Pain Inventory (BPI) - current pain rati...,Accept Candidate
5,BPILeastPnLst24HRtngScl,bpisev,BPI Pain Severity,Brief Pain Inventory (BPI) - least pain rating...,Accept Candidate
6,BPIWrstPnLast24HRtngScl,bpisev,BPI Pain Severity,Brief Pain Inventory (BPI) - worst pain rating...,Accept Candidate
7,GAD2FeelNervScl,gad7,GAD2 Pain,Generalized Anxiety Disorder (GAD-2) - feeling...,Accept Candidate
8,GAD2NotStopWryScl,gad7,GAD7,Generalized Anxiety Disorder (GAD-7) - not sto...,Accept Candidate
9,GAD7EasyAnnoyedScl,gad7,GAD7,Generalized Anxiety Disorder (GAD-7) - easily ...,Accept Candidate



📄 final-mapping sheet preview:


,schemaVersion,section,name,title,description,type,format,constraints.required,constraints.maxLength,constraints.enum,...,Normalized Encoding,Normalized Variable Name,Normalized Form Name,Normalized Combined,Best Match CDE Name,Best Match CRF Name,Final Decision,recon_confidence,recon_rationale,recon_result_source
0,0.3.2,a_infant_medical_history_01_month,imh_birthhcr,5. Head circumference at birth,5. Head circumference at birth,number,NaN,NaN,NaN,NaN,...,,imh birthhcr,a infant medical history 01 month,imh birthhcr 5 head circumference at birth nan,,,,,,
1,0.3.2,a_infant_medical_history_01_month,imh_birthlt,6. Length at birth,6. Length at birth,number,NaN,NaN,NaN,NaN,...,,imh birthlt,a infant medical history 01 month,imh birthlt 6 length at birth nan,,,,,,
2,0.3.2,a_infant_medical_history_01_month,imhbirthwt,4. Weight at birth,4. Weight at birth,number,NaN,NaN,NaN,NaN,...,,imhbirthwt,a infant medical history 01 month,imhbirthwt 4 weight at birth nan,,,No Match,High,None of the candidates correspond to birth wei...,LLM Recon
3,0.3.2,bpisev,BPIAvgPainRtngScl,Average Pain,Average Pain,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,0=no pain 0 1=1 2=2 3=3 4=4 5=5 6=6 7=7 8=8 9=...,bpiavgpainrtngscl,bpisev,bpiavgpainrtngscl average pain 0=no pain 0 1=1...,Brief Pain Inventory (BPI) - average pain rati...,BPI Pain Severity,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon
4,0.3.2,bpisev,BPICurrentPainRtngScl,Current Pain,Current Pain,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,0=no pain 0 1=1 2=2 3=3 4=4 5=5 6=6 7=7 8=8 9=...,bpicurrentpainrtngscl,bpisev,bpicurrentpainrtngscl current pain 0=no pain 0...,Brief Pain Inventory (BPI) - current pain rati...,BPI Pain Severity,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon
5,0.3.2,bpisev,BPILeastPnLst24HRtngScl,Least Pain in the Last 24 Hours,Least Pain in the Last 24 Hours,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,0=no pain 0 1=1 2=2 3=3 4=4 5=5 6=6 7=7 8=8 9=...,bpileastpnlst24hrtngscl,bpisev,bpileastpnlst24hrtngscl least pain in the last...,Brief Pain Inventory (BPI) - least pain rating...,BPI Pain Severity,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon
6,0.3.2,bpisev,BPIWrstPnLast24HRtngScl,Worst Pain in the Last 24 Hours,Worst Pain in the Last 24 Hours,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,0=no pain 0 1=1 2=2 3=3 4=4 5=5 6=6 7=7 8=8 9=...,bpiwrstpnlast24hrtngscl,bpisev,bpiwrstpnlast24hrtngscl worst pain in the last...,Brief Pain Inventory (BPI) - worst pain rating...,BPI Pain Severity,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon
7,0.3.2,gad7,GAD2FeelNervScl,Feeling Nervous,Feeling Nervous,integer,NaN,NaN,NaN,0|1|2|3,...,0=not at all 1=several days 2=more than half t...,gad2feelnervscl,gad7,gad2feelnervscl feeling nervous 0=not at all 1...,Generalized Anxiety Disorder (GAD-2) - feeling...,GAD2 Pain,Accept Candidate,High,The candidate exactly matches the study variab...,LLM Recon
8,0.3.2,gad7,GAD2NotStopWryScl,Unable to Stop Worrying,Unable to Stop Worrying,integer,NaN,NaN,NaN,0|1|2|3,...,0=not at all 1=several days 2=more than half t...,gad2notstopwryscl,gad7,gad2notstopwryscl unable to stop worrying 0=no...,Generalized Anxiety Disorder (GAD-7) - not sto...,GAD7,Accept Candidate,High,The top-ranked candidate exactly matches the s...,LLM Recon
9,0.3.2,gad7,GAD7EasyAnnoyedScl,Becoming Easily Annoyed/Irritable,Becoming Easily Annoyed/Irritable,integer,NaN,NaN,NaN,0|1|2|3,...,0=not at all 1=several days 2=more than half t...,gad7easyannoyedscl,gad7,gad7easyannoyedscl becoming easily annoyed irr...,Generalized Anxiety Disorder (GAD-7) - easily ...,GAD7,Accept Candidate,High,The candidate perfectly matches the study vari...,LLM Recon



📄 full outputs sheet preview:


,schemaVersion,section,name,title,description,type,format,constraints.required,constraints.maxLength,constraints.enum,...,recon_confidence,recon_rationale,recon_result_source,recon_was_skipped,recon_review_flag,recon_protected_family_anchor,recon_parsed_successfully,recon_raw_response,recon_error,recon_processed_this_run
0,0.3.2,a_infant_medical_history_01_month,imh_birthhcr,5. Head circumference at birth,5. Head circumference at birth,number,NaN,NaN,NaN,NaN,...,,,,False,,,False,,,False
1,0.3.2,a_infant_medical_history_01_month,imh_birthlt,6. Length at birth,6. Length at birth,number,NaN,NaN,NaN,NaN,...,,,,False,,,False,,,False
2,0.3.2,a_infant_medical_history_01_month,imhbirthwt,4. Weight at birth,4. Weight at birth,number,NaN,NaN,NaN,NaN,...,High,None of the candidates correspond to birth wei...,LLM Recon,False,,,True,"{""best_best_match_cde"":"""",""best_best_match_crf...",,True
3,0.3.2,bpisev,BPIAvgPainRtngScl,Average Pain,Average Pain,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,High,The candidate perfectly matches the study vari...,LLM Recon,False,,,True,"{""best_best_match_cde"":""Brief Pain Inventory (...",,True
4,0.3.2,bpisev,BPICurrentPainRtngScl,Current Pain,Current Pain,integer,NaN,NaN,NaN,0|1|2|3|4|5|6|7|8|9|10,...,High,The candidate exactly matches the study variab...,LLM Recon,False,,,True,"{""best_best_match_cde"":""Brief Pain Inventory (...",,True



📌 metadata_sheet_df columns:
['Original Variable Name', 'Original Form Name', 'Best Best Match CRF', 'Best Best Match CDE', 'Recon Decision']

📌 final_mapping_sheet_df columns:
['schemaVersion', 'section', 'name', 'title', 'description', 'type', 'format', 'constraints.required', 'constraints.maxLength', 'constraints.enum', 'constraints.pattern', 'constraints.maximum', 'constraints.minimum', 'enumLabels', 'enumOrdered', 'missingValues', 'trueValues', 'falseValues', 'custom', 'standardsMappings[0].instrument.url', 'standardsMappings[0].instrument.source', 'standardsMappings[0].instrument.title', 'standardsMappings[0].instrument.id', 'standardsMappings[0].item.url', 'standardsMappings[0].item.source', 'standardsMappings[0].item.id', 'relatedConcepts[0].url', 'relatedConcepts[0].title', 'relatedConcepts[0].source', 'relatedConcepts[0].id', 'CDE Acronym Finder', 'Parsed Full Response', 'Parse Status', 'Parse Error', 'Prestep Run Status', 'Prestep Attempts', 'Prestep Error', 'Match Rational

In [155]:
# ============================================================
# STEP 9c: Export final Excel workbook with 3 sheets
# ============================================================
# Goal:
# - Export the final reconciled results to Excel
# - Create 3 sheets:
#   1. metadata
#   2. final-mapping
#   3. all-outputs
# - Apply light formatting for readability

from pathlib import Path
import pandas as pd


# ----------------------------
# Safety checks
# ----------------------------
required_export_dfs = [
    "metadata_sheet_df",
    "final_mapping_sheet_df",
    "full_outputs_sheet_df",
]

for df_name in required_export_dfs:
    if df_name not in globals():
        raise NameError(f"{df_name} does not exist. Run Step 9b first.")


# ----------------------------
# Resolve output path from config
# ----------------------------
output_file_raw = config.get(
    "Files",
    "output_file",
    fallback="out/CDE_ID_recon_output.xlsx"
).strip()

output_path = Path(output_file_raw)

# Add a recon-friendly filename so we do not accidentally overwrite earlier outputs
if output_path.suffix.lower() == ".xlsx":
    output_path = output_path.with_name(
        output_path.stem + "_reconciled" + output_path.suffix
    )
else:
    output_path = output_path / "CDE_ID_recon_output_reconciled.xlsx"

# Make sure output folder exists
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"📁 Export path: {output_path}")


# ----------------------------
# Helper: clean sheet names
# ----------------------------
def safe_sheet_name(name):
    """
    Excel sheet names must be <= 31 characters and cannot contain certain characters.
    """
    invalid_chars = ["\\", "/", "*", "?", ":", "[", "]"]

    clean_name = str(name)

    for char in invalid_chars:
        clean_name = clean_name.replace(char, "-")

    return clean_name[:31]


# ----------------------------
# Helper: write dataframe with formatting
# ----------------------------
def write_formatted_sheet(writer, df, sheet_name, freeze_panes=(1, 0)):
    """
    Write a dataframe to Excel with basic formatting:
    - bold wrapped header
    - frozen header row
    - autofilter
    - reasonable column widths
    - wrapped text for long columns
    """
    sheet_name = safe_sheet_name(sheet_name)

    df.to_excel(
        writer,
        sheet_name=sheet_name,
        index=False
    )

    workbook = writer.book
    worksheet = writer.sheets[sheet_name]

    header_format = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "align": "center",
        "bg_color": "#D9EAF2",
        "border": 1
    })

    body_format = workbook.add_format({
        "text_wrap": True,
        "valign": "top"
    })

    # Header formatting
    for col_num, column_name in enumerate(df.columns):
        worksheet.write(0, col_num, column_name, header_format)

    # Freeze top row
    worksheet.freeze_panes(*freeze_panes)

    # Add autofilter
    if len(df.columns) > 0:
        worksheet.autofilter(0, 0, max(len(df), 1), len(df.columns) - 1)

    # Set reasonable column widths
    for col_num, column_name in enumerate(df.columns):
        col_values = df[column_name].astype(str).fillna("")

        max_content_width = col_values.map(len).max() if len(col_values) > 0 else 0
        header_width = len(str(column_name))

        width = max(header_width, max_content_width) + 2

        # Cap very wide columns
        if any(term in str(column_name).lower() for term in ["rationale", "response", "payload", "definition", "description", "notes"]):
            width = min(max(width, 25), 60)
        else:
            width = min(max(width, 12), 35)

        worksheet.set_column(col_num, col_num, width, body_format)

    # Set row height a little taller for wrapped text
    worksheet.set_default_row(24)

    return worksheet


# ----------------------------
# Export workbook
# ----------------------------
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    write_formatted_sheet(
        writer,
        metadata_sheet_df,
        sheet_name="metadata"
    )

    write_formatted_sheet(
        writer,
        final_mapping_sheet_df,
        sheet_name="final-mapping"
    )

    write_formatted_sheet(
        writer,
        full_outputs_sheet_df,
        sheet_name="all-outputs"
    )


print("✅ Final Excel workbook exported!")
print(f"Workbook path: {output_path}")
print("\nSheets created:")
print("  - metadata")
print("  - final-mapping")
print("  - all-outputs")

print("\nRow counts:")
print(f"  metadata: {len(metadata_sheet_df)}")
print(f"  final-mapping: {len(final_mapping_sheet_df)}")
print(f"  all-outputs: {len(full_outputs_sheet_df)}")

📁 Export path: out\Testfile_2026-05-05_reconciled.xlsx
✅ Final Excel workbook exported!
Workbook path: out\Testfile_2026-05-05_reconciled.xlsx

Sheets created:
  - metadata
  - final-mapping
  - all-outputs

Row counts:
  metadata: 14
  final-mapping: 14
  all-outputs: 14
